# UPDATED NOTEBOOK 12 — INCLUDES STAGE 4

**Verified edition: 45 cells. Stage 4 starts at cell 38 and continues to the final cell.**

# 12 - Freddie population drift and reliability
## Stage 1: setup and input validation

This section verifies inputs for the documented Notebook 12 methodology.
Stage 2 follows below and calculates feature-level drift; association testing remains later.

- Primary model: frozen XGBoost; secondary comparison: frozen Logistic Regression.
- Drift reference: 2015-2017, separately for eligible loans and full sampled originations.
- Evaluation: 12 half-years, 2018-2023; 24-month composite credit-event target.
- Reliability comparison anchor: 2018-H1, descriptive only.
- Origination predictors only; retain pre-imputation missingness and raw category coverage.

Run cells in order using the project's Python 3.12 virtual environment. Stage 1
reads existing artifacts and writes three small validation summaries to a new timestamped
folder. Stage 2 writes aggregate drift summaries and a review ZIP to a separate new folder. It does not retrain, rescore, change earlier outputs, or compute drift thresholds.

Reviewed GitHub source: `feature/freddie-baseline-training`, commit
`e9ec2fea9a3654c4f4d46015f0f38314eed900d7` (5 October 2026).

Stage 3 now builds and validates the cohort comparison table, then pauses for review.


In [10]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib, json, sys
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from IPython.display import display

def require(condition, message):
    if not condition:
        raise ValueError(message)

def sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
ROOT = ROOT.resolve()
SOURCE = ROOT / 'src' / 'freddie_features.py'
require(SOURCE.is_file(), 'Open the repository root or its notebooks folder before running.')
# Compare LF-normalized source against the reviewed Git blob; CRLF checkout is allowed.
source_bytes = SOURCE.read_bytes().replace(b'\r\n', b'\n')
source_blob = hashlib.sha1(b'blob ' + str(len(source_bytes)).encode() + b'\0' + source_bytes).hexdigest()
require(source_blob == '7da952013fd70095b001483f94256b4cf0a55e7d',
        'Feature source differs from the reviewed version. Share this error for reconciliation.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
features_module = importlib.import_module('src.freddie_features')
require(Path(features_module.__file__).resolve() == SOURCE.resolve(), 'Imported another repository feature module.')
features_module = importlib.reload(features_module)
PREDICTOR_COLUMNS = features_module.PREDICTOR_COLUMNS
NUMERIC_FEATURES = features_module.NUMERIC_FEATURES
CATEGORICAL_FEATURES = features_module.CATEGORICAL_FEATURES
REQUIRED_SOURCE_COLUMNS = features_module.REQUIRED_SOURCE_COLUMNS
LABEL_COLUMNS = features_module.LABEL_COLUMNS
build_engineered_features = features_module.build_engineered_features
apply_preprocessing_contract = features_module.apply_preprocessing_contract

KEYS = ['SOURCE YEAR', 'LOAN IDENTIFIER']
TARGET = 'TARGET_COMPOSITE_CREDIT_EVENT'
YEARS = list(range(2015, 2024))
FAMILIES = ['xgboost', 'logistic']
PERIODS = [f'{year}-{half}' for year in range(2018, 2024) for half in ['H1', 'H2']]
CONTINUOUS = NUMERIC_FEATURES[:7]
BINARY = NUMERIC_FEATURES[7:]
require(len(CONTINUOUS) == 7 and len(BINARY) == 8 and len(CATEGORICAL_FEATURES) == 14,
        'Predictor-type counts differ from the design.')
DATA = ROOT / 'data' / 'processed' / 'Freddie_Mac'
BASE = DATA / 'model_baselines_2015_2017'
OOT = BASE / 'oot_evaluation'
paths = {
    'modeling': DATA / 'freddie_modeling_dataset_2015_2023_24m.parquet',
    'manifest': BASE / 'model_manifest.json',
    'contract': BASE / 'freddie_feature_contract.json',
    'predictions': OOT / 'oot_predictions_2018_2023.parquet',
    'reliability': OOT / 'cohort_reliability_summary.csv',
    'bootstrap': OOT / 'half_year_bootstrap_intervals.csv',
    'calibration': OOT / 'half_year_calibration_bins.csv',
    'calibration_edges': OOT / 'frozen_calibration_bins.json',
    'feature_source': SOURCE,
}
paths.update({f'origination_{year}': DATA / f'origination_{year}_clean.parquet' for year in YEARS})
for name, path in paths.items():
    require(path.is_file(), f'Required input missing ({name}): {path}')
manifest = json.loads(paths['manifest'].read_text(encoding='utf-8'))
contract = json.loads(paths['contract'].read_text(encoding='utf-8'))
require(manifest['target'] == TARGET and manifest['horizon_months'] == 24, 'Target or horizon changed.')
require(manifest['development_years'] == [2015, 2016, 2017], 'Development years changed.')
require(manifest['predictors'] == PREDICTOR_COLUMNS, 'Manifest predictor order differs.')
require(manifest['numeric_predictors'] == NUMERIC_FEATURES and
        manifest['categorical_predictors'] == CATEGORICAL_FEATURES, 'Manifest feature types differ.')
require(contract['training_years'] == [2015, 2016, 2017], 'Frozen contract years differ.')
require(set(contract['numeric_medians']) == set(NUMERIC_FEATURES) and
        set(contract['category_levels']) == set(CATEGORICAL_FEATURES), 'Frozen contract schema differs.')
for family in FAMILIES:
    artifact = Path(manifest['models'][family]['artifact'])
    require(artifact.name == str(artifact), 'Model artifact must be a filename inside the baseline folder.')
    paths[f'model_{family}'] = BASE / artifact
    require(paths[f'model_{family}'].is_file(), f'Missing frozen model: {family}')
input_hashes = {name: sha256(path) for name, path in paths.items()}
require(input_hashes['modeling'] == manifest['input_file_sha256'], 'Modeling data changed since baseline fitting.')
require(input_hashes['contract'] == manifest['feature_contract_sha256'], 'Frozen contract changed.')
for family in FAMILIES:
    require(input_hashes[f'model_{family}'] == manifest['models'][family]['sha256'], f'{family} model changed.')
print('Verified frozen input, contract, source and model hashes. Models were not loaded or rescored.')


Verified frozen input, contract, source and model hashes. Models were not loaded or rescored.


## 1. Keys, cohorts, target and prediction alignment
Normalize Q1-Q4 safely. Require exact loan-year keys and quarter assignments;
checking total counts alone is insufficient.


In [11]:
def normalize(frame, label):
    frame = frame.copy()
    required = [*KEYS, 'ORIGINATION YEAR', 'ORIGINATION QUARTER', 'ORIGINATION COHORT']
    require(set(required).issubset(frame.columns), f'{label}: required key/cohort columns missing.')
    require(not frame[required].isna().any().any(), f'{label}: missing key/cohort values.')
    for column in ['SOURCE YEAR', 'ORIGINATION YEAR']:
        values = pd.to_numeric(frame[column], errors='raise')
        require(np.isfinite(values).all() and values.eq(np.floor(values)).all(), f'{label}: invalid year.')
        frame[column] = values.astype(int)
    frame['LOAN IDENTIFIER'] = frame['LOAN IDENTIFIER'].astype('string')
    parts = frame['LOAN IDENTIFIER'].str.extract(r'^[FA](\d{2})Q([1-4])\d{7}$')
    require(not parts.isna().any().any(), f'{label}: invalid loan identifier format.')
    quarter = pd.to_numeric(frame['ORIGINATION QUARTER'].astype('string').str.strip().str.removeprefix('Q'), errors='raise')
    require(quarter.isin([1, 2, 3, 4]).all(), f'{label}: invalid quarter.')
    if 'quarter' in frame:
        require(pd.to_numeric(frame['quarter'], errors='raise').eq(quarter).all(), f'{label}: saved quarter field mismatch.')
    if 'year' in frame:
        require(pd.to_numeric(frame['year'], errors='raise').eq(frame['SOURCE YEAR']).all(), f'{label}: saved year field mismatch.')
    frame['quarter'] = quarter.astype(int)
    require(frame['SOURCE YEAR'].eq(frame['ORIGINATION YEAR']).all(), f'{label}: year mismatch.')
    require(frame['SOURCE YEAR'].eq(2000 + parts[0].astype(int)).all() and
            frame['quarter'].eq(parts[1].astype(int)).all(), f'{label}: identifier/cohort mismatch.')
    require(not frame.duplicated(KEYS).any(), f'{label}: duplicate loan-year keys.')
    require(frame['SOURCE YEAR'].isin(YEARS).all(), f'{label}: unsupported years.')
    frame['ORIGINATION QUARTER'] = 'Q' + frame['quarter'].astype(str)
    expected_cohort = frame['ORIGINATION YEAR'].astype(str) + frame['ORIGINATION QUARTER']
    require(frame['ORIGINATION COHORT'].astype('string').eq(expected_cohort).all(), f'{label}: cohort text mismatch.')
    frame['period'] = frame['SOURCE YEAR'].astype(str) + '-' + np.where(frame['quarter'] <= 2, 'H1', 'H2')
    return frame

modeling = normalize(pd.read_parquet(paths['modeling'], columns=list(dict.fromkeys(
    [*REQUIRED_SOURCE_COLUMNS, 'HORIZON_MONTHS']))), 'modeling')
require(set(modeling['SOURCE YEAR']) == set(YEARS), 'Incomplete modeling year coverage.')
require(modeling['HORIZON_MONTHS'].eq(24).all(), 'Non-24-month target rows found.')
for label in LABEL_COLUMNS:
    require(modeling[label].isin([0, 1]).all(), f'{label}: missing or invalid label.')
dev_mask = modeling['SOURCE YEAR'].le(2017)
require(int(dev_mask.sum()) == manifest['development_rows'] == 143598, 'Development count changed.')
require(int(modeling.loc[dev_mask, TARGET].sum()) == manifest['development_events'], 'Development events changed.')
pred = normalize(pd.read_parquet(paths['predictions']), 'Notebook 11 predictions')
require(pred['SOURCE YEAR'].between(2018, 2023).all(), 'Predictions include non-evaluation years.')
require(pred['half_year'].eq(pred['period']).all(), 'Saved half-year assignment differs.')
require(pd.to_numeric(pred['year'], errors='raise').eq(pred['SOURCE YEAR']).all() and
        pd.to_numeric(pred['quarter'], errors='raise').eq(
            pd.to_numeric(pred['ORIGINATION QUARTER'].str.removeprefix('Q'))).all(), 'Prediction year/quarter fields differ.')
require(pred['y_true'].isin([0, 1]).all(), 'Prediction outcomes invalid.')
require(len(pred) == 294070 and int(pred['y_true'].sum()) == 6481, 'Notebook 11 evaluation totals changed.')
for family in FAMILIES:
    values = pd.to_numeric(pred[f'p_{family}'], errors='raise')
    require(np.isfinite(values).all() and values.between(0, 1).all(), f'{family}: invalid probabilities.')
matched = modeling.loc[~dev_mask, KEYS + ['period', 'quarter', TARGET]].merge(
    pred[KEYS + ['period', 'quarter', 'y_true']], on=KEYS, how='outer',
    suffixes=('_model', '_pred'), indicator=True, validate='one_to_one')
require(matched['_merge'].eq('both').all(), 'Modeling and prediction loan sets differ.')
require(matched['period_model'].eq(matched['period_pred']).all() and
        matched['quarter_model'].eq(matched['quarter_pred']).all(), 'Prediction cohort assignments differ.')
require(matched[TARGET].eq(matched['y_true']).all(), 'Prediction labels differ from modeling labels.')
del matched
print('Verified 143,598 development loans and 294,070 evaluation loans / 6,481 evaluation events.')


Verified 143,598 development loans and 294,070 evaluation loans / 6,481 evaluation events.


## 2. Full sampled origination coverage and source values
Load the origination source fields for all 29 predictors without adding labels to the
full sample. Compare eligible source values by loan key before transformations.
The full sample is our annual sampled files, not the entire Freddie Mac universe.


In [12]:
origin_columns = [column for column in REQUIRED_SOURCE_COLUMNS if column not in LABEL_COLUMNS and column != 'SOURCE YEAR']
frames = []
for year in YEARS:
    part = pd.read_parquet(paths[f'origination_{year}'], columns=origin_columns)
    part['SOURCE YEAR'] = year
    part = normalize(part, f'origination {year}')
    require(len(part) == 50000, f'{year}: sampled origination count changed.')
    frames.append(part)
originations = pd.concat(frames, ignore_index=True)
require(not originations.duplicated(KEYS).any(), 'Duplicate full-sample loan keys.')
expected_all = [f'{year}-{half}' for year in YEARS for half in ['H1', 'H2']]
require(set(originations['period']) == set(expected_all), 'Expected 18 origination half-years.')
require(originations.groupby('period').size().eq(25000).all(), 'Half-year sample sizes differ from the audit.')
require(not set(LABEL_COLUMNS).intersection(originations.columns), 'Outcome columns entered full origination sample.')
origin_index = originations.set_index(KEYS)
model_index = modeling.set_index(KEYS)
require(model_index.index.isin(origin_index.index).all(), 'Eligible loan absent from origination sample.')
source_compare = [column for column in origin_columns if column != 'LOAN IDENTIFIER']
left = origin_index.loc[model_index.index, source_compare]
right = model_index[source_compare]
for column in source_compare:
    try:
        pd.testing.assert_series_equal(left[column], right[column], check_dtype=False, check_names=False, check_exact=True)
    except AssertionError:
        raise ValueError(f'Origination/modeling source values differ: {column}') from None
originations['included'] = pd.MultiIndex.from_frame(originations[KEYS]).isin(model_index.index)
population_counts = originations.groupby('period', sort=True).agg(
    origination_loans=('included', 'size'), included_loans=('included', 'sum')).reset_index()
population_counts['excluded_loans'] = population_counts['origination_loans'] - population_counts['included_loans']
population_counts['inclusion_percentage'] = 100 * population_counts['included_loans'] / population_counts['origination_loans']
population_counts['role'] = np.where(population_counts['period'].str[:4].astype(int) <= 2017, 'development', 'evaluation')
require(int(population_counts['included_loans'].sum()) == 437668, 'Eligible total differs from the audit.')
del frames, origin_index, model_index, left, right
display(population_counts)


,period,origination_loans,included_loans,excluded_loans,inclusion_percentage,role
0,2015-H1,25000,23602,1398,94.408,development
1,2015-H2,25000,23642,1358,94.568,development
2,2016-H1,25000,24039,961,96.156,development
3,2016-H2,25000,24037,963,96.148,development
4,2017-H1,25000,24091,909,96.364,development
5,2017-H2,25000,24187,813,96.748,development
6,2018-H1,25000,23699,1301,94.796,evaluation
7,2018-H2,25000,23718,1282,94.872,evaluation
8,2019-H1,25000,24326,674,97.304,evaluation
9,2019-H2,25000,24285,715,97.140,evaluation


## 3. Predictor inventory and frozen preprocessing compatibility
Build eligible features using the existing source function. Keep the pre-contract
features for later drift measurement. Applying the saved contract here is an in-memory
compatibility check only; no rules are fitted and no models are loaded.
Full-sample predictor construction will be implemented in the next stage without targets.


In [13]:
engineered_eligible = build_engineered_features(modeling)
require(not engineered_eligible.duplicated(KEYS).any(), 'Engineered keys are duplicated.')
numeric_sources = {
    'FICO_SCORE': 'CLASSIC FICO_CLEAN',
    'MI_PERCENT': 'MORTGAGE INSURANCE PERCENTAGE (MI %)_CLEAN',
    'ORIGINAL_DTI': 'ORIGINAL DEBT-TO-INCOME (DTI) RATIO_CLEAN',
    'ORIGINAL_LTV': 'ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN',
    'ORIGINAL_INTEREST_RATE': 'ORIGINAL INTEREST RATE',
    'LOG_ORIGINAL_UPB': 'ORIGINAL UPB',
    'CLTV_MINUS_LTV': 'ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN',
}
for column in numeric_sources.values():
    for label, frame in [('modeling', modeling), ('origination', originations)]:
        observed = pd.to_numeric(frame[column], errors='raise').dropna()
        require(np.isfinite(observed).all(), f'{label}/{column}: nonfinite values.')
for column in CONTINUOUS:
    observed = engineered_eligible[column].dropna()
    require(np.isfinite(observed).all(), f'{column}: engineered values not finite.')
for column in BINARY:
    require(engineered_eligible[column].isin([0, 1]).all(), f'{column}: invalid binary feature.')
require(engineered_eligible['ORIGINAL_DTI_MISSING'].eq(engineered_eligible['ORIGINAL_DTI'].isna().astype(int)).all(),
        'DTI missingness flag differs from raw engineered values.')
processed_check = apply_preprocessing_contract(engineered_eligible, contract)
require(not processed_check[PREDICTOR_COLUMNS].isna().any().any(), 'Frozen contract leaves missing predictors.')
require(np.isfinite(processed_check[NUMERIC_FEATURES].to_numpy(dtype=float)).all(), 'Processed numeric values not finite.')
del processed_check
main_signals = {'FICO_SCORE', 'ORIGINAL_DTI', 'ORIGINAL_LTV', 'CHANNEL', 'ORIGINAL_DTI_MISSING'}
feature_inventory = pd.DataFrame([
    {'feature': feature,
     'type': 'numeric' if feature in CONTINUOUS else 'binary' if feature in BINARY else 'categorical',
     'measure': 'PSI; raw/IQR-scaled Wasserstein' if feature in CONTINUOUS else
                'absolute proportion change (percentage points)' if feature in BINARY else 'total variation',
     'main_association_signal': feature in main_signals,
     'eligible_missing_before_contract': int(engineered_eligible[feature].isna().sum())}
    for feature in PREDICTOR_COLUMNS
])
display(feature_inventory)
print('Pre-imputation eligible features are ready. Full origination source fields are ready.')


,feature,type,measure,main_association_signal,eligible_missing_before_contract
0,FICO_SCORE,numeric,PSI; raw/IQR-scaled Wasserstein,True,67
1,MI_PERCENT,numeric,PSI; raw/IQR-scaled Wasserstein,False,0
2,ORIGINAL_DTI,numeric,PSI; raw/IQR-scaled Wasserstein,True,8615
3,ORIGINAL_LTV,numeric,PSI; raw/IQR-scaled Wasserstein,True,2
4,ORIGINAL_INTEREST_RATE,numeric,PSI; raw/IQR-scaled Wasserstein,False,0
5,LOG_ORIGINAL_UPB,numeric,PSI; raw/IQR-scaled Wasserstein,False,0
6,CLTV_MINUS_LTV,numeric,PSI; raw/IQR-scaled Wasserstein,False,2
7,HAS_MORTGAGE_INSURANCE,binary,absolute proportion change (percentage points),False,0
8,HAS_SUBORDINATE_FINANCING,binary,absolute proportion change (percentage points),False,0
9,HIGH_LTV_GT_80,binary,absolute proportion change (percentage points),False,0


Pre-imputation eligible features are ready. Full origination source fields are ready.


## 4. Reconcile Notebook 11 reliability, intervals and calibration
Verify all saved annual, half-year and quarter counts and metric values against the
saved predictions. Validate existing intervals structurally; do not rerun bootstrap.
Within-cohort or model-difference intervals are not baseline-change intervals.


In [14]:
summary = pd.read_csv(paths['reliability'], dtype={'period': 'string'})
required_summary = ['period_kind', 'period', 'model', 'n', 'positives', 'negatives', 'prevalence',
                    'roc_auc', 'average_precision', 'brier', 'risk_gap', 'mean_predicted_risk']
require(set(required_summary).issubset(summary.columns), 'Reliability summary schema differs.')
require(not summary.duplicated(['period_kind', 'period', 'model']).any(), 'Duplicate reliability summary rows.')
expected_summary = {(kind, period, family) for kind, periods in [
    ('half_year', PERIODS), ('annual', [str(year) for year in range(2018, 2024)]),
    ('quarter', [f'{year}-Q{q}' for year in range(2018, 2024) for q in range(1, 5)])]
    for period in periods for family in FAMILIES}
require(set(summary[['period_kind', 'period', 'model']].itertuples(index=False, name=None)) == expected_summary,
        'Reliability cohort/model coverage differs.')
for row in summary.itertuples(index=False):
    if row.period_kind == 'half_year':
        part = pred.loc[pred['period'].eq(row.period)]
    elif row.period_kind == 'annual':
        part = pred.loc[pred['SOURCE YEAR'].eq(int(row.period))]
    else:
        year, quarter = row.period.split('-Q')
        part = pred.loc[pred['SOURCE YEAR'].eq(int(year)) & pred['quarter'].eq(int(quarter))]
    y = part['y_true'].to_numpy(dtype=int)
    probs = part[f'p_{row.model}'].to_numpy(dtype=float)
    require(row.n == len(y) and row.positives == int(y.sum()) and row.negatives == len(y)-int(y.sum()),
            f'{row.period}/{row.model}: summary counts differ.')
    values = {'prevalence': y.mean(), 'mean_predicted_risk': probs.mean(),
              'brier': brier_score_loss(y, probs), 'risk_gap': y.mean()-probs.mean(),
              'roc_auc': roc_auc_score(y, probs) if np.unique(y).size == 2 else np.nan,
              'average_precision': average_precision_score(y, probs) if np.unique(y).size == 2 else np.nan}
    for metric, expected in values.items():
        require(np.isclose(getattr(row, metric), expected, rtol=1e-9, atol=1e-12, equal_nan=True),
                f'{row.period}/{row.model}: saved {metric} differs from predictions.')

boot = pd.read_csv(paths['bootstrap'])
metrics = ['roc_auc', 'average_precision', 'brier', 'risk_gap']
require(set(['period', 'comparison', 'metric', 'ci_low', 'ci_high', 'valid_draws', 'invalid_rank_draws']).issubset(boot.columns),
        'Bootstrap schema differs.')
require(not boot.duplicated(['period', 'comparison', 'metric']).any(), 'Duplicate bootstrap rows.')
require(set(boot[['period', 'comparison', 'metric']].itertuples(index=False, name=None)) ==
        {(period, family, metric) for period in PERIODS for family in [*FAMILIES, 'xgboost_minus_logistic'] for metric in metrics},
        'Bootstrap coverage differs.')
require(boot['valid_draws'].between(0, 1000).all() and boot['invalid_rank_draws'].between(0, 1000).all(), 'Invalid bootstrap counts.')
require(boot[['valid_draws', 'invalid_rank_draws']].eq(np.floor(boot[['valid_draws', 'invalid_rank_draws']])).all().all(), 'Fractional bootstrap counts.')
rank_rows = boot['metric'].isin(['roc_auc', 'average_precision'])
require((boot.loc[rank_rows, 'valid_draws'] + boot.loc[rank_rows, 'invalid_rank_draws']).eq(1000).all(), 'Ranking draw counts differ.')
require(boot.loc[~rank_rows, 'valid_draws'].eq(1000).all(), 'Probability metric draw counts differ.')
valid_intervals = boot['valid_draws'].gt(0)
require(np.isfinite(boot.loc[valid_intervals, ['ci_low', 'ci_high']].to_numpy()).all(), 'Missing/nonfinite intervals with valid draws.')
require(boot.loc[valid_intervals, 'ci_low'].le(boot.loc[valid_intervals, 'ci_high']).all(), 'Interval bounds reversed.')
require(boot.loc[~valid_intervals, ['ci_low', 'ci_high']].isna().all().all(), 'Intervals exist without valid draws.')

edges_payload = json.loads(paths['calibration_edges'].read_text(encoding='utf-8'))
require(set(edges_payload) == set(FAMILIES), 'Frozen calibration model keys differ.')
cal = pd.read_csv(paths['calibration'], float_precision='round_trip')
require(set(['period', 'model', 'bin_index', 'lower_edge', 'upper_edge', 'n', 'events', 'mean_predicted', 'observed_rate']).issubset(cal.columns),
        'Calibration schema differs.')
require(not cal.duplicated(['period', 'model', 'bin_index']).any(), 'Duplicate calibration rows.')
expected_cal = set()
for family in FAMILIES:
    inner = np.asarray(edges_payload[family], dtype=float)
    require(np.isfinite(inner).all() and (np.diff(inner) > 0).all() and ((inner >= 0) & (inner <= 1)).all(), 'Invalid frozen calibration edges.')
    edges = np.r_[-np.inf, inner, np.inf]
    for period in PERIODS:
        part = pred.loc[pred['period'].eq(period)]
        indices = np.searchsorted(inner, part[f'p_{family}'].to_numpy(), side='right')
        for j in range(len(edges)-1):
            expected_cal.add((period, family, j+1))
            saved = cal.loc[cal['period'].eq(period) & cal['model'].eq(family) & cal['bin_index'].eq(j+1)]
            require(len(saved) == 1, f'{period}/{family}: missing calibration bin {j+1}.')
            saved = saved.iloc[0]
            subset = part.loc[indices == j]
            require(saved['n'] == len(subset) and saved['events'] == int(subset['y_true'].sum()), 'Calibration bin counts differ.')
            require(saved['lower_edge'] == edges[j] and saved['upper_edge'] == edges[j+1], 'Calibration bin boundaries differ.')
            mean = subset[f'p_{family}'].mean() if len(subset) else np.nan
            rate = subset['y_true'].mean() if len(subset) else np.nan
            require(np.isclose(saved['mean_predicted'], mean, equal_nan=True, rtol=1e-9, atol=1e-12) and
                    np.isclose(saved['observed_rate'], rate, equal_nan=True, rtol=1e-9, atol=1e-12), 'Calibration bin risks differ.')
require(set(cal[['period', 'model', 'bin_index']].itertuples(index=False, name=None)) == expected_cal, 'Unexpected calibration bins.')
half_year_reliability = summary.loc[summary['period_kind'].eq('half_year')].copy()
display(half_year_reliability[['period', 'model', 'n', 'positives', 'prevalence']])
print('All saved reliability point estimates reconcile. Existing interval and calibration structures validated.')


,period,model,n,positives,prevalence
0,2018-H1,xgboost,23699,327,0.013798
1,2018-H1,logistic,23699,327,0.013798
2,2018-H2,xgboost,23718,895,0.037735
3,2018-H2,logistic,23718,895,0.037735
4,2019-H1,xgboost,24326,1106,0.045466
5,2019-H1,logistic,24326,1106,0.045466
6,2019-H2,xgboost,24285,1121,0.046160
7,2019-H2,logistic,24285,1121,0.046160
8,2020-H1,xgboost,24841,607,0.024435
9,2020-H1,logistic,24841,607,0.024435


All saved reliability point estimates reconcile. Existing interval and calibration structures validated.


## 5. Save Stage 1 validation summaries
All checks must pass before a new output folder is created. No loan-level data are exported.
Rerunning creates another timestamped folder and does not overwrite prior runs.


In [15]:
require(all(sha256(path) == input_hashes[name] for name, path in paths.items()), 'An input changed during validation.')
validation = {
    'status': 'passed', 'stage': 'Notebook 12 Stage 1 - input validation only',
    'reviewed_branch': 'feature/freddie-baseline-training',
    'reviewed_commit': 'e9ec2fea9a3654c4f4d46015f0f38314eed900d7',
    'origination_rows': int(len(originations)), 'eligible_rows': int(len(modeling)),
    'development_rows': int(dev_mask.sum()), 'development_events': int(modeling.loc[dev_mask, TARGET].sum()),
    'oot_rows': int(len(pred)), 'oot_events': int(pred['y_true'].sum()),
    'predictor_count': len(PREDICTOR_COLUMNS), 'numeric_count': len(CONTINUOUS),
    'binary_count': len(BINARY), 'categorical_count': len(CATEGORICAL_FEATURES),
    'reconciled_reliability_rows': len(summary), 'validated_bootstrap_rows': len(boot),
    'validated_calibration_rows': len(cal),
    'input_files': [{'name': str(path.relative_to(ROOT)), 'sha256': input_hashes[name],
                     'bytes': path.stat().st_size} for name, path in paths.items()],
    'limitations': [
        'Full population means annual sampled origination files, not the entire Freddie Mac universe.',
        'Eligible membership uses later outcome availability; full-sample origination signals do not.',
        'Frozen 2015-2017 models with 24-month labels were not deployable in January 2018; this is retrospective.',
        'No bootstrap was rerun; interval structures were checked, not independently regenerated.',
        'No drift, association, early-warning validation, deterioration or retraining threshold is calculated here.'
    ]
}
stage1_output = DATA / 'notebook_12_validation' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
stage1_output.mkdir(parents=True, exist_ok=False)
population_counts.to_csv(stage1_output / 'notebook_12_population_counts.csv', index=False)
feature_inventory.to_csv(stage1_output / 'notebook_12_feature_inventory.csv', index=False)
(stage1_output / 'notebook_12_input_validation.json').write_text(json.dumps(validation, indent=2), encoding='utf-8')
print('Notebook 12 Stage 1 passed.')
print('Three small summaries saved to:', stage1_output)
print('Share the three summaries for review before extending the notebook to drift calculations.')


Notebook 12 Stage 1 passed.
Three small summaries saved to: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_validation\20261018T010128524808Z
Share the three summaries for review before extending the notebook to drift calculations.


## Stage 1 review completed

The three Stage 1 summaries were reviewed and passed on 5 October 2026.
Rerunning Stage 1 creates another validation folder with the same checks.
Continue to Stage 2 below; keep loan-level data and model artifacts local.


---
# Stage 2: feature-level drift

Stage 1's uploaded validation outputs were reviewed and passed on 5 October 2026.
Run Stage 1 first in this kernel, then continue below. The earlier cells are preserved.

This stage constructs full sampled-origination predictors without outcome columns,
freezes separate 2015-2017 references, and calculates drift for all 29 predictors
across 12 later half-years. It does not test drift-reliability associations.

Numeric distributions are measured before imputation. Categorical values are measured
before the frozen model contract groups them. These are borrower-population comparisons,
not changes to model preprocessing. Preserve existing definitions and transformations.

PSI bins are right-closed: `(-infinity, cut1]`, `(cut1, cut2]`, ..., `(lastcut, infinity)`.
This explicitly includes a repeated minimum, such as zero, in the first bin. Duplicate
reference decile cut points are removed. Empty reference bins, when present, are retained
and identified. Smoothing adds 0.5 per bin count; 0.1 and 1.0 are sensitivity settings.


In [16]:
from scipy.stats import wasserstein_distance
import zipfile

require(validation['status'] == 'passed', 'Run and pass Stage 1 in this kernel first.')
require(all(sha256(path) == input_hashes[name] for name, path in paths.items()), 'Inputs changed since Stage 1.')

def build_origination_predictors(source):
    """Mirror the reviewed feature transformations without creating or reading labels."""
    require(not set(LABEL_COLUMNS).intersection(source.columns), 'Full sample contains outcome columns.')
    needed = [c for c in REQUIRED_SOURCE_COLUMNS if c not in LABEL_COLUMNS]
    require(set(needed).issubset(source.columns), 'Full-sample predictor source columns are missing.')
    out = source[KEYS + ['ORIGINATION YEAR', 'ORIGINATION QUARTER', 'ORIGINATION COHORT']].copy()
    def numeric(column):
        values = pd.to_numeric(source[column], errors='raise')
        require(np.isfinite(values.dropna()).all(), f'Full-sample source is nonfinite: {column}')
        return values
    fico = numeric('CLASSIC FICO_CLEAN')
    mi = numeric('MORTGAGE INSURANCE PERCENTAGE (MI %)_CLEAN')
    dti = numeric('ORIGINAL DEBT-TO-INCOME (DTI) RATIO_CLEAN')
    ltv = numeric('ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN')
    cltv = numeric('ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN')
    upb = numeric('ORIGINAL UPB')
    out['FICO_SCORE'] = fico
    out['MI_PERCENT'] = mi
    out['ORIGINAL_DTI'] = dti
    out['ORIGINAL_LTV'] = ltv
    out['ORIGINAL_INTEREST_RATE'] = numeric('ORIGINAL INTEREST RATE')
    out['LOG_ORIGINAL_UPB'] = np.log1p(upb.where(upb.ge(0)))
    out['CLTV_MINUS_LTV'] = (cltv - ltv).clip(lower=0)
    # Match existing binary behavior, including unknown raw numeric inputs.
    # Missingness is reported separately; no new feature definitions are introduced.
    out['HAS_MORTGAGE_INSURANCE'] = mi.gt(0).astype('int8')
    out['HAS_SUBORDINATE_FINANCING'] = cltv.gt(ltv).astype('int8')
    out['HIGH_LTV_GT_80'] = ltv.gt(80).astype('int8')
    for name, values in {'FICO_SCORE_MISSING': fico, 'MI_PERCENT_MISSING': mi,
                         'ORIGINAL_DTI_MISSING': dti, 'ORIGINAL_LTV_MISSING': ltv,
                         'CLTV_MISSING': cltv}.items():
        out[name] = values.isna().astype('int8')
    categorical_map = {
        'FIRST_TIME_HOMEBUYER': 'FIRST TIME HOMEBUYER INDICATOR_CLEAN',
        'NUMBER_OF_UNITS': 'NUMBER OF UNITS_CLEAN',
        'OCCUPANCY_STATUS': 'OCCUPANCY STATUS_CLEAN',
        'CHANNEL': 'CHANNEL_CLEAN', 'AMORTIZATION_TYPE': 'AMORTIZATION TYPE',
        'PROPERTY_STATE': 'PROPERTY STATE', 'PROPERTY_TYPE': 'PROPERTY TYPE_CLEAN',
        'LOAN_PURPOSE': 'LOAN PURPOSE_CLEAN', 'ORIGINAL_LOAN_TERM': 'ORIGINAL LOAN TERM',
        'NUMBER_OF_BORROWERS_GROUP': 'NUMBER OF BORROWERS GROUP_CLEAN',
        'SUPER_CONFORMING_FLAG': 'SUPER CONFORMING FLAG',
        'SPECIAL_ELIGIBILITY_PROGRAM': 'SPECIAL ELIGIBILITY PROGRAM',
        'HARP_INDICATOR': 'HARP INDICATOR',
        'INTEREST_ONLY_INDICATOR': 'INTEREST ONLY (I/O) INDICATOR',
    }
    for name, column in categorical_map.items():
        out[name] = source[column].astype('string').str.strip()
    out['period'] = source['period'].to_numpy()
    return out[KEYS + ['ORIGINATION YEAR', 'ORIGINATION QUARTER', 'ORIGINATION COHORT', 'period'] + PREDICTOR_COLUMNS]

full_predictors = build_origination_predictors(originations)
require(not set(LABEL_COLUMNS).intersection(full_predictors.columns), 'Outcome column entered full predictors.')
# Prove all 29 adapter outputs agree with the original feature builder on eligible loans.
adapter_check = full_predictors.set_index(KEYS).loc[
    pd.MultiIndex.from_frame(engineered_eligible[KEYS]), PREDICTOR_COLUMNS]
original_check = engineered_eligible.set_index(KEYS)[PREDICTOR_COLUMNS]
pd.testing.assert_frame_equal(adapter_check, original_check, check_dtype=True, check_exact=True)
del adapter_check, original_check
eligible_predictors = engineered_eligible[KEYS + ['ORIGINATION YEAR', 'ORIGINATION QUARTER',
    'ORIGINATION COHORT'] + PREDICTOR_COLUMNS].copy()
eligible_predictors['period'] = modeling['period'].to_numpy()
drift_populations = {'eligible': eligible_predictors, 'full_sample': full_predictors}
for population, frame in drift_populations.items():
    require(not frame.duplicated(KEYS).any(), f'{population}: duplicate predictor keys.')
    require(not set(LABEL_COLUMNS).intersection(frame.columns), f'{population}: labels entered drift data.')
    for feature in CONTINUOUS:
        require(np.isfinite(frame[feature].dropna()).all(), f'{population}/{feature}: nonfinite feature.')
    for feature in BINARY:
        require(frame[feature].isin([0, 1]).all(), f'{population}/{feature}: invalid binary feature.')
print('Full-sample predictors built without outcomes; all 29 match the original builder on eligible loans.')


Full-sample predictors built without outcomes; all 29 match the original builder on eligible loans.


## 6. Freeze the two development references

Create references only from pooled 2015-2017 loans, with equal weight per loan.
Missing numeric values are excluded from numeric distributions and tracked separately.
JSON stores infinite outer boundaries as strings and unavailable values as null.
Categorical levels are tagged internally so an actual category named "(missing)" cannot
collide with missing values. The saved specifications include their meanings and counts.

A nonconstant feature may have zero reference IQR; keep raw Wasserstein, mark the scaled
version unavailable, and retain valid PSI bins. A constant reference uses the dedicated
change-from-constant percentage instead of a degenerate PSI.


In [17]:
SMOOTHING = [0.1, 0.5, 1.0]
PRIMARY_SMOOTHING = 0.5
BIN_QUANTILES = np.arange(0.1, 1.0, 0.1)
MISSING_TOKEN = 'M:'

def observed_numeric(series):
    values = pd.to_numeric(series, errors='raise').dropna().to_numpy(dtype=float)
    require(np.isfinite(values).all(), 'Nonfinite numeric values cannot enter drift calculations.')
    return values

def category_tokens(series):
    values = series.astype('string').str.strip()
    # Prefix every observed level, including empty strings, to avoid missing-marker collisions.
    return ('V:' + values).fillna(MISSING_TOKEN)

def category_description(token):
    return {'level': None if token == MISSING_TOKEN else token[2:],
            'is_missing_level': token == MISSING_TOKEN}

def bin_counts(values, cuts):
    return np.bincount(np.searchsorted(cuts, values, side='left'), minlength=len(cuts)+1)

def numeric_reference(series):
    values = observed_numeric(series)
    n = len(series)
    spec = {'n': n, 'observed_n': len(values), 'missing_n': n-len(values),
            'missing_percentage': 100*(n-len(values))/n if n else None,
            'mean': float(values.mean()) if len(values) else None,
            'median': float(np.median(values)) if len(values) else None,
            'minimum': float(values.min()) if len(values) else None,
            'maximum': float(values.max()) if len(values) else None,
            'iqr': float(np.quantile(values, .75)-np.quantile(values, .25)) if len(values) else None,
            'cuts': [], 'bin_boundaries': [], 'reference_bin_counts': [],
            'bin_count': 0, 'empty_reference_bins': 0, 'constant_value': None}
    if not len(values):
        spec['status'] = 'reference_no_observations'
    elif np.unique(values).size == 1:
        spec['status'] = 'reference_constant'
        spec['constant_value'] = float(values[0])
    else:
        cuts = np.unique(np.quantile(values, BIN_QUANTILES, method='linear'))
        counts = bin_counts(values, cuts)
        spec.update(status='ok', cuts=cuts.tolist(), bin_boundaries=['-Infinity', *cuts.tolist(), 'Infinity'],
                    reference_bin_counts=counts.tolist(), bin_count=len(counts),
                    empty_reference_bins=int((counts == 0).sum()))
    return spec

reference_specs = {}
reference_values = {}
for population, frame in drift_populations.items():
    reference = frame.loc[frame['SOURCE YEAR'].le(2017)]
    expected_n = int(population_counts.loc[population_counts['role'].eq('development'),
                   'included_loans' if population == 'eligible' else 'origination_loans'].sum())
    require(len(reference) == expected_n, f'{population}: development reference count differs.')
    specs = {'reference_years': [2015, 2016, 2017], 'reference_rows': len(reference),
             'weighting': 'equal weight per loan', 'features': {}}
    reference_values[population] = {}
    for feature in PREDICTOR_COLUMNS:
        if feature in CONTINUOUS:
            spec = numeric_reference(reference[feature])
            spec['type'] = 'numeric'
            reference_values[population][feature] = observed_numeric(reference[feature])
        elif feature in BINARY:
            spec = {'type': 'binary', 'status': 'ok', 'n': len(reference),
                    'ones_n': int(reference[feature].sum()), 'proportion_one': float(reference[feature].mean())}
        else:
            counts = category_tokens(reference[feature]).value_counts().to_dict()
            # Always retain the missing category in support, even when its reference count is zero.
            counts.setdefault(MISSING_TOKEN, 0)
            spec = {'type': 'categorical', 'status': 'ok', 'n': len(reference),
                    'levels': [{'token': token, **category_description(token), 'count': int(count),
                                'proportion': count/len(reference)} for token, count in sorted(counts.items())]}
        specs['features'][feature] = spec
    reference_specs[population] = specs

reference_payload = {
    'protocol': 'Notebook 12 Stage 2 - working methodology, 2026-10-05',
    'quantiles': BIN_QUANTILES.tolist(), 'quantile_method': 'linear',
    'interval_convention': 'right-closed interior bins; searchsorted side=left',
    'duplicate_cutpoints': 'removed', 'outer_boundaries': ['-Infinity', 'Infinity'],
    'smoothing_count_additions': SMOOTHING, 'primary_smoothing': PRIMARY_SMOOTHING,
    'numeric_missingness': 'separate; numeric distributions use nonmissing values only',
    'categorical_missingness': 'explicit tagged missing level, included in TV denominator',
    'unseen_category_rate': 'percentage of all cohort loans with observed levels absent from the corresponding reference',
    'binary_units': 'percentage points', 'total_variation_units': '0 to 1',
    'references': reference_specs
}
# Serialize now to verify strict JSON; no file is written before result checks pass.
json.dumps(reference_payload, allow_nan=False)
inventory_rows = [{'population': population, 'feature': feature, 'type': spec['type'],
                   'reference_status': spec['status'], 'bin_count': spec.get('bin_count'),
                   'reference_iqr': spec.get('iqr')} for population, payload in reference_specs.items()
                  for feature, spec in payload['features'].items()]
display(pd.DataFrame(inventory_rows).query("type == 'numeric'"))
print('Separate development references frozen in memory. Later cohorts have not defined bins or scaling.')


,population,feature,type,reference_status,bin_count,reference_iqr
0,eligible,FICO_SCORE,numeric,ok,10.0,73.000000
1,eligible,MI_PERCENT,numeric,ok,4.0,12.000000
2,eligible,ORIGINAL_DTI,numeric,ok,10.0,14.000000
3,eligible,ORIGINAL_LTV,numeric,ok,9.0,20.000000
4,eligible,ORIGINAL_INTEREST_RATE,numeric,ok,10.0,0.500000
5,eligible,LOG_ORIGINAL_UPB,numeric,ok,10.0,0.801074
6,eligible,CLTV_MINUS_LTV,numeric,ok,2.0,0.000000
29,full_sample,FICO_SCORE,numeric,ok,10.0,72.000000
30,full_sample,MI_PERCENT,numeric,ok,4.0,12.000000
31,full_sample,ORIGINAL_DTI,numeric,ok,10.0,14.000000


Separate development references frozen in memory. Later cohorts have not defined bins or scaling.


## 7. Calculate continuous feature-level drift

Report all 29 features, including unavailable comparisons. Numeric PSI and Wasserstein
describe the observed distribution; numeric missingness changes are separate columns.
Categorical TV includes missing values. Novel-category rates exclude the missing marker
and use all cohort loans as the denominator.

Distances do not identify whether risk increased. Reference/current means, medians,
category proportions, and signed binary/missingness changes provide direction.
No thresholds, alerts, composite drift score, or reliability associations are calculated.


In [18]:
def psi_from_counts(reference_counts, current_counts, addition):
    r = np.asarray(reference_counts, dtype=float)
    c = np.asarray(current_counts, dtype=float)
    require(r.shape == c.shape and r.sum() > 0 and c.sum() > 0, 'Invalid PSI count arrays.')
    require(addition > 0, 'PSI count addition must be positive.')
    r = (r + addition) / (r.sum() + addition*len(r))
    c = (c + addition) / (c.sum() + addition*len(c))
    return float(np.sum((c-r)*np.log(c/r)))

rows, psi_rows, bin_rows, category_rows = [], [], [], []
for population, frame in drift_populations.items():
    require(set(frame.loc[frame['SOURCE YEAR'].ge(2018), 'period']) == set(PERIODS), f'{population}: cohort coverage differs.')
    for period in PERIODS:
        current = frame.loc[frame['period'].eq(period)]
        count_row = population_counts.loc[population_counts['period'].eq(period)].iloc[0]
        expected_n = int(count_row['included_loans' if population == 'eligible' else 'origination_loans'])
        require(len(current) == expected_n, f'{population}/{period}: sample count differs.')
        for feature in PREDICTOR_COLUMNS:
            spec = reference_specs[population]['features'][feature]
            row = {'population': population, 'period': period, 'feature': feature, 'type': spec['type'],
                   'reference_n': int(reference_specs[population]['reference_rows']), 'current_n': len(current),
                   'main_association_signal': feature in main_signals, 'status': 'ok',
                   'psi': np.nan, 'wasserstein_raw': np.nan, 'wasserstein_iqr_scaled': np.nan,
                   'wasserstein_scaled_status': 'not_applicable', 'total_variation': np.nan,
                   'binary_signed_change_pp': np.nan, 'binary_absolute_change_pp': np.nan,
                   'reference_missing_percentage': np.nan, 'current_missing_percentage': np.nan,
                   'missing_signed_change_pp': np.nan, 'missing_absolute_change_pp': np.nan,
                   'unseen_category_percentage': np.nan, 'constant_changed_percentage': np.nan}
            if spec['type'] == 'numeric':
                values = observed_numeric(current[feature])
                row.update(reference_observed_n=spec['observed_n'], current_observed_n=len(values),
                           reference_mean=spec['mean'], current_mean=float(values.mean()) if len(values) else np.nan,
                           reference_median=spec['median'], current_median=float(np.median(values)) if len(values) else np.nan,
                           reference_iqr=spec['iqr'], bin_count=spec['bin_count'],
                           empty_reference_bins=spec['empty_reference_bins'],
                           reference_missing_percentage=spec['missing_percentage'],
                           current_missing_percentage=100*(len(current)-len(values))/len(current))
                change = row['current_missing_percentage'] - row['reference_missing_percentage']
                row.update(missing_signed_change_pp=change, missing_absolute_change_pp=abs(change))
                if not spec['observed_n']:
                    row['status'] = 'reference_no_observations'
                    row['wasserstein_scaled_status'] = 'reference_no_observations'
                elif not len(values):
                    row['status'] = 'comparison_no_observations'
                    row['wasserstein_scaled_status'] = 'comparison_no_observations'
                else:
                    row['wasserstein_raw'] = float(wasserstein_distance(reference_values[population][feature], values))
                    if spec['iqr'] > 0:
                        row['wasserstein_iqr_scaled'] = row['wasserstein_raw'] / spec['iqr']
                        row['wasserstein_scaled_status'] = 'ok'
                    else:
                        row['wasserstein_scaled_status'] = 'reference_iqr_zero'
                    if spec['status'] == 'reference_constant':
                        row['status'] = 'reference_constant'
                        row['constant_changed_percentage'] = 100*float(np.mean(values != spec['constant_value']))
                    else:
                        counts = bin_counts(values, spec['cuts'])
                        require(int(counts.sum()) == len(values), 'Numeric bins lost observations.')
                        row['psi'] = psi_from_counts(spec['reference_bin_counts'], counts, PRIMARY_SMOOTHING)
                        boundaries = spec['bin_boundaries']
                        for j, (ref_count, count) in enumerate(zip(spec['reference_bin_counts'], counts)):
                            bin_rows.append({'population': population, 'period': period, 'feature': feature,
                                'bin_index': j+1, 'lower_edge': boundaries[j], 'upper_edge': boundaries[j+1],
                                'reference_count': ref_count, 'reference_observed_n': spec['observed_n'],
                                'reference_proportion': ref_count/spec['observed_n'], 'current_count': int(count),
                                'current_observed_n': len(values), 'current_proportion': int(count)/len(values)})
                for addition in SMOOTHING:
                    value = psi_from_counts(spec['reference_bin_counts'], bin_counts(values, spec['cuts']), addition) if row['status'] == 'ok' else np.nan
                    psi_rows.append({'population': population, 'period': period, 'feature': feature,
                                     'smoothing_count_addition': addition, 'psi': value, 'status': row['status']})
            elif spec['type'] == 'binary':
                one = float(current[feature].mean())
                change = 100*(one-spec['proportion_one'])
                row.update(reference_proportion_one=spec['proportion_one'], current_proportion_one=one,
                           binary_signed_change_pp=change, binary_absolute_change_pp=abs(change))
            else:
                ref_counts = {level['token']: level['count'] for level in spec['levels']}
                counts = category_tokens(current[feature]).value_counts().to_dict()
                union = sorted(set(ref_counts) | set(counts))
                require(sum(counts.values()) == len(current), 'Categorical counts lost observations.')
                difference_sum = 0.0
                unseen_count = 0
                for token in union:
                    reference_count = ref_counts.get(token, 0)
                    count = int(counts.get(token, 0))
                    ref_prop, cur_prop = reference_count/spec['n'], count/len(current)
                    novel = token != MISSING_TOKEN and token not in ref_counts
                    unseen_count += count if novel else 0
                    difference_sum += abs(cur_prop-ref_prop)
                    category_rows.append({'population': population, 'period': period, 'feature': feature,
                        **category_description(token), 'reference_count': reference_count, 'reference_n': spec['n'],
                        'reference_percentage': 100*ref_prop, 'current_count': count, 'current_n': len(current),
                        'current_percentage': 100*cur_prop, 'signed_change_pp': 100*(cur_prop-ref_prop),
                        'unseen_in_reference': novel})
                missing_ref = 100*ref_counts.get(MISSING_TOKEN, 0)/spec['n']
                missing_cur = 100*counts.get(MISSING_TOKEN, 0)/len(current)
                row.update(total_variation=0.5*difference_sum, unseen_category_percentage=100*unseen_count/len(current),
                           reference_missing_percentage=missing_ref, current_missing_percentage=missing_cur,
                           missing_signed_change_pp=missing_cur-missing_ref, missing_absolute_change_pp=abs(missing_cur-missing_ref))
            rows.append(row)
        print('Calculated feature drift:', population, period)

feature_drift = pd.DataFrame(rows)
psi_sensitivity = pd.DataFrame(psi_rows)
numeric_bins = pd.DataFrame(bin_rows, columns=['population', 'period', 'feature', 'bin_index', 'lower_edge',
    'upper_edge', 'reference_count', 'reference_observed_n', 'reference_proportion',
    'current_count', 'current_observed_n', 'current_proportion'])
category_proportions = pd.DataFrame(category_rows)
display(feature_drift.loc[feature_drift['main_association_signal'],
    ['population', 'period', 'feature', 'status', 'psi', 'wasserstein_raw', 'wasserstein_iqr_scaled',
     'total_variation', 'binary_signed_change_pp', 'binary_absolute_change_pp']].round(6))


Calculated feature drift: eligible 2018-H1
Calculated feature drift: eligible 2018-H2
Calculated feature drift: eligible 2019-H1
Calculated feature drift: eligible 2019-H2
Calculated feature drift: eligible 2020-H1
Calculated feature drift: eligible 2020-H2
Calculated feature drift: eligible 2021-H1
Calculated feature drift: eligible 2021-H2
Calculated feature drift: eligible 2022-H1
Calculated feature drift: eligible 2022-H2
Calculated feature drift: eligible 2023-H1
Calculated feature drift: eligible 2023-H2
Calculated feature drift: full_sample 2018-H1
Calculated feature drift: full_sample 2018-H2
Calculated feature drift: full_sample 2019-H1
Calculated feature drift: full_sample 2019-H2
Calculated feature drift: full_sample 2020-H1
Calculated feature drift: full_sample 2020-H2
Calculated feature drift: full_sample 2021-H1
Calculated feature drift: full_sample 2021-H2
Calculated feature drift: full_sample 2022-H1
Calculated feature drift: full_sample 2022-H2
Calculated feature drift

,population,period,feature,status,psi,wasserstein_raw,wasserstein_iqr_scaled,total_variation,binary_signed_change_pp,binary_absolute_change_pp
0,eligible,2018-H1,FICO_SCORE,ok,0.004227,1.900492,0.026034,NaN,NaN,NaN
2,eligible,2018-H1,ORIGINAL_DTI,ok,0.043115,1.352951,0.096639,NaN,NaN,NaN
3,eligible,2018-H1,ORIGINAL_LTV,ok,0.032198,1.834451,0.091723,NaN,NaN,NaN
12,eligible,2018-H1,ORIGINAL_DTI_MISSING,ok,NaN,NaN,NaN,NaN,-4.142917,4.142917
18,eligible,2018-H1,CHANNEL,ok,NaN,NaN,NaN,0.035164,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
667,full_sample,2023-H2,FICO_SCORE,ok,0.009535,3.285655,0.045634,NaN,NaN,NaN
669,full_sample,2023-H2,ORIGINAL_DTI,ok,0.281044,4.123137,0.294510,NaN,NaN,NaN
670,full_sample,2023-H2,ORIGINAL_LTV,ok,0.118173,2.878928,0.143946,NaN,NaN,NaN
679,full_sample,2023-H2,ORIGINAL_DTI_MISSING,ok,NaN,NaN,NaN,NaN,-5.569333,5.569333


## 8. Validate and save Stage 2 summaries

Save continuous measurements and distribution details, including unavailable cases.
The output folder contains six summary files and one ZIP containing those six files only.
No loan identifiers, predictions, outcomes, or model artifacts are exported in the review bundle.


In [19]:
require(len(feature_drift) == 2*len(PERIODS)*len(PREDICTOR_COLUMNS), 'Feature drift row coverage differs.')
require(not feature_drift.duplicated(['population', 'period', 'feature']).any(), 'Duplicate drift rows.')
require(len(psi_sensitivity) == 2*len(PERIODS)*len(CONTINUOUS)*len(SMOOTHING), 'PSI sensitivity coverage differs.')
require(not psi_sensitivity.duplicated(['population', 'period', 'feature', 'smoothing_count_addition']).any(), 'Duplicate PSI sensitivity rows.')
for metric in ['psi', 'wasserstein_raw', 'wasserstein_iqr_scaled', 'total_variation',
               'binary_absolute_change_pp', 'missing_absolute_change_pp', 'constant_changed_percentage']:
    observed = feature_drift[metric].dropna().to_numpy(dtype=float)
    require(np.isfinite(observed).all() and (observed >= -1e-12).all(), f'Invalid drift values: {metric}')
require(feature_drift['total_variation'].dropna().between(0, 1+1e-12).all(), 'TV outside 0-1.')
require(feature_drift['binary_absolute_change_pp'].dropna().between(0, 100+1e-12).all(), 'Binary change outside 0-100 pp.')
for column in ['reference_missing_percentage', 'current_missing_percentage', 'unseen_category_percentage', 'constant_changed_percentage']:
    require(feature_drift[column].dropna().between(0, 100+1e-12).all(), f'Invalid percentage: {column}')
numeric_result = feature_drift.loc[feature_drift['type'].eq('numeric')]
require(numeric_result.loc[numeric_result['status'].eq('ok'), 'psi'].notna().all(), 'Available numeric PSI is missing.')
require(numeric_result.loc[~numeric_result['status'].eq('ok'), 'psi'].isna().all(), 'Unavailable or constant PSI must remain missing.')
primary_psi = psi_sensitivity.loc[psi_sensitivity['smoothing_count_addition'].eq(PRIMARY_SMOOTHING)]
joined = numeric_result.merge(primary_psi, on=['population', 'period', 'feature'], suffixes=('_main', '_sensitivity'), validate='one_to_one')
require(np.allclose(joined['psi_main'], joined['psi_sensitivity'], equal_nan=True), 'Primary PSI differs from sensitivity table.')
for _, part in category_proportions.groupby(['population', 'period', 'feature']):
    require(np.isclose(part['reference_percentage'].sum(), 100) and np.isclose(part['current_percentage'].sum(), 100), 'Category proportions do not sum to 100.')
    require(part['reference_count'].sum() == part['reference_n'].iloc[0] and
            part['current_count'].sum() == part['current_n'].iloc[0], 'Category counts differ from denominators.')
if len(numeric_bins):
    for _, part in numeric_bins.groupby(['population', 'period', 'feature']):
        require(np.isclose(part['reference_proportion'].sum(), 1) and np.isclose(part['current_proportion'].sum(), 1), 'Numeric bin proportions do not sum to 1.')
        require(part['reference_count'].sum() == part['reference_observed_n'].iloc[0] and
                part['current_count'].sum() == part['current_observed_n'].iloc[0], 'Numeric bin counts differ.')
require(all(sha256(path) == input_hashes[name] for name, path in paths.items()), 'Inputs changed during Stage 2.')
stage2_validation = {
    'status': 'passed', 'stage': 'Notebook 12 Stage 2 - feature-level drift only',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'full_sample_predictors_match_original_builder_on_eligible_loans': True,
    'outcomes_excluded_from_drift_frames': True,
    'reference_rows': {name: payload['reference_rows'] for name, payload in reference_specs.items()},
    'feature_drift_rows': len(feature_drift), 'psi_sensitivity_rows': len(psi_sensitivity),
    'numeric_bin_rows': len(numeric_bins), 'category_proportion_rows': len(category_proportions),
    'status_counts': {str(k): int(v) for k, v in feature_drift['status'].value_counts().items()},
    'scaled_wasserstein_status_counts': {str(k): int(v) for k, v in numeric_result['wasserstein_scaled_status'].value_counts().items()},
    'binning_and_smoothing': 'Reference deciles; unique cutpoints; right-closed bins; 0.5 count smoothing with 0.1 and 1.0 sensitivity.',
    'input_files': validation['input_files'],
    'limitations': validation['limitations'][:-1] + [
        'Drift magnitudes do not establish reliability deterioration, harmlessness, or retraining need.',
        'The two population views have separate reference bins and scaling; compare their cohort patterns with this distinction acknowledged.',
        'No drift-reliability association testing or alert thresholds are implemented in Stage 2.',
        'Categorical blanks follow existing feature definitions; missing flags distinguish actual missing values.',
        'Binary financing/insurance/high-LTV flags preserve existing behavior when their raw inputs are missing; consult separate missingness measures.'
    ]
}
json.dumps(stage2_validation, allow_nan=False)
stage2_output = DATA / 'notebook_12_drift' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
stage2_output.mkdir(parents=True, exist_ok=False)
file_names = [
    'frozen_drift_reference_specifications.json', 'half_year_feature_drift.csv',
    'half_year_psi_sensitivity.csv', 'half_year_numeric_bin_proportions.csv',
    'half_year_category_proportions.csv', 'notebook_12_stage2_validation.json'
]
(stage2_output/file_names[0]).write_text(json.dumps(reference_payload, indent=2, allow_nan=False), encoding='utf-8')
feature_drift.to_csv(stage2_output/file_names[1], index=False)
psi_sensitivity.to_csv(stage2_output/file_names[2], index=False)
numeric_bins.to_csv(stage2_output/file_names[3], index=False)
category_proportions.to_csv(stage2_output/file_names[4], index=False)
(stage2_output/file_names[5]).write_text(json.dumps(stage2_validation, indent=2, allow_nan=False), encoding='utf-8')
bundle = stage2_output/'notebook_12_stage2_review.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in file_names:
        archive.write(stage2_output/name, arcname=name)
require(all(sha256(path) == input_hashes[name] for name, path in paths.items()), 'An input changed before Stage 2 finished.')
print('Notebook 12 Stage 2 passed.')
print('Summary folder:', stage2_output)
print('Upload this one review bundle:', bundle)
print('Drift/reliability relationships and thresholds have not been tested.')


Notebook 12 Stage 2 passed.
Summary folder: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_drift\20261018T010147787365Z
Upload this one review bundle: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_drift\20261018T010147787365Z\notebook_12_stage2_review.zip
Drift/reliability relationships and thresholds have not been tested.


## Stage 2 review completed

The uploaded Stage 2 aggregate bundle passed review on 5 October 2026: all 696
feature comparisons and 504 PSI sensitivity rows were covered. Reduced bin counts and
zero-IQR scaling follow the frozen rules. Interest-rate PSI magnitude is sensitive to
smoothing; retain all descriptive results without expanding the five selected signals.
Continue below. Rerunning earlier cells creates new folders without overwriting prior runs.


---
# Stage 3A: build the cohort comparison table

Run Stages 1 and 2 in this kernel first. This step joins the five previously selected
signals to the saved Notebook 11 half-year results. It stops before plots or associations.

Each row represents one **population view × half-year × frozen model** (48 rows).
Reliability is always measured on the eligible modeling population. Full-sample drift
is therefore paired with eligible-population reliability; it is not full-sample model
performance. Both models share the same labels and eligible cohort counts.

The three numeric signals use Wasserstein distance divided by their own population's
development IQR, with raw distance and PSI retained for context. Channel uses total
variation; DTI missingness uses absolute percentage-point change in the existing flag.
Keep missing/unavailable measurements as missing, never zero.

Attach existing within-cohort bootstrap intervals for ROC-AUC, average precision,
Brier score and **signed** risk gap. Absolute risk gap is a point estimate only: taking
absolute interval endpoints would not produce its confidence interval. These intervals
are neither changes from 2018-H1 nor prediction intervals for later reliability.

Timing: origination signals are matched to the **same cohort's subsequent 24-month
outcomes**, not the next half-year. This remains a retrospective analysis: eligible
membership depends on later observability, and the final development model uses
24-month labels from 2015-2017. No prospective warning claim is established here.


In [20]:
require(stage2_validation['status'] == 'passed', 'Run and pass Stage 2 first.')
require(all(sha256(path) == input_hashes[name] for name, path in paths.items()), 'Inputs changed since validation.')
# Use this run's explicit output folder; never silently select a latest folder.
stage3_sources = {
    'feature_drift': stage2_output / 'half_year_feature_drift.csv',
    'stage2_validation': stage2_output / 'notebook_12_stage2_validation.json',
    'reference_specifications': stage2_output / 'frozen_drift_reference_specifications.json',
    'reliability': paths['reliability'], 'bootstrap': paths['bootstrap'],
}
stage3_hashes = {name: sha256(path) for name, path in stage3_sources.items()}
stage3_drift = pd.read_csv(stage3_sources['feature_drift'], float_precision='round_trip')
stage3_reliability = pd.read_csv(stage3_sources['reliability'], float_precision='round_trip')
stage3_boot = pd.read_csv(stage3_sources['bootstrap'], float_precision='round_trip')
pd.testing.assert_frame_equal(stage3_drift, feature_drift, check_dtype=False, check_exact=True)
pd.testing.assert_frame_equal(stage3_reliability, summary, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-15)
pd.testing.assert_frame_equal(stage3_boot, boot, check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-15)
SIGNAL_RULES = {
    'FICO_SCORE': ('fico_wasserstein_iqr_scaled', 'wasserstein_iqr_scaled', 'development IQR units'),
    'ORIGINAL_DTI': ('dti_wasserstein_iqr_scaled', 'wasserstein_iqr_scaled', 'development IQR units'),
    'ORIGINAL_LTV': ('ltv_wasserstein_iqr_scaled', 'wasserstein_iqr_scaled', 'development IQR units'),
    'CHANNEL': ('channel_total_variation', 'total_variation', 'proportion distance'),
    'ORIGINAL_DTI_MISSING': ('dti_missing_absolute_change_pp', 'binary_absolute_change_pp', 'percentage points'),
}
require(set(SIGNAL_RULES) == main_signals, 'Selected signal set changed.')


In [21]:
def assemble_cohort_comparison(drift, reliability, bootstrap, counts):
    expected_signals = {(pop, period, feature) for pop in ['eligible', 'full_sample']
                        for period in PERIODS for feature in SIGNAL_RULES}
    selected = drift.loc[drift['feature'].isin(SIGNAL_RULES)].copy()
    require(not selected.duplicated(['population', 'period', 'feature']).any(), 'Duplicate selected drift rows.')
    require(set(selected[['population', 'period', 'feature']].itertuples(index=False, name=None)) == expected_signals,
            'Selected signal coverage differs.')
    selected['signal'] = selected['feature'].map(lambda f: SIGNAL_RULES[f][0])
    selected['signal_measure'] = selected['feature'].map(lambda f: SIGNAL_RULES[f][1])
    selected['signal_units'] = selected['feature'].map(lambda f: SIGNAL_RULES[f][2])
    selected['signal_value'] = [row[ SIGNAL_RULES[row['feature']][1] ] for _, row in selected.iterrows()]
    selected['signal_status'] = np.where(selected['signal_value'].notna(), 'available', 'unavailable')
    selected['signal_calculation_status'] = [row['wasserstein_scaled_status'] if row['signal_measure'] == 'wasserstein_iqr_scaled'
                                             else row['status'] for _, row in selected.iterrows()]
    require(np.isfinite(selected['signal_value'].dropna()).all(), 'Nonfinite selected signal.')
    # DTI missingness has two independently exported representations; reconcile them.
    missing = selected.loc[selected['feature'].eq('ORIGINAL_DTI_MISSING')]
    dti_missing = drift.loc[drift['feature'].eq('ORIGINAL_DTI'),
                           ['population', 'period', 'missing_absolute_change_pp', 'missing_signed_change_pp']]
    missing = missing.merge(dti_missing, on=['population', 'period'], validate='one_to_one')
    require(np.allclose(missing['signal_value'], missing['missing_absolute_change_pp_y'], equal_nan=True),
            'DTI missingness flag and numeric missingness disagree.')
    require(np.allclose(missing['binary_signed_change_pp'], missing['missing_signed_change_pp_y'], equal_nan=True),
            'DTI signed missingness representations disagree.')
    half = reliability.loc[reliability['period_kind'].eq('half_year')].copy()
    require(not half.duplicated(['period', 'model']).any(), 'Duplicate half-year reliability rows.')
    require(set(half[['period', 'model']].itertuples(index=False, name=None)) ==
            {(period, family) for period in PERIODS for family in FAMILIES}, 'Half-year reliability coverage differs.')
    half['absolute_risk_gap'] = half['risk_gap'].abs()
    for metric in ['roc_auc', 'average_precision', 'brier', 'risk_gap']:
        intervals = bootstrap.loc[bootstrap['metric'].eq(metric) & bootstrap['comparison'].isin(FAMILIES),
                                  ['period', 'comparison', 'ci_low', 'ci_high', 'valid_draws', 'invalid_rank_draws']].copy()
        require(not intervals.duplicated(['period', 'comparison']).any(), 'Duplicate model interval rows.')
        require(set(intervals[['period', 'comparison']].itertuples(index=False, name=None)) ==
                {(period, family) for period in PERIODS for family in FAMILIES}, 'Model interval coverage differs.')
        intervals = intervals.rename(columns={'comparison': 'model', **{c: f'{metric}_{c}' for c in
                    ['ci_low', 'ci_high', 'valid_draws', 'invalid_rank_draws']}})
        half = half.merge(intervals, on=['period', 'model'], validate='one_to_one')
    wide = selected.pivot(index=['population', 'period'], columns='signal', values='signal_value').reset_index()
    wide.columns.name = None
    for signal in [rule[0] for rule in SIGNAL_RULES.values()]:
        status = selected.loc[selected['signal'].eq(signal), ['population', 'period', 'signal_calculation_status']]
        wide = wide.merge(status.rename(columns={'signal_calculation_status': signal+'_status'}),
                          on=['population', 'period'], validate='one_to_one')
    cohort_counts = counts.loc[counts['period'].isin(PERIODS),
                               ['period', 'origination_loans', 'included_loans', 'excluded_loans', 'inclusion_percentage']]
    require(not cohort_counts.duplicated('period').any() and set(cohort_counts['period']) == set(PERIODS),
            'Population count coverage differs.')
    wide = wide.merge(cohort_counts, on='period', validate='many_to_one')
    comparison = wide.merge(half, on='period', validate='many_to_many')
    require(len(comparison) == 2*len(PERIODS)*len(FAMILIES) and
            not comparison.duplicated(['population', 'period', 'model']).any(), 'Comparison join coverage differs.')
    require(comparison['n'].eq(comparison['included_loans']).all(), 'Reliability denominator differs from eligible count.')
    comparison['drift_current_n'] = np.where(comparison['population'].eq('eligible'),
                                            comparison['included_loans'], comparison['origination_loans'])
    comparison['drift_reference_n'] = comparison['population'].map(
        {pop: spec['reference_rows'] for pop, spec in reference_specs.items()})
    comparison['reliability_population'] = 'eligible'
    comparison['outcome_horizon_months'] = 24
    comparison['population_pairing'] = np.where(comparison['population'].eq('eligible'),
                                               'eligible drift / eligible reliability', 'full sampled drift / eligible reliability')
    # Verify every original reliability field and interval survived the join unchanged.
    for population in ['eligible', 'full_sample']:
        actual = comparison.loc[comparison['population'].eq(population)].set_index(['period', 'model'])
        expected = half.set_index(['period', 'model'])
        pd.testing.assert_frame_equal(actual[expected.columns].sort_index(), expected.sort_index(), check_exact=True)
    for row in selected.itertuples(index=False):
        actual = comparison.loc[comparison['population'].eq(row.population) & comparison['period'].eq(row.period), row.signal]
        require(np.allclose(actual, row.signal_value, equal_nan=True), 'Selected drift value changed during join.')
        require(row.current_n == comparison.loc[comparison['population'].eq(row.population) & comparison['period'].eq(row.period),
                                                'drift_current_n'].iloc[0], 'Drift cohort denominator differs.')
    return comparison.sort_values(['population', 'period', 'model']).reset_index(drop=True), selected.sort_values(
        ['population', 'period', 'feature']).reset_index(drop=True)

cohort_comparison, selected_signal_details = assemble_cohort_comparison(
    stage3_drift, stage3_reliability, stage3_boot, population_counts)
display(cohort_comparison[['population', 'period', 'model', 'n', 'positives', 'prevalence',
    *[rule[0] for rule in SIGNAL_RULES.values()], 'roc_auc', 'average_precision', 'brier', 'risk_gap', 'absolute_risk_gap']].round(6))
print('Verified all 48 population/cohort/model rows. Reliability is measured on eligible loans in both population views.')


,population,period,model,n,positives,prevalence,fico_wasserstein_iqr_scaled,dti_wasserstein_iqr_scaled,ltv_wasserstein_iqr_scaled,channel_total_variation,dti_missing_absolute_change_pp,roc_auc,average_precision,brier,risk_gap,absolute_risk_gap
0,eligible,2018-H1,logistic,23699,327,0.013798,0.026034,0.096639,0.091723,0.035164,4.142917,0.745899,0.041886,0.013520,0.003852,0.003852
1,eligible,2018-H1,xgboost,23699,327,0.013798,0.026034,0.096639,0.091723,0.035164,4.142917,0.743211,0.044413,0.013467,0.004657,0.004657
2,eligible,2018-H2,logistic,23718,895,0.037735,0.024336,0.108408,0.123139,0.035309,5.042159,0.720536,0.095681,0.036261,0.026076,0.026076
3,eligible,2018-H2,xgboost,23718,895,0.037735,0.024336,0.108408,0.123139,0.035309,5.042159,0.709291,0.089432,0.036470,0.027610,0.027610
4,eligible,2019-H1,logistic,24326,1106,0.045466,0.025050,0.092015,0.135450,0.078645,5.500776,0.708452,0.098268,0.044066,0.036003,0.036003
5,eligible,2019-H1,xgboost,24326,1106,0.045466,0.025050,0.092015,0.135450,0.078645,5.500776,0.702694,0.098105,0.044132,0.036682,0.036682
6,eligible,2019-H2,logistic,24285,1121,0.046160,0.059646,0.038981,0.053986,0.077957,5.628212,0.704759,0.102958,0.045153,0.040200,0.040200
7,eligible,2019-H2,xgboost,24285,1121,0.046160,0.059646,0.038981,0.053986,0.077957,5.628212,0.700339,0.102976,0.045166,0.040479,0.040479
8,eligible,2020-H1,logistic,24841,607,0.024435,0.121148,0.053223,0.108045,0.028245,5.628212,0.730006,0.062825,0.024037,0.020243,0.020243
9,eligible,2020-H1,xgboost,24841,607,0.024435,0.121148,0.053223,0.108045,0.028245,5.628212,0.717559,0.058308,0.024060,0.020322,0.020322


Verified all 48 population/cohort/model rows. Reliability is measured on eligible loans in both population views.


## 9. Save the comparison review bundle

Four aggregate files are saved: the 48-row comparison table, the 120-row selected-signal
detail table, the comparison specification, and validation/provenance. The ZIP contains
only those four files. No loan records or model artifacts are included.


In [22]:
require(all(sha256(path) == input_hashes[name] for name, path in paths.items()), 'Original inputs changed during Stage 3A.')
require(all(sha256(path) == stage3_hashes[name] for name, path in stage3_sources.items()), 'Comparison sources changed.')
comparison_specification = {
    'row_key': ['population', 'period', 'model'], 'half_years': PERIODS,
    'primary_model': 'xgboost', 'secondary_model': 'logistic',
    'drift_reference_years': [2015, 2016, 2017], 'separate_population_references': True,
    'signals': [{'feature': f, 'column': rule[0], 'measure': rule[1], 'units': rule[2]}
                for f, rule in SIGNAL_RULES.items()],
    'primary_outcomes': ['roc_auc', 'average_precision', 'brier', 'absolute_risk_gap'],
    'context': ['n', 'positives', 'negatives', 'prevalence', 'mean_predicted_risk', 'risk_gap'],
    'reliability_population_for_both_views': 'eligible',
    'timing': 'Origination drift and the same cohort subsequent 24-month outcomes.',
    'descriptive_reliability_anchor_for_later_steps': '2018-H1',
    'intervals': 'Existing Notebook 11 within-cohort paired loan bootstrap intervals for signed metrics only.',
    'absolute_risk_gap_interval': 'Not computed; signed interval endpoints are not transformed.',
    'unavailable_signals': 'Keep missing with calculation status; never substitute zero.',
    'limitations': [
        'Full sampled originations are not the entire Freddie Mac universe.',
        'Eligible membership uses future observability; compare both population views retrospectively.',
        'Final 2015-2017 development fitting uses 24-month labels, so this is not literal January 2018 deployment.',
        'No correlations, significance tests, baseline-change intervals or prospective validation are computed here.',
        'No deterioration, drift-alert or retraining thresholds are set.'
    ]
}
stage3_validation = {
    'status': 'passed', 'stage': 'Notebook 12 Stage 3A - cohort comparison table only',
    'created_utc': datetime.now(timezone.utc).isoformat(), 'comparison_rows': len(cohort_comparison),
    'selected_signal_rows': len(selected_signal_details), 'half_year_count': len(PERIODS),
    'population_views': ['eligible', 'full_sample'], 'models': FAMILIES,
    'reliability_point_estimates_and_intervals_preserved': True,
    'eligible_reliability_denominators_reconciled': True,
    'selected_signal_values_preserved': True, 'dti_missingness_representations_reconciled': True,
    'unavailable_signal_rows': int(selected_signal_details['signal_value'].isna().sum()),
    'source_files': [{'name': str(path.relative_to(ROOT)), 'sha256': stage3_hashes[name]}
                     for name, path in stage3_sources.items()],
    'original_input_files': validation['input_files'],
    'plots_or_associations_computed': False, 'thresholds_set': False,
}
json.dumps(stage3_validation, allow_nan=False)
stage3_output = DATA / 'notebook_12_comparison' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
stage3_output.mkdir(parents=True, exist_ok=False)
stage3_file_names = ['half_year_drift_reliability_comparison.csv', 'selected_drift_signal_details.csv',
                     'notebook_12_comparison_specification.json', 'notebook_12_stage3_validation.json']
cohort_comparison.to_csv(stage3_output/stage3_file_names[0], index=False)
selected_signal_details.to_csv(stage3_output/stage3_file_names[1], index=False)
(stage3_output/stage3_file_names[2]).write_text(json.dumps(comparison_specification, indent=2, allow_nan=False), encoding='utf-8')
(stage3_output/stage3_file_names[3]).write_text(json.dumps(stage3_validation, indent=2, allow_nan=False), encoding='utf-8')
stage3_bundle = stage3_output/'notebook_12_stage3_review.zip'
with zipfile.ZipFile(stage3_bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in stage3_file_names:
        archive.write(stage3_output/name, arcname=name)
print('Notebook 12 Stage 3A passed.')
print('Summary folder:', stage3_output)
print('Upload this review bundle:', stage3_bundle)


Notebook 12 Stage 3A passed.
Summary folder: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_comparison\20261018T010149208751Z
Upload this review bundle: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_comparison\20261018T010149208751Z\notebook_12_stage3_review.zip


## Stage 3A review completed

The uploaded 48-row comparison table and 120-row signal table passed review on
5 October 2026. Source hashes, selected drift values, eligible denominators, risk-gap
identities and reliability consistency across population views were checked.
Continue to the descriptive plots below; associations remain a later step.


---
# Stage 3B: descriptive cohort plots

Run the preceding stages first. This step produces a seven-page PDF containing:

1. The five selected drift signals across all 12 cohorts and both population views.
2. Absolute reliability and prevalence, one page for each frozen model.
3. Descriptive reliability changes from 2018-H1, one page for each model.
4. Frozen-bin calibration for every half-year, one page for each model.

Numeric drift is scaled using each population's own development IQR. Direction remains
in Stage 2's means, medians, signed changes and distributions; larger distance does not
mean higher credit risk. The full sample is sampled originations, not all Freddie loans.
Reliability in every page is measured on eligible loans.

Absolute reliability pages retain the existing within-cohort bootstrap intervals.
Absolute risk gap has no interval here. Change pages show point estimates only:
within-cohort intervals are not intervals for changes from a shared baseline.
The 2018-H1 anchor is descriptive, not an acceptable-performance threshold.

Calibration panels use Notebook 11's frozen development risk bins. Empty bins are
omitted; each plotted point is labelled with bin loan count and event count. Bin
averages retain Notebook 11's existing Wilson intervals for observed event rates. Connecting points
helps read the risk-group pattern and does not fit a calibration model.

No association statistics, fitted drift-to-performance model, alert thresholds or
retraining rules are computed. Review these plots before proceeding.


In [23]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import PercentFormatter

require(stage3_validation['status'] == 'passed', 'Run and pass Stage 3A first.')
plot_sources = {
    'comparison': stage3_output / 'half_year_drift_reliability_comparison.csv',
    'stage3_validation': stage3_output / 'notebook_12_stage3_validation.json',
    'comparison_specification': stage3_output / 'notebook_12_comparison_specification.json',
    'calibration': paths['calibration'],
}
plot_hashes = {name: sha256(path) for name, path in plot_sources.items()}
plot_table = pd.read_csv(plot_sources['comparison'], float_precision='round_trip')
pd.testing.assert_frame_equal(plot_table, cohort_comparison, check_dtype=False, check_exact=True)
plot_cal = pd.read_csv(plot_sources['calibration'], float_precision='round_trip')
pd.testing.assert_frame_equal(plot_cal, cal, check_dtype=False, check_exact=True)
require({'wilson_low', 'wilson_high'}.issubset(plot_cal.columns), 'Existing calibration Wilson interval columns are missing.')
nonempty = plot_cal['n'].gt(0)
require(np.isfinite(plot_cal.loc[nonempty, ['wilson_low', 'wilson_high']]).all().all() and
        plot_cal.loc[nonempty, 'wilson_low'].le(plot_cal.loc[nonempty, 'wilson_high']).all(),
        'Invalid existing calibration intervals.')
require(len(plot_table) == 48 and not plot_table.duplicated(['population', 'period', 'model']).any(),
        'Plot table coverage differs.')
require(plot_table['reliability_population'].eq('eligible').all(), 'Reliability population changed.')
ANCHOR = '2018-H1'
OUTCOME_COLUMNS = ['roc_auc', 'average_precision', 'brier', 'risk_gap', 'absolute_risk_gap']
change_rows = []
for family in FAMILIES:
    part = plot_table.loc[plot_table['population'].eq('eligible') & plot_table['model'].eq(family)].set_index('period').loc[PERIODS]
    require(len(part) == len(PERIODS), 'Reliability plot cohort coverage differs.')
    for metric in OUTCOME_COLUMNS:
        baseline = float(part.loc[ANCHOR, metric])
        for period in PERIODS:
            value = float(part.loc[period, metric])
            change_rows.append({'period': period, 'model': family, 'metric': metric,
                'anchor_period': ANCHOR, 'anchor_value': baseline, 'cohort_value': value,
                'change_from_anchor': value-baseline,
                'change_status': 'available' if np.isfinite(value) and np.isfinite(baseline) else 'unavailable',
                'interval_status': 'not_computed'})
reliability_changes = pd.DataFrame(change_rows)
require(len(reliability_changes) == 2*12*5 and
        not reliability_changes.duplicated(['period', 'model', 'metric']).any(), 'Change table coverage differs.')
require(reliability_changes.loc[reliability_changes['period'].eq(ANCHOR), 'change_from_anchor'].dropna().eq(0).all(),
        'Anchor does not have zero change.')
for _, group in plot_cal.groupby(['period', 'model']):
    expected = plot_table.loc[plot_table['population'].eq('eligible') & plot_table['period'].eq(group['period'].iloc[0]) &
                             plot_table['model'].eq(group['model'].iloc[0])].iloc[0]
    require(group['n'].sum() == expected['n'] and group['events'].sum() == expected['positives'],
            'Calibration panel counts do not reconcile.')
require(all(sha256(path) == input_hashes[name] for name, path in paths.items()), 'Original inputs changed before plotting.')


In [24]:
MODEL_LABELS = {'xgboost': 'XGBoost (primary)', 'logistic': 'Logistic Regression (secondary)'}
SIGNAL_LABELS = {
    'fico_wasserstein_iqr_scaled': ('FICO distribution', 'Wasserstein / ref. IQR'),
    'dti_wasserstein_iqr_scaled': ('DTI distribution', 'Wasserstein / ref. IQR'),
    'ltv_wasserstein_iqr_scaled': ('LTV distribution', 'Wasserstein / ref. IQR'),
    'channel_total_variation': ('Origination channel mix', 'Total variation (0-1)'),
    'dti_missing_absolute_change_pp': ('DTI missingness', 'Missing-rate change (pp)'),
}
METRIC_LABELS = {'roc_auc': 'ROC-AUC', 'average_precision': 'Average precision', 'brier': 'Brier score',
                 'risk_gap': 'Signed risk gap', 'absolute_risk_gap': 'Absolute risk gap', 'prevalence': 'Event prevalence'}
x_positions = np.arange(len(PERIODS))

def cohort_axis(ax):
    ax.set_xticks(x_positions, PERIODS, rotation=45, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=.25)

def page_finish(fig, title, note):
    fig.suptitle(title, fontsize=16, y=.98)
    fig.text(.04, .015, note, fontsize=8, va='bottom')
    fig.tight_layout(rect=[.025, .075, .98, .95])

def generate_cohort_report(destination):
    pages = []
    with plt.rc_context({'font.size': 10}), PdfPages(destination) as pdf:
        fig, axes = plt.subplots(5, 1, figsize=(11.7, 12), sharex=True)
        for ax, (signal, (label, units)) in zip(axes, SIGNAL_LABELS.items()):
            for population, color, label_pop in [('eligible', '#2166ac', 'Eligible loans'),
                                                 ('full_sample', '#b35806', 'Full sampled originations')]:
                part = plot_table.loc[plot_table['population'].eq(population) & plot_table['model'].eq('xgboost')].set_index('period').loc[PERIODS]
                ax.plot(x_positions, part[signal], marker='o', markersize=4, color=color, label=label_pop)
            ax.set_title(label, loc='left', fontsize=11);ax.set_ylabel(units, fontsize=9);ax.set_ylim(bottom=0)
            cohort_axis(ax)
        axes[0].legend(loc='upper left', fontsize=9)
        page_finish(fig, 'Origination drift: five preselected signals',
            'Separate 2015-2017 references for the two views. Distance measures magnitude, not credit-risk direction.\n'
            'Full sample means sampled files. Missing measurements remain gaps; no alert bands are applied.')
        pdf.savefig(fig);plt.close(fig);pages.append('five_signal_cohort_trends')
        for family in FAMILIES:
            part = plot_table.loc[plot_table['population'].eq('eligible') & plot_table['model'].eq(family)].set_index('period').loc[PERIODS]
            fig, axes = plt.subplots(3, 2, figsize=(11.7, 9))
            for ax, metric in zip(axes.flat, [*OUTCOME_COLUMNS, 'prevalence']):
                values = part[metric].to_numpy(dtype=float)
                ax.plot(x_positions, values, marker='o', markersize=4, color='#2166ac')
                if metric in ['roc_auc', 'average_precision', 'brier', 'risk_gap']:
                    # Draw interval bounds directly; do not assume the estimate lies inside a percentile interval.
                    ax.vlines(x_positions, part[metric+'_ci_low'], part[metric+'_ci_high'], color='#2166ac', alpha=.6)
                if metric in ['prevalence', 'risk_gap', 'absolute_risk_gap']:
                    ax.yaxis.set_major_formatter(PercentFormatter(1))
                if metric == 'risk_gap':ax.axhline(0, color='gray', linewidth=.7)
                ax.set_title(METRIC_LABELS[metric], loc='left', fontsize=11);cohort_axis(ax)
            page_finish(fig, 'Absolute cohort reliability: '+MODEL_LABELS[family],
                'Eligible loans; 24-month outcomes of the same cohort. Vertical lines: existing within-cohort bootstrap intervals.\n'
                'Absolute gap has no interval. AP and Brier depend on prevalence; signed gap > 0 means underprediction.')
            pdf.savefig(fig);plt.close(fig);pages.append('absolute_reliability_'+family)
            fig, axes = plt.subplots(3, 2, figsize=(11.7, 9))
            for ax, metric in zip(list(axes.flat)[:5], OUTCOME_COLUMNS):
                change = reliability_changes.loc[reliability_changes['model'].eq(family) &
                                                reliability_changes['metric'].eq(metric)].set_index('period').loc[PERIODS]
                ax.plot(x_positions, change['change_from_anchor'], marker='o', markersize=4, color='#2166ac')
                ax.axhline(0, color='gray', linewidth=.7);ax.set_title(METRIC_LABELS[metric]+' change', loc='left', fontsize=11)
                ax.set_ylabel('Later minus 2018-H1');cohort_axis(ax)
                if metric in ['risk_gap', 'absolute_risk_gap']:
                    ax.yaxis.set_major_formatter(PercentFormatter(1, symbol=''))
                    ax.set_ylabel('Change (percentage points)')
            axes.flat[5].axis('off')
            axes.flat[5].text(0, .95, 'Reading the changes\n\nROC-AUC / AP: negative = lower metric.\nBrier: positive = higher error.\nAbsolute gap: positive = farther from zero.\nSigned gap: indicates direction of movement.\n\nAP and Brier need prevalence context.\nNo change-confidence intervals are shown.\nThe baseline is not a performance standard.', va='top', fontsize=10, linespacing=1.5)
            page_finish(fig, 'Descriptive changes from 2018-H1: '+MODEL_LABELS[family],
                'Point estimates only; shared-baseline changes are dependent. Zero is a comparison reference, not a deterioration threshold.\n'
                'Changes in signed gap and absolute gap have different meanings. No deterioration decision is made.')
            pdf.savefig(fig);plt.close(fig);pages.append('anchor_changes_'+family)
            fig, axes = plt.subplots(3, 4, figsize=(14, 12))
            famcal = plot_cal.loc[plot_cal['model'].eq(family)]
            observed_max = float(famcal[['mean_predicted', 'observed_rate', 'wilson_high']].max().max())
            upper = min(1., max(.01, observed_max*1.15))
            for ax, period in zip(axes.flat, PERIODS):
                bins = famcal.loc[famcal['period'].eq(period) & famcal['n'].gt(0)].sort_values('bin_index')
                ax.plot([0, upper], [0, upper], color='gray', linestyle='--', linewidth=.8)
                ax.vlines(bins['mean_predicted'], bins['wilson_low'], bins['wilson_high'], color='#2166ac', alpha=.5)
                ax.plot(bins['mean_predicted'], bins['observed_rate'], marker='o', markersize=4, color='#2166ac')
                for row in bins.itertuples(index=False):
                    ax.annotate(f'b{row.bin_index}', (row.mean_predicted, row.observed_rate),
                                xytext=(3, -7), textcoords='offset points', fontsize=6, va='top')
                all_bins = famcal.loc[famcal['period'].eq(period)].sort_values('bin_index')
                table = ax.table(cellText=[all_bins['n'].astype(int).astype(str).tolist(),
                                           all_bins['events'].astype(int).astype(str).tolist()],
                                 rowLabels=['n', 'e'], colLabels=[f'b{j}' for j in all_bins['bin_index']],
                                 cellLoc='center', bbox=[0, -.44, 1, .23])
                table.auto_set_font_size(False);table.set_fontsize(6.5)
                ax.set_xlim(0, upper);ax.set_ylim(0, upper)
                ax.xaxis.set_major_formatter(PercentFormatter(1));ax.yaxis.set_major_formatter(PercentFormatter(1))
                ax.tick_params(labelsize=8);ax.grid(alpha=.2);ax.set_title(period, fontsize=10)
                ax.set_xlabel('Mean predicted risk', fontsize=8);ax.set_ylabel('Observed event rate', fontsize=8)
            page_finish(fig, 'Frozen-bin calibration: '+MODEL_LABELS[family],
                'All 12 eligible cohorts; Notebook 11 development-defined risk bins. e = events; n = loans. Empty bins are omitted.\n'
                'Axes share a scale within this model. Dashed diagonal is equality, not a threshold. Vertical lines retain existing Wilson intervals for observed bin rates.')
            fig.subplots_adjust(top=.91, bottom=.14, left=.07, right=.97, hspace=.95, wspace=.35)
            pdf.savefig(fig);plt.close(fig);pages.append('frozen_bin_calibration_'+family)
    return pages

# The output folder is created only after all input/table checks above pass.
plot_output = DATA / 'notebook_12_plots' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plot_output.mkdir(parents=True, exist_ok=False)
plot_pdf = plot_output / 'notebook_12_cohort_plots.pdf'
plot_pages = generate_cohort_report(plot_pdf)
require(len(plot_pages) == 7, 'Plot report page coverage differs.')
print('Generated all seven descriptive plot pages.')


Generated all seven descriptive plot pages.


In [25]:
require(all(sha256(path) == input_hashes[name] for name, path in paths.items()), 'Original inputs changed during plotting.')
require(all(sha256(path) == plot_hashes[name] for name, path in plot_sources.items()), 'Plot source files changed.')
plot_validation = {
    'status': 'passed', 'stage': 'Notebook 12 Stage 3B - descriptive plots only',
    'created_utc': datetime.now(timezone.utc).isoformat(), 'pages': plot_pages, 'page_count': len(plot_pages),
    'half_years': PERIODS, 'signal_count': len(SIGNAL_LABELS), 'models': FAMILIES,
    'reliability_population': 'eligible', 'anchor_period': ANCHOR,
    'change_rows': len(reliability_changes), 'change_intervals_computed': False,
    'absolute_risk_gap_intervals_computed': False, 'calibration_bin_intervals_reused': 'Notebook 11 Wilson intervals for observed event rates',
    'new_calibration_bin_intervals_computed': False,
    'associations_computed': False, 'thresholds_set': False,
    'source_files': [{'name': str(path.relative_to(ROOT)), 'sha256': plot_hashes[name]}
                     for name, path in plot_sources.items()],
    'limitations': comparison_specification['limitations'] + [
        'Twelve temporal cohorts; duplicated population/model views do not add independent time observations.',
        'AP and Brier require prevalence context; Brier is not purely a calibration measure.',
        'Plots alone do not establish harmlessness, causal effects or validated early-warning prediction.'
    ]
}
plot_file_names = ['notebook_12_cohort_plots.pdf', 'half_year_reliability_changes_from_2018H1.csv',
                   'notebook_12_plot_validation.json']
reliability_changes.to_csv(plot_output/plot_file_names[1], index=False)
(plot_output/plot_file_names[2]).write_text(json.dumps(plot_validation, indent=2, allow_nan=False), encoding='utf-8')
plot_bundle = plot_output/'notebook_12_stage3b_review.zip'
with zipfile.ZipFile(plot_bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in plot_file_names:
        archive.write(plot_output/name, arcname=name)
print('Notebook 12 Stage 3B passed.')
print('Summary folder:', plot_output)
print('Upload this review bundle:', plot_bundle)


Notebook 12 Stage 3B passed.
Summary folder: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_plots\20261018T010157230841Z
Upload this review bundle: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_plots\20261018T010157230841Z\notebook_12_stage3b_review.zip


## Stage 3B review completed

All seven plot pages were reviewed on 5 October 2026. All 120 changes from 2018-H1
were independently checked, and all 24 calibration panels reconciled loan/event counts.
The plots show different patterns for ranking, probability error and calibration.
Continue to the complete exploratory grid below; do not select signals by the results.


---
# Stage 3C: complete exploratory association grid

Calculate **five signals × four absolute outcomes × two models × two population views**:
80 rows, with 12 half-years per comparison. XGBoost is primary; Logistic Regression is
secondary. Keep both views separate: both reuse eligible-loan reliability and do not add
independent temporal observations.

Use Spearman correlation: Pearson correlation of average ranks, preserving ties.
Each half-year has equal weight. Numeric primary signals are IQR-scaled Wasserstein;
channel is total variation; DTI missingness is absolute percentage-point change.
Outcomes are ROC-AUC, average precision, Brier and absolute risk gap.

A negative correlation with ROC-AUC/AP or positive correlation with Brier/absolute gap
is a descriptive association with lower reliability on that outcome. Contrary and weak
patterns must also be reported. Neither sign proves causation or predictive validity.
No ordinary correlation p-values, significance stars, confidence intervals, signal ranking
or thresholds are produced. Shared time trends, 24-month outcome-window overlap and
prevalence changes limit inference from these 12 observations.

Supporting checks (reported separately, never added to the primary grid):

- Leave out each half-year once to expose sensitivity to individual cohorts. This is
  a robustness check, not a confidence interval or chronological prediction test.
- Repeat the three numeric signals with PSI at count additions 0.1, 0.5 and 1.0.
- Verify raw and IQR-scaled Wasserstein give the same ranks when scaling is positive
  and fixed. This equivalence is expected, not independent supporting evidence.
- Describe correlations with prevalence and chronological cohort order to flag
  possible context/trend effects. These do not adjust or isolate a drift effect.

Unavailable/constant comparisons remain explicitly unavailable. Do not replace missing
signals with zero. Annual/quarterly temporal sensitivity and any chronological warning
validation remain later steps; this stage does not complete those parts of the roadmap.


In [26]:
require(plot_validation['status'] == 'passed', 'Run and pass Stage 3B first.')
association_sources = {
    'comparison': stage3_output / 'half_year_drift_reliability_comparison.csv',
    'signal_details': stage3_output / 'selected_drift_signal_details.csv',
    'psi_sensitivity': stage2_output / 'half_year_psi_sensitivity.csv',
    'comparison_specification': stage3_output / 'notebook_12_comparison_specification.json',
}
association_hashes = {name: sha256(path) for name, path in association_sources.items()}
association_table = pd.read_csv(association_sources['comparison'], float_precision='round_trip')
association_details = pd.read_csv(association_sources['signal_details'], float_precision='round_trip')
association_psi = pd.read_csv(association_sources['psi_sensitivity'], float_precision='round_trip')
pd.testing.assert_frame_equal(association_table, cohort_comparison, check_dtype=False, check_exact=True)
pd.testing.assert_frame_equal(association_details, selected_signal_details, check_dtype=False, check_exact=True)
pd.testing.assert_frame_equal(association_psi, psi_sensitivity, check_dtype=False, check_exact=True)
ASSOCIATION_OUTCOMES = ['roc_auc', 'average_precision', 'brier', 'absolute_risk_gap']
ASSOCIATION_SIGNALS = [rule[0] for rule in SIGNAL_RULES.values()]

def exploratory_spearman(x, y):
    x = np.asarray(x, dtype=float);y = np.asarray(y, dtype=float)
    require(x.shape == y.shape and x.ndim == 1, 'Association arrays differ.')
    require(not np.isinf(x).any() and not np.isinf(y).any(), 'Infinite association input.')
    keep = np.isfinite(x) & np.isfinite(y)
    a, b = x[keep], y[keep]
    result = {'n_total': len(x), 'n_pairs': len(a), 'n_unavailable_pairs': int((~keep).sum()),
              'unique_signal_values': len(np.unique(a)), 'unique_outcome_values': len(np.unique(b)),
              'spearman_rho': np.nan, 'status': 'ok'}
    if len(a) < 3:result['status'] = 'insufficient_pairs'
    elif len(np.unique(a)) < 2:result['status'] = 'constant_signal'
    elif len(np.unique(b)) < 2:result['status'] = 'constant_outcome'
    else:
        rank_x = pd.Series(a).rank(method='average').to_numpy()
        rank_y = pd.Series(b).rank(method='average').to_numpy()
        result['spearman_rho'] = float(np.corrcoef(rank_x, rank_y)[0, 1])
    return result

association_rows, leave_out_rows, sensitivity_rows, context_rows = [], [], [], []
for population in ['eligible', 'full_sample']:
    for family in FAMILIES:
        part = association_table.loc[association_table['population'].eq(population) &
                                     association_table['model'].eq(family)].set_index('period')
        require(not part.index.duplicated().any() and set(part.index) == set(PERIODS), 'Association cohort coverage differs.')
        part = part.loc[PERIODS]
        for signal in ASSOCIATION_SIGNALS:
            for outcome in ASSOCIATION_OUTCOMES:
                identity = {'population': population, 'model': family, 'signal': signal, 'outcome': outcome}
                full = exploratory_spearman(part[signal], part[outcome])
                association_rows.append({**identity, **full})
                for omitted in PERIODS:
                    smaller = part.drop(index=omitted)
                    estimate = exploratory_spearman(smaller[signal], smaller[outcome])
                    leave_out_rows.append({**identity, 'omitted_period': omitted, **estimate,
                        'full_spearman_rho': full['spearman_rho'],
                        'rho_change_from_full': estimate['spearman_rho']-full['spearman_rho']})
        for feature in ['FICO_SCORE', 'ORIGINAL_DTI', 'ORIGINAL_LTV']:
            raw = association_details.loc[association_details['population'].eq(population) &
                                          association_details['feature'].eq(feature)].set_index('period').loc[PERIODS]
            for outcome in ASSOCIATION_OUTCOMES:
                identity = {'population': population, 'model': family, 'feature': feature, 'outcome': outcome}
                scaled = exploratory_spearman(part[SIGNAL_RULES[feature][0]], part[outcome])
                raw_result = exploratory_spearman(raw['wasserstein_raw'], part[outcome])
                if raw['reference_iqr'].gt(0).all() and raw['reference_iqr'].nunique() == 1:
                    require(raw_result['status'] == scaled['status'] and
                            np.isclose(raw_result['spearman_rho'], scaled['spearman_rho'], equal_nan=True),
                            'Raw/scaled Wasserstein rank equivalence failed.')
                sensitivity_rows.append({**identity, 'alternative_measure': 'wasserstein_raw',
                                         'smoothing_count_addition': np.nan, **raw_result})
                for addition in SMOOTHING:
                    alt = association_psi.loc[association_psi['population'].eq(population) &
                        association_psi['feature'].eq(feature) & association_psi['smoothing_count_addition'].eq(addition)].set_index('period')
                    require(not alt.index.duplicated().any() and set(alt.index) == set(PERIODS), 'PSI association coverage differs.')
                    sensitivity_rows.append({**identity, 'alternative_measure': 'psi',
                        'smoothing_count_addition': addition, **exploratory_spearman(alt.loc[PERIODS, 'psi'], part[outcome])})
        # Outcomes/prevalence are identical across views; report their context only once.
        if population == 'eligible':
            for outcome in ASSOCIATION_OUTCOMES:
                for context in ['prevalence', 'cohort_order']:
                    values = part['prevalence'] if context == 'prevalence' else np.arange(len(PERIODS))
                    context_rows.append({'population': 'eligible', 'model': family, 'quantity': outcome,
                        'context': context, **exploratory_spearman(part[outcome], values)})
    # Drift signals are identical across model rows; report their context only once.
    signal_part = association_table.loc[association_table['population'].eq(population) &
                                       association_table['model'].eq('xgboost')].set_index('period').loc[PERIODS]
    for signal in ASSOCIATION_SIGNALS:
        for context in ['prevalence', 'cohort_order']:
            values = signal_part['prevalence'] if context == 'prevalence' else np.arange(len(PERIODS))
            context_rows.append({'population': population, 'model': 'not_model_specific', 'quantity': signal,
                'context': context, **exploratory_spearman(signal_part[signal], values)})

association_grid = pd.DataFrame(association_rows)
leave_one_cohort_out = pd.DataFrame(leave_out_rows)
association_sensitivity = pd.DataFrame(sensitivity_rows)
association_context = pd.DataFrame(context_rows)
require(len(association_grid) == 80 and len(leave_one_cohort_out) == 960 and
        len(association_sensitivity) == 192 and len(association_context) == 36, 'Association table row coverage differs.')
require(not association_grid.duplicated(['population', 'model', 'signal', 'outcome']).any(), 'Duplicate primary association.')
for frame in [association_grid, leave_one_cohort_out, association_sensitivity, association_context]:
    require(frame['spearman_rho'].dropna().between(-1-1e-12, 1+1e-12).all(), 'Correlation outside valid bounds.')
    require(frame.loc[~frame['status'].eq('ok'), 'spearman_rho'].isna().all(), 'Unavailable correlation must remain missing.')
display(association_grid)
print('Complete 80-row exploratory grid computed; all comparisons retained.')


,population,model,signal,outcome,n_total,n_pairs,n_unavailable_pairs,unique_signal_values,unique_outcome_values,spearman_rho,status
0,eligible,xgboost,fico_wasserstein_iqr_scaled,roc_auc,12,12,0,12,12,-0.090909,ok
1,eligible,xgboost,fico_wasserstein_iqr_scaled,average_precision,12,12,0,12,12,-0.279720,ok
2,eligible,xgboost,fico_wasserstein_iqr_scaled,brier,12,12,0,12,12,-0.237762,ok
3,eligible,xgboost,fico_wasserstein_iqr_scaled,absolute_risk_gap,12,12,0,12,12,-0.195804,ok
4,eligible,xgboost,dti_wasserstein_iqr_scaled,roc_auc,12,12,0,12,12,0.405594,ok
...,...,...,...,...,...,...,...,...,...,...,...
75,full_sample,logistic,channel_total_variation,absolute_risk_gap,12,12,0,12,12,0.104895,ok
76,full_sample,logistic,dti_missing_absolute_change_pp,roc_auc,12,12,0,6,12,0.094982,ok
77,full_sample,logistic,dti_missing_absolute_change_pp,average_precision,12,12,0,6,12,0.131513,ok
78,full_sample,logistic,dti_missing_absolute_change_pp,brier,12,12,0,6,12,0.102288,ok


Complete 80-row exploratory grid computed; all comparisons retained.


In [27]:
# Summarize influence without labelling any result stable/unstable using a cutoff.
influence_rows = []
for key, group in leave_one_cohort_out.groupby(['population', 'model', 'signal', 'outcome'], sort=False):
    full = float(group['full_spearman_rho'].iloc[0])
    valid = group.loc[group['status'].eq('ok')].copy()
    nonzero = valid['spearman_rho'].ne(0) & np.isfinite(full) & (full != 0)
    opposite = int((np.sign(valid.loc[nonzero, 'spearman_rho']) != np.sign(full)).sum()) if len(valid) else 0
    influence_rows.append(dict(zip(['population', 'model', 'signal', 'outcome'], key),
        full_spearman_rho=full, valid_leave_out_runs=len(valid), unavailable_leave_out_runs=len(group)-len(valid),
        leave_out_rho_min=float(valid['spearman_rho'].min()) if len(valid) else np.nan,
        leave_out_rho_max=float(valid['spearman_rho'].max()) if len(valid) else np.nan,
        max_absolute_rho_change=float(valid['rho_change_from_full'].abs().max()) if len(valid) else np.nan,
        opposite_nonzero_sign_runs=opposite,
        zero_rho_runs=int(valid['spearman_rho'].eq(0).sum())))
association_influence_summary = pd.DataFrame(influence_rows)
require(len(association_influence_summary) == 80, 'Influence summary coverage differs.')
require(all(sha256(path) == input_hashes[name] for name, path in paths.items()), 'Original inputs changed during associations.')
require(all(sha256(path) == association_hashes[name] for name, path in association_sources.items()), 'Association sources changed.')
association_output = DATA / 'notebook_12_associations' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
association_output.mkdir(parents=True, exist_ok=False)
association_pdf = association_output / 'exploratory_association_grid.pdf'
with PdfPages(association_pdf) as pdf:
    for population in ['eligible', 'full_sample']:
        for family in FAMILIES:
            group = association_grid.loc[association_grid['population'].eq(population) & association_grid['model'].eq(family)]
            matrix = group.pivot(index='signal', columns='outcome', values='spearman_rho').loc[ASSOCIATION_SIGNALS, ASSOCIATION_OUTCOMES]
            fig, ax = plt.subplots(figsize=(11.7, 8.3))
            im = ax.imshow(np.ma.masked_invalid(matrix.to_numpy()), vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
            ax.set_xticks(np.arange(4), [METRIC_LABELS[x] for x in ASSOCIATION_OUTCOMES], fontsize=11)
            ax.set_yticks(np.arange(5), [SIGNAL_LABELS[x][0] for x in ASSOCIATION_SIGNALS], fontsize=11)
            for i, signal in enumerate(ASSOCIATION_SIGNALS):
                for j, outcome in enumerate(ASSOCIATION_OUTCOMES):
                    row = group.loc[group['signal'].eq(signal) & group['outcome'].eq(outcome)].iloc[0]
                    rho = row['spearman_rho']
                    label = f'{rho:+.2f}\nn={row["n_pairs"]}' if np.isfinite(rho) else 'Unavailable'
                    ax.text(j, i, label, ha='center', va='center', fontsize=13,
                            color='white' if np.isfinite(rho) and abs(rho) > .65 else 'black')
            fig.colorbar(im, ax=ax, label='Spearman correlation (signed)', fraction=.04, pad=.04)
            page_finish(fig, 'Exploratory grid: '+MODEL_LABELS[family]+' / '+population,
                'All preselected comparisons. Negative ROC-AUC/AP or positive Brier/absolute gap describes lower reliability on that outcome.\n'
                '12 temporal cohorts; no p-values or confidence intervals. Colour shows signed correlation, not importance or a retraining decision.\n'
                'Shared time trends and prevalence can influence results. Full-sample drift is paired with eligible-loan reliability.')
            fig.subplots_adjust(left=.26, right=.88, top=.89, bottom=.17)
            pdf.savefig(fig);plt.close(fig)
association_validation = {
    'status': 'passed', 'stage': 'Notebook 12 Stage 3C - exploratory associations',
    'created_utc': datetime.now(timezone.utc).isoformat(), 'primary_grid_rows': len(association_grid),
    'leave_one_cohort_out_rows': len(leave_one_cohort_out), 'influence_summary_rows': len(association_influence_summary),
    'alternative_measure_rows': len(association_sensitivity), 'context_rows': len(association_context),
    'method': 'Pearson correlation of average ranks (Spearman); equal weight per half-year; pairwise available observations.',
    'main_signals': ASSOCIATION_SIGNALS, 'absolute_outcomes': ASSOCIATION_OUTCOMES, 'half_years': PERIODS,
    'status_counts': {str(k): int(v) for k, v in association_grid['status'].value_counts().items()},
    'p_values_or_confidence_intervals_computed': False, 'thresholds_set': False,
    'temporal_annual_quarterly_sensitivity_completed': False, 'chronological_prediction_validation_completed': False,
    'source_files': [{'name': str(path.relative_to(ROOT)), 'sha256': association_hashes[name]}
                     for name, path in association_sources.items()],
    'limitations': [line for line in plot_validation['limitations'] if line !=
        'No correlations, significance tests, baseline-change intervals or prospective validation are computed here.'] + [
        'Exploratory correlations are computed; significance tests, baseline-change intervals and prospective validation remain unimplemented.',
        'Leave-one-cohort-out ranges are influence checks, not confidence intervals.',
        'Prevalence/time context correlations do not adjust for confounding or isolate population-drift effects.',
        'Raw and IQR-scaled Wasserstein have identical ranks under positive fixed scaling; their agreement is not independent validation.',
        'Main signal choices preceded association testing but some were informed by the descriptive population audit.'
    ]
}
exports = {
    'exploratory_association_grid.csv': association_grid,
    'leave_one_cohort_out.csv': leave_one_cohort_out,
    'association_influence_summary.csv': association_influence_summary,
    'alternative_measure_associations.csv': association_sensitivity,
    'prevalence_and_time_context.csv': association_context,
}
for name, frame in exports.items():frame.to_csv(association_output/name, index=False)
(association_output/'notebook_12_association_validation.json').write_text(json.dumps(association_validation, indent=2, allow_nan=False), encoding='utf-8')
association_bundle = association_output/'notebook_12_stage3c_review.zip'
with zipfile.ZipFile(association_bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in [*exports, 'exploratory_association_grid.pdf', 'notebook_12_association_validation.json']:
        archive.write(association_output/name, arcname=name)
print('Notebook 12 Stage 3C passed.')
print('Summary folder:', association_output)
print('Upload this review bundle:', association_bundle)


Notebook 12 Stage 3C passed.
Summary folder: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_associations\20261018T010215495232Z
Upload this review bundle: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_associations\20261018T010215495232Z\notebook_12_stage3c_review.zip


## Stage 3C review point

Upload `notebook_12_stage3c_review.zip`. Review the full grid, cohort influence, measure
sensitivity and prevalence/time context together. No signal will be chosen solely by the
largest correlation. Annual/quarterly sensitivity remains pending, as does chronological
prediction validation. Do not infer harmless drift, causation or retraining need here.


---
# Stage 4: annual context and quarterly drift sensitivity

The 8 October project extension records review through Stage 3C. This new section
checks temporal sensitivity; its outputs have **not yet been run or reviewed on real data**.

**Run the existing notebook cells in order in your local repository, then run these cells.**
If Stages 1–3C already ran in the current kernel, run only this section. Do not skip
prerequisites after a kernel restart. Earlier review ZIPs remain valid; new runs create
new timestamped folders. This section preserves earlier cells and input artifacts.

Recalculate all 29 predictors from six annual and 24 quarterly cohorts, retaining
both population views and the exact frozen Stage 2 references. The five selected
signals and four absolute reliability outcomes remain unchanged. Half-years remain
primary; annual and quarterly results are sensitivity/context, not independent replication.

Join matching Notebook 11 annual/quarterly reliability, already reconciled to loan-level
predictions in Stage 1. Do not average half-year PSI, Wasserstein, ROC-AUC or AP.
No models are fitted, loaded or rescored. No alert thresholds, bootstrap intervals,
early-warning validation or retraining decisions are added.


In [28]:
require(association_validation['status'] == 'passed', 'Run and pass Stage 3C in this kernel first.')
require(all(sha256(path) == input_hashes[name] for name, path in paths.items()), 'Original inputs changed.')
require(all(sha256(path) == association_hashes[name] for name, path in association_sources.items()),
        'Stage 3C sources changed.')
require(set(SIGNAL_RULES) == main_signals and len(SIGNAL_RULES) == 5, 'Selected signals changed.')
stage4_reference_text = json.dumps(reference_payload, sort_keys=True, allow_nan=False)
stage4_reference_hash = hashlib.sha256(stage4_reference_text.encode()).hexdigest()
TEMPORAL_PERIODS = {
    'annual': [str(year) for year in range(2018, 2024)],
    'quarter': [f'{year}-Q{q}' for year in range(2018, 2024) for q in range(1, 5)],
}
def temporal_labels(frame, kind):
    year = frame['SOURCE YEAR'].astype(int).astype(str)
    q = pd.to_numeric(frame['ORIGINATION QUARTER'].astype('string').str.removeprefix('Q'), errors='raise')
    require(q.isin([1, 2, 3, 4]).all(), 'Invalid quarter in temporal sensitivity.')
    if kind == 'annual': return year
    if kind == 'quarter': return year + '-Q' + q.astype(int).astype(str)
    if kind == 'half_year': return year + '-' + np.where(q <= 2, 'H1', 'H2')
    raise ValueError('Unknown temporal unit: ' + kind)


In [29]:
def calculate_temporal_drift(drift_populations, PERIODS, expected_counts):
    rows, psi_rows, bin_rows, category_rows = [], [], [], []
    for population, frame in drift_populations.items():
        require(set(frame.loc[frame['SOURCE YEAR'].ge(2018), 'period']) == set(PERIODS), f'{population}: cohort coverage differs.')
        for period in PERIODS:
            current = frame.loc[frame['period'].eq(period)]
            expected_n = int(expected_counts.loc[period, 'included_loans' if population == 'eligible' else 'origination_loans'])
            require(len(current) == expected_n, f'{population}/{period}: sample count differs.')
            for feature in PREDICTOR_COLUMNS:
                spec = reference_specs[population]['features'][feature]
                row = {'population': population, 'period': period, 'feature': feature, 'type': spec['type'],
                       'reference_n': int(reference_specs[population]['reference_rows']), 'current_n': len(current),
                       'main_association_signal': feature in main_signals, 'status': 'ok',
                       'psi': np.nan, 'wasserstein_raw': np.nan, 'wasserstein_iqr_scaled': np.nan,
                       'wasserstein_scaled_status': 'not_applicable', 'total_variation': np.nan,
                       'binary_signed_change_pp': np.nan, 'binary_absolute_change_pp': np.nan,
                       'reference_missing_percentage': np.nan, 'current_missing_percentage': np.nan,
                       'missing_signed_change_pp': np.nan, 'missing_absolute_change_pp': np.nan,
                       'unseen_category_percentage': np.nan, 'constant_changed_percentage': np.nan}
                if spec['type'] == 'numeric':
                    values = observed_numeric(current[feature])
                    row.update(reference_observed_n=spec['observed_n'], current_observed_n=len(values),
                               reference_mean=spec['mean'], current_mean=float(values.mean()) if len(values) else np.nan,
                               reference_median=spec['median'], current_median=float(np.median(values)) if len(values) else np.nan,
                               reference_iqr=spec['iqr'], bin_count=spec['bin_count'],
                               empty_reference_bins=spec['empty_reference_bins'],
                               reference_missing_percentage=spec['missing_percentage'],
                               current_missing_percentage=100*(len(current)-len(values))/len(current))
                    change = row['current_missing_percentage'] - row['reference_missing_percentage']
                    row.update(missing_signed_change_pp=change, missing_absolute_change_pp=abs(change))
                    if not spec['observed_n']:
                        row['status'] = 'reference_no_observations'
                        row['wasserstein_scaled_status'] = 'reference_no_observations'
                    elif not len(values):
                        row['status'] = 'comparison_no_observations'
                        row['wasserstein_scaled_status'] = 'comparison_no_observations'
                    else:
                        row['wasserstein_raw'] = float(wasserstein_distance(reference_values[population][feature], values))
                        if spec['iqr'] > 0:
                            row['wasserstein_iqr_scaled'] = row['wasserstein_raw'] / spec['iqr']
                            row['wasserstein_scaled_status'] = 'ok'
                        else:
                            row['wasserstein_scaled_status'] = 'reference_iqr_zero'
                        if spec['status'] == 'reference_constant':
                            row['status'] = 'reference_constant'
                            row['constant_changed_percentage'] = 100*float(np.mean(values != spec['constant_value']))
                        else:
                            counts = bin_counts(values, spec['cuts'])
                            require(int(counts.sum()) == len(values), 'Numeric bins lost observations.')
                            row['psi'] = psi_from_counts(spec['reference_bin_counts'], counts, PRIMARY_SMOOTHING)
                            boundaries = spec['bin_boundaries']
                            for j, (ref_count, count) in enumerate(zip(spec['reference_bin_counts'], counts)):
                                bin_rows.append({'population': population, 'period': period, 'feature': feature,
                                    'bin_index': j+1, 'lower_edge': boundaries[j], 'upper_edge': boundaries[j+1],
                                    'reference_count': ref_count, 'reference_observed_n': spec['observed_n'],
                                    'reference_proportion': ref_count/spec['observed_n'], 'current_count': int(count),
                                    'current_observed_n': len(values), 'current_proportion': int(count)/len(values)})
                    for addition in SMOOTHING:
                        value = psi_from_counts(spec['reference_bin_counts'], bin_counts(values, spec['cuts']), addition) if row['status'] == 'ok' else np.nan
                        psi_rows.append({'population': population, 'period': period, 'feature': feature,
                                         'smoothing_count_addition': addition, 'psi': value, 'status': row['status']})
                elif spec['type'] == 'binary':
                    one = float(current[feature].mean())
                    change = 100*(one-spec['proportion_one'])
                    row.update(reference_proportion_one=spec['proportion_one'], current_proportion_one=one,
                               binary_signed_change_pp=change, binary_absolute_change_pp=abs(change))
                else:
                    ref_counts = {level['token']: level['count'] for level in spec['levels']}
                    counts = category_tokens(current[feature]).value_counts().to_dict()
                    union = sorted(set(ref_counts) | set(counts))
                    require(sum(counts.values()) == len(current), 'Categorical counts lost observations.')
                    difference_sum = 0.0
                    unseen_count = 0
                    for token in union:
                        reference_count = ref_counts.get(token, 0)
                        count = int(counts.get(token, 0))
                        ref_prop, cur_prop = reference_count/spec['n'], count/len(current)
                        novel = token != MISSING_TOKEN and token not in ref_counts
                        unseen_count += count if novel else 0
                        difference_sum += abs(cur_prop-ref_prop)
                        category_rows.append({'population': population, 'period': period, 'feature': feature,
                            **category_description(token), 'reference_count': reference_count, 'reference_n': spec['n'],
                            'reference_percentage': 100*ref_prop, 'current_count': count, 'current_n': len(current),
                            'current_percentage': 100*cur_prop, 'signed_change_pp': 100*(cur_prop-ref_prop),
                            'unseen_in_reference': novel})
                    missing_ref = 100*ref_counts.get(MISSING_TOKEN, 0)/spec['n']
                    missing_cur = 100*counts.get(MISSING_TOKEN, 0)/len(current)
                    row.update(total_variation=0.5*difference_sum, unseen_category_percentage=100*unseen_count/len(current),
                               reference_missing_percentage=missing_ref, current_missing_percentage=missing_cur,
                               missing_signed_change_pp=missing_cur-missing_ref, missing_absolute_change_pp=abs(missing_cur-missing_ref))
                rows.append(row)
            print('Calculated feature drift:', population, period)

    feature_drift = pd.DataFrame(rows)
    psi_sensitivity = pd.DataFrame(psi_rows)
    numeric_bins = pd.DataFrame(bin_rows, columns=['population', 'period', 'feature', 'bin_index', 'lower_edge',
        'upper_edge', 'reference_count', 'reference_observed_n', 'reference_proportion',
        'current_count', 'current_observed_n', 'current_proportion'])
    category_proportions = pd.DataFrame(category_rows)
    return feature_drift, psi_sensitivity, numeric_bins, category_proportions


In [30]:
# A runtime regression check confirms the reused calculation agrees with Stage 2.
check_periods = PERIODS[:1]
# Restrict source frames to development plus the first evaluation half-year.
check_populations = {name: frame.loc[frame['SOURCE YEAR'].le(2017) | frame['period'].isin(check_periods)].copy()
                     for name, frame in drift_populations.items()}
check_drift, _, _, _ = calculate_temporal_drift(check_populations, check_periods,
                                               population_counts.set_index('period'))
check_columns = sorted(feature_drift.columns)
check_keys = ['population', 'period', 'feature']
pd.testing.assert_frame_equal(
    check_drift.sort_values(check_keys)[check_columns].reset_index(drop=True),
    feature_drift.loc[feature_drift['period'].isin(check_periods)].sort_values(check_keys)[check_columns].reset_index(drop=True),
    check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-15)
del check_populations, check_drift

stage4_parts = {'drift': [], 'psi': [], 'bins': [], 'categories': [], 'counts': []}
for kind, periods in TEMPORAL_PERIODS.items():
    counts_source = originations[['SOURCE YEAR', 'ORIGINATION QUARTER', 'included']].copy()
    counts_source['period'] = temporal_labels(counts_source, kind)
    counts = counts_source.loc[counts_source['SOURCE YEAR'].ge(2018)].groupby('period').agg(
        origination_loans=('included', 'size'), included_loans=('included', 'sum')).reindex(periods)
    require(not counts.isna().any().any(), 'Missing temporal population counts.')
    counts['excluded_loans'] = counts['origination_loans'] - counts['included_loans']
    counts['inclusion_percentage'] = 100 * counts['included_loans'] / counts['origination_loans']
    require(int(counts['origination_loans'].sum()) == 300000 and int(counts['included_loans'].sum()) == 294070,
            'Temporal sample totals do not reconcile.')
    rel = summary.loc[summary['period_kind'].eq(kind)]
    for row in rel.itertuples(index=False):
        require(int(counts.loc[row.period, 'included_loans']) == row.n, 'Eligible/reliability counts differ.')
    current_populations = {}
    for name, frame in drift_populations.items():
        current_populations[name] = frame.copy()
        current_populations[name]['period'] = temporal_labels(frame, kind).to_numpy()
    outputs = calculate_temporal_drift(current_populations, periods, counts)
    for label, frame in zip(['drift', 'psi', 'bins', 'categories'], outputs):
        frame.insert(0, 'period_kind', kind)
        stage4_parts[label].append(frame)
    stage4_parts['counts'].append(counts.reset_index().assign(period_kind=kind))
    del current_populations
stage4_drift = pd.concat(stage4_parts['drift'], ignore_index=True)
stage4_psi = pd.concat(stage4_parts['psi'], ignore_index=True)
stage4_bins = pd.concat(stage4_parts['bins'], ignore_index=True)
stage4_categories = pd.concat(stage4_parts['categories'], ignore_index=True)
stage4_counts = pd.concat(stage4_parts['counts'], ignore_index=True)
require(len(stage4_drift) == 2 * 30 * 29, 'Expected 1,740 drift rows.')
require(len(stage4_psi) == 2 * 30 * len(CONTINUOUS) * len(SMOOTHING), 'Expected 1,260 PSI sensitivity rows.')
require(not stage4_drift.duplicated(['period_kind', 'population', 'period', 'feature']).any(), 'Duplicate drift keys.')
for row in stage4_bins.groupby(['period_kind', 'population', 'period', 'feature']):
    key, bins = row
    record = stage4_drift.set_index(['period_kind', 'population', 'period', 'feature']).loc[key]
    require(int(bins['current_count'].sum()) == record['current_observed_n'], 'Numeric bin totals differ.')
    require(np.isclose(psi_from_counts(bins['reference_count'], bins['current_count'], PRIMARY_SMOOTHING), record['psi']),
            'Numeric counts do not reproduce PSI.')
for key, levels in stage4_categories.groupby(['period_kind', 'population', 'period', 'feature']):
    record = stage4_drift.set_index(['period_kind', 'population', 'period', 'feature']).loc[key]
    require(int(levels['current_count'].sum()) == record['current_n'], 'Category totals differ.')
    require(np.isclose(0.5 * levels['signed_change_pp'].abs().sum() / 100, record['total_variation']),
            'Category proportions do not reproduce total variation.')
print('Annual and quarterly drift checks passed; frozen references retained.')


Calculated feature drift: eligible 2018-H1
Calculated feature drift: full_sample 2018-H1
Calculated feature drift: eligible 2018
Calculated feature drift: eligible 2019
Calculated feature drift: eligible 2020
Calculated feature drift: eligible 2021
Calculated feature drift: eligible 2022
Calculated feature drift: eligible 2023
Calculated feature drift: full_sample 2018
Calculated feature drift: full_sample 2019
Calculated feature drift: full_sample 2020
Calculated feature drift: full_sample 2021
Calculated feature drift: full_sample 2022
Calculated feature drift: full_sample 2023
Calculated feature drift: eligible 2018-Q1
Calculated feature drift: eligible 2018-Q2
Calculated feature drift: eligible 2018-Q3
Calculated feature drift: eligible 2018-Q4
Calculated feature drift: eligible 2019-Q1
Calculated feature drift: eligible 2019-Q2
Calculated feature drift: eligible 2019-Q3
Calculated feature drift: eligible 2019-Q4
Calculated feature drift: eligible 2020-Q1
Calculated feature drift: 

## Join matching reliability and retain event support

The reliability population is eligible loans for both views. Full-sample drift therefore
retains the recorded population mismatch. Signed risk gap remains observed rate minus
mean predicted risk; absolute gap measures the magnitude only.

Export event and non-event counts without imposing a new low-event cutoff. A cohort
with only one observed class has unavailable ROC-AUC and AP. Existing estimates with
both classes are retained, even with few events, and must be interpreted with the counts.
Annual/quarterly intervals have not been calculated; half-year intervals cannot be reused.


In [31]:
stage4_details = stage4_drift.loc[stage4_drift['feature'].isin(SIGNAL_RULES)].copy()
require(len(stage4_details) == 300, 'Expected 300 selected-signal detail rows.')
signal_rows = []
for (kind, population, period), group in stage4_details.groupby(['period_kind', 'population', 'period']):
    require(set(group['feature']) == set(SIGNAL_RULES), 'Missing selected temporal signal.')
    record = {'period_kind': kind, 'population': population, 'period': period}
    for feature, (signal, measure, units) in SIGNAL_RULES.items():
        detail = group.loc[group['feature'].eq(feature)].iloc[0]
        record[signal] = detail[measure]
        record[signal + '_status'] = (detail['wasserstein_scaled_status']
                                      if measure == 'wasserstein_iqr_scaled' else detail['status'])
    signal_rows.append(record)
reliability = summary.loc[summary['period_kind'].isin(TEMPORAL_PERIODS)].copy()
require(len(reliability) == 60, 'Expected 60 temporal reliability rows.')
reliability['absolute_risk_gap'] = reliability['risk_gap'].abs()
reliability['both_classes_observed'] = reliability['positives'].gt(0) & reliability['negatives'].gt(0)
reliability['ranking_status'] = np.where(reliability['both_classes_observed'], 'both_classes_observed', 'single_class_unavailable')
require(reliability.loc[~reliability['both_classes_observed'], ['roc_auc', 'average_precision']].isna().all().all(),
        'Single-class ranking metrics must be unavailable.')
reliability['interval_status'] = 'annual_quarterly_intervals_not_computed'
stage4_comparison = pd.DataFrame(signal_rows).merge(reliability, on=['period_kind', 'period'], how='left', validate='many_to_many')
require(len(stage4_comparison) == 120 and stage4_comparison['n'].notna().all(), 'Expected 120 complete comparison rows.')
require(not stage4_comparison.duplicated(['period_kind', 'population', 'period', 'model']).any(), 'Duplicate comparison keys.')
require(stage4_comparison.groupby(['period_kind', 'period', 'model'])['prevalence'].nunique().eq(1).all(),
        'Reliability differs across population views.')
stage4_comparison['reliability_population'] = 'eligible'
stage4_comparison['population_mismatch'] = stage4_comparison['population'].eq('full_sample')
display(reliability[['period_kind', 'period', 'model', 'n', 'positives', 'negatives', 'prevalence', 'ranking_status']])

# Complete grid at each resolution, retaining unavailable/constant comparisons.
temporal_associations, temporal_context = [], []
for kind, periods in TEMPORAL_PERIODS.items():
    for population in ['eligible', 'full_sample']:
        for family in FAMILIES:
            part = stage4_comparison.loc[stage4_comparison['period_kind'].eq(kind) &
                stage4_comparison['population'].eq(population) & stage4_comparison['model'].eq(family)].set_index('period').loc[periods]
            for signal in ASSOCIATION_SIGNALS:
                for outcome in ASSOCIATION_OUTCOMES:
                    temporal_associations.append({'period_kind': kind, 'population': population, 'model': family,
                        'signal': signal, 'outcome': outcome, **exploratory_spearman(part[signal], part[outcome])})
            if population == 'eligible':
                for outcome in ASSOCIATION_OUTCOMES:
                    for context in ['prevalence', 'cohort_order']:
                        values = part['prevalence'] if context == 'prevalence' else np.arange(len(periods))
                        temporal_context.append({'period_kind': kind, 'population': population, 'model': family,
                            'quantity': outcome, 'context': context, **exploratory_spearman(part[outcome], values)})
        signal_part = stage4_comparison.loc[stage4_comparison['period_kind'].eq(kind) &
            stage4_comparison['population'].eq(population) & stage4_comparison['model'].eq('xgboost')].set_index('period').loc[periods]
        for signal in ASSOCIATION_SIGNALS:
            for context in ['prevalence', 'cohort_order']:
                values = signal_part['prevalence'] if context == 'prevalence' else np.arange(len(periods))
                temporal_context.append({'period_kind': kind, 'population': population, 'model': 'not_model_specific',
                    'quantity': signal, 'context': context, **exploratory_spearman(signal_part[signal], values)})
stage4_associations = pd.DataFrame(temporal_associations)
stage4_context = pd.DataFrame(temporal_context)
require(len(stage4_associations) == 160 and len(stage4_context) == 72, 'Association/context grid coverage differs.')
for frame in [stage4_associations, stage4_context]:
    require(frame['spearman_rho'].dropna().between(-1-1e-12, 1+1e-12).all(), 'Invalid correlation.')
    require(frame.loc[~frame['status'].eq('ok'), 'spearman_rho'].isna().all(), 'Unavailable association replaced with a value.')
# Direct side-by-side comparison: six annual, twelve half-year, 24 quarterly cohorts.
stage4_resolution_comparison = pd.concat([
    association_grid.assign(period_kind='half_year'), stage4_associations], ignore_index=True)
require(len(stage4_resolution_comparison) == 240, 'Expected 240 resolution comparison rows.')
display(stage4_associations)


,period_kind,period,model,n,positives,negatives,prevalence,ranking_status
24,annual,2018,xgboost,47417,1222,46195,0.025771,both_classes_observed
25,annual,2018,logistic,47417,1222,46195,0.025771,both_classes_observed
26,annual,2019,xgboost,48611,2227,46384,0.045813,both_classes_observed
27,annual,2019,logistic,48611,2227,46384,0.045813,both_classes_observed
28,annual,2020,xgboost,49665,802,48863,0.016148,both_classes_observed
29,annual,2020,logistic,49665,802,48863,0.016148,both_classes_observed
30,annual,2021,xgboost,49490,451,49039,0.009113,both_classes_observed
31,annual,2021,logistic,49490,451,49039,0.009113,both_classes_observed
32,annual,2022,xgboost,49386,903,48483,0.018285,both_classes_observed
33,annual,2022,logistic,49386,903,48483,0.018285,both_classes_observed


,period_kind,population,model,signal,outcome,n_total,n_pairs,n_unavailable_pairs,unique_signal_values,unique_outcome_values,spearman_rho,status
0,annual,eligible,xgboost,fico_wasserstein_iqr_scaled,roc_auc,6,6,0,6,6,0.371429,ok
1,annual,eligible,xgboost,fico_wasserstein_iqr_scaled,average_precision,6,6,0,6,6,-0.771429,ok
2,annual,eligible,xgboost,fico_wasserstein_iqr_scaled,brier,6,6,0,6,6,-0.657143,ok
3,annual,eligible,xgboost,fico_wasserstein_iqr_scaled,absolute_risk_gap,6,6,0,6,6,-0.428571,ok
4,annual,eligible,xgboost,dti_wasserstein_iqr_scaled,roc_auc,6,6,0,6,6,0.371429,ok
...,...,...,...,...,...,...,...,...,...,...,...,...
155,quarter,full_sample,logistic,channel_total_variation,absolute_risk_gap,24,24,0,24,24,0.146957,ok
156,quarter,full_sample,logistic,dti_missing_absolute_change_pp,roc_auc,24,24,0,10,24,0.104976,ok
157,quarter,full_sample,logistic,dti_missing_absolute_change_pp,average_precision,24,24,0,10,24,0.073346,ok
158,quarter,full_sample,logistic,dti_missing_absolute_change_pp,brier,24,24,0,10,24,-0.004126,ok


## Export and stop for review

Review all planned comparisons, counts, unavailable metrics and prevalence/time context.
Six annual points offer limited evidence; 24 quarters do not create independent observations
because the 24-month outcome windows overlap. Agreement across resolutions is sensitivity,
not new validation. Differences in rho alone are not evidence of significant differences.
PSI smoothing results are exported as measurement sensitivity; no new alternative-measure
association search is performed here. No p-values or association confidence intervals.

**STOP after this section. Share `notebook_12_stage4_review.zip` for interpretation.**
Do not proceed to chronological warning validation or intervention testing from this notebook.


In [38]:
# Stage 4: validate exports and create the review ZIP

require(
    json.dumps(reference_payload, sort_keys=True, allow_nan=False)
    == stage4_reference_text,
    "Frozen reference specifications changed."
)

require(
    all(sha256(path) == input_hashes[name]
        for name, path in paths.items()),
    "Original inputs changed during Stage 4."
)

require(
    all(sha256(path) == association_hashes[name]
        for name, path in association_sources.items()),
    "Stage 3C sources changed."
)

stage4_exports = {
    "annual_quarterly_feature_drift.csv": stage4_drift,
    "annual_quarterly_psi_sensitivity.csv": stage4_psi,
    "annual_quarterly_numeric_bin_counts.csv": stage4_bins,
    "annual_quarterly_category_proportions.csv": stage4_categories,
    "annual_quarterly_population_counts.csv": stage4_counts,
    "annual_quarterly_selected_signal_details.csv": stage4_details,
    "annual_quarterly_drift_reliability_comparison.csv": stage4_comparison,
    "annual_quarterly_association_grid.csv": stage4_associations,
    "annual_quarterly_prevalence_time_context.csv": stage4_context,
    "three_resolution_association_comparison.csv": stage4_resolution_comparison,
}

stage4_validation = {
    "status": "passed",
    "stage": "Notebook 12 Stage 4 - calculations; interpretation pending",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "periods": TEMPORAL_PERIODS,
    "primary_resolution": "half_year",
    "reference_years": [2015, 2016, 2017],
    "target_horizon_months": 24,
    "frozen_reference_specification_sha256": stage4_reference_hash,
    "main_signals": ASSOCIATION_SIGNALS,
    "outcomes": ASSOCIATION_OUTCOMES,
    "output_row_counts": {
        name: len(frame)
        for name, frame in stage4_exports.items()
    },
    "ranking_support": reliability[
        [
            "period_kind", "period", "model", "n",
            "positives", "negatives", "ranking_status"
        ]
    ].to_dict("records"),
    "association_status_counts": (
        stage4_associations["status"].value_counts().to_dict()
    ),
    "half_year_regression_check": (
        "First evaluation half-year, both populations, all 29 predictors"
    ),
    "reliability_reconciliation": (
        "Stage 1 checked annual/quarterly metrics against loan predictions"
    ),
    "csv_validation": (
        "Row order and values checked after resetting non-exported indexes; "
        "bin boundaries compared numerically"
    ),
    "p_values_or_intervals_computed": False,
    "models_refitted_or_rescored": False,
    "thresholds_set": False,
    "chronological_warning_validation_completed": False,
    "interpretation_review_completed": False,
    "input_files": [
        {
            "name": str(path.relative_to(ROOT)),
            "sha256": input_hashes[name]
        }
        for name, path in paths.items()
    ],
    "stage3c_sources": [
        {
            "name": str(path.relative_to(ROOT)),
            "sha256": association_hashes[name]
        }
        for name, path in association_sources.items()
    ],
    "limitations": [
        "Retrospective frozen model; not a deployable January 2018 simulation.",
        "Six annual, 12 half-year and 24 quarterly cohorts; overlapping outcomes limit independence.",
        "Models and population views reuse eligible outcomes and are not independent replications.",
        "Full sample means sampled originations, not the entire Freddie Mac universe.",
        "Quarterly estimates require event-count context; no new intervals or low-event cutoff.",
        "Prevalence/time correlations do not isolate population drift effects.",
        "Temporal sensitivity does not validate warnings or justify retraining."
    ]
}

json.dumps(stage4_validation, allow_nan=False)

stage4_output = (
    DATA
    / "notebook_12_temporal_sensitivity"
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
stage4_output.mkdir(parents=True, exist_ok=False)

for name, frame in stage4_exports.items():
    print("Checking export:", name)

    frame.to_csv(
        stage4_output / name,
        index=False,
        na_rep="__STAGE4_MISSING__"
    )

    recovered = pd.read_csv(
        stage4_output / name,
        float_precision="round_trip",
        dtype={"period": "string"},
        keep_default_na=False,
        na_values=["__STAGE4_MISSING__"]
    )

    # CSV does not contain the original DataFrame row index.
    # Reset comparison copies without changing row order or source tables.
    expected = frame.reset_index(drop=True).copy()
    actual = recovered.reset_index(drop=True).copy()

    # Compare textual and numeric infinity boundaries consistently.
    for column in ["lower_edge", "upper_edge"]:
        if column in expected.columns:
            expected[column] = pd.to_numeric(
                expected[column], errors="raise"
            )
            actual[column] = pd.to_numeric(
                actual[column], errors="raise"
            )

    pd.testing.assert_frame_equal(
        expected,
        actual,
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-15
    )

    print("  Passed")

(stage4_output / "notebook_12_stage4_validation.json").write_text(
    json.dumps(stage4_validation, indent=2, allow_nan=False),
    encoding="utf-8"
)

(stage4_output / "frozen_drift_reference_specifications.json").write_text(
    json.dumps(reference_payload, indent=2, allow_nan=False),
    encoding="utf-8"
)

review_note = (
    "Stage 4 calculations passed; interpretation pending.\n"
    "Review annual/quarterly results alongside half-year results, "
    "event counts and prevalence/time context.\n"
    "No financial savings, validated warnings or retraining benefit "
    "has been demonstrated.\n"
    "STOP: share this ZIP before proceeding.\n"
)

(stage4_output / "READ_ME.txt").write_text(
    review_note,
    encoding="utf-8"
)

stage4_bundle = stage4_output / "notebook_12_stage4_review.zip"

bundle_files = [
    *stage4_exports,
    "notebook_12_stage4_validation.json",
    "frozen_drift_reference_specifications.json",
    "READ_ME.txt"
]

with zipfile.ZipFile(
    stage4_bundle,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:
    for name in bundle_files:
        archive.write(stage4_output / name, arcname=name)

print("\nNotebook 12 Stage 4 calculation checks passed.")
print("Interpretation review pending.")
print("Output folder:", stage4_output)
print("Share this review bundle:", stage4_bundle)
print("STOP here. No warning validation or retraining performed.")

Checking export: annual_quarterly_feature_drift.csv
  Passed
Checking export: annual_quarterly_psi_sensitivity.csv
  Passed
Checking export: annual_quarterly_numeric_bin_counts.csv
  Passed
Checking export: annual_quarterly_category_proportions.csv
  Passed
Checking export: annual_quarterly_population_counts.csv
  Passed
Checking export: annual_quarterly_selected_signal_details.csv
  Passed
Checking export: annual_quarterly_drift_reliability_comparison.csv
  Passed
Checking export: annual_quarterly_association_grid.csv
  Passed
Checking export: annual_quarterly_prevalence_time_context.csv
  Passed
Checking export: three_resolution_association_comparison.csv
  Passed

Notebook 12 Stage 4 calculation checks passed.
Interpretation review pending.
Output folder: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_temporal_sensitivity\20261018T011119407550Z
Share this review bundle: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\n

In [39]:
# Stage 5A: chronological availability audit — planning only
# No thresholds, warning models, retraining, or file exports.

require(
    stage4_validation["status"] == "passed",
    "Stage 4 must have passed first."
)

# Zero reporting lag is an optimistic planning assumption.
# Actual publication delays and historical data revisions are not verified.
REPORTING_LAG_MONTHS = 0

timing_parts = []

for year in YEARS:
    part = pd.read_parquet(
        paths[f"origination_{year}"],
        columns=["LOAN IDENTIFIER", "FIRST PAYMENT DATE"]
    )

    part["SOURCE YEAR"] = year
    part["LOAN IDENTIFIER"] = (
        part["LOAN IDENTIFIER"].astype("string")
    )

    payment_text = (
        part["FIRST PAYMENT DATE"]
        .astype("string")
        .str.strip()
    )

    require(
        payment_text.str.fullmatch(r"\d{6}", na=False).all(),
        f"{year}: FIRST PAYMENT DATE must contain YYYYMM values."
    )

    part["first_payment_month"] = pd.to_datetime(
        payment_text,
        format="%Y%m",
        errors="raise"
    ).dt.to_period("M")

    timing_parts.append(
        part[KEYS + ["first_payment_month"]]
    )

timing_source = pd.concat(timing_parts, ignore_index=True)

require(
    not timing_source.duplicated(KEYS).any(),
    "Duplicate loan keys in first-payment timing data."
)

# Use the exact existing eligible population.
timing = modeling[KEYS + ["period"]].merge(
    timing_source,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

require(
    timing["first_payment_month"].notna().all(),
    "An eligible loan is missing first-payment timing."
)

# Conservative full-window timing:
# wait for all 24 months, even if a loan terminated earlier.
timing["window_end_month"] = (
    timing["first_payment_month"] + 23
)

# Assume a reporting month is usable from the next month.
timing["planning_available_month"] = (
    timing["window_end_month"] + 1 + REPORTING_LAG_MONTHS
)

development_timing = timing.loc[
    timing["SOURCE YEAR"].le(2017)
]

model_available_month = (
    development_timing["planning_available_month"].max()
)

evaluation_timing = timing.loc[
    timing["SOURCE YEAR"].between(2018, 2023)
]

cohort_available_month = (
    evaluation_timing.groupby("period")[
        "planning_available_month"
    ]
    .max()
    .reindex(PERIODS)
)

require(
    cohort_available_month.notna().all(),
    "Missing evaluation cohort timing."
)

audit_rows = []

for period in PERIODS:
    year = int(period[:4])
    closing_month = 6 if period.endswith("H1") else 12

    # Full-cohort drift is available only after the cohort closes,
    # under this simplified zero-input-publication-lag assumption.
    decision_month = (
        pd.Period(
            f"{year}-{closing_month:02d}",
            freq="M"
        ) + 1
    )

    earlier_periods = PERIODS[:PERIODS.index(period)]

    available_history = [
        earlier
        for earlier in earlier_periods
        if cohort_available_month.loc[earlier] <= decision_month
    ]

    audit_rows.append({
        "cohort": period,
        "planning_decision_month": str(decision_month),
        "model_planning_available_month": str(model_available_month),
        "model_available_by_decision": (
            model_available_month <= decision_month
        ),
        "cohort_outcome_planning_available_month": str(
            cohort_available_month.loc[period]
        ),
        "matured_earlier_evaluation_cohorts": len(
            available_history
        ),
        "available_earlier_cohorts": (
            ", ".join(available_history)
            if available_history else "None"
        )
    })

chronological_availability_audit = pd.DataFrame(audit_rows)

display(chronological_availability_audit)

print(
    "Planning assumption: reporting lag =",
    REPORTING_LAG_MONTHS,
    "months."
)
print(
    "Dates use complete first-payment-relative outcome windows."
)
print(
    "Historical publication dates and data revisions are not verified."
)
print(
    "2018–2023 outcomes were already explored; this is not an "
    "untouched-test-set claim."
)
print("STOP: review this table before designing warning rules.")

,cohort,planning_decision_month,model_planning_available_month,model_available_by_decision,cohort_outcome_planning_available_month,matured_earlier_evaluation_cohorts,available_earlier_cohorts
0,2018-H1,2018-07,2022-10,False,2022-01,0,None
1,2018-H2,2019-01,2022-10,False,2022-12,0,None
2,2019-H1,2019-07,2022-10,False,2023-03,0,None
3,2019-H2,2020-01,2022-10,False,2023-06,0,None
4,2020-H1,2020-07,2022-10,False,2023-11,0,None
5,2020-H2,2021-01,2022-10,False,2024-02,0,None
6,2021-H1,2021-07,2022-10,False,2024-07,0,None
7,2021-H2,2022-01,2022-10,False,2025-02,1,2018-H1
8,2022-H1,2022-07,2022-10,False,2025-08,1,2018-H1
9,2022-H2,2023-01,2022-10,True,2026-01,2,"2018-H1, 2018-H2"


Planning assumption: reporting lag = 0 months.
Dates use complete first-payment-relative outcome windows.
Historical publication dates and data revisions are not verified.
2018–2023 outcomes were already explored; this is not an untouched-test-set claim.
STOP: review this table before designing warning rules.


In [40]:
# Inspect records driving the conservative availability dates.
# Read-only: no data, target, or model changes.

audit_columns = [
    "TARGET_ELIGIBLE",
    "FOLLOW_UP_COMPLETE",
    "TERMINATION_WITHIN_HORIZON",
    "LEFT_TRUNCATED",
    "FIRST_OBSERVED_MONTH",
    "LAST_OBSERVED_MONTH",
    "LAST_MONTH_WITHIN_HORIZON",
    "HORIZON_MONTHS",
    "TARGET_COMPOSITE_CREDIT_EVENT",
]

available_columns = [
    column for column in audit_columns
    if column in modeling.columns
]

timing_diagnostic = timing.merge(
    modeling[KEYS + available_columns],
    on=KEYS,
    how="left",
    validate="one_to_one"
)

print("First-payment ranges by assigned cohort:")

display(
    timing_diagnostic.groupby("period").agg(
        loans=("LOAN IDENTIFIER", "size"),
        earliest_first_payment=("first_payment_month", "min"),
        latest_first_payment=("first_payment_month", "max"),
        latest_full_window=("window_end_month", "max"),
    )
)

# Show the three latest first-payment records in each cohort,
# including development cohorts.
latest_records = (
    timing_diagnostic
    .sort_values(
        ["period", "first_payment_month", "LOAN IDENTIFIER"]
    )
    .groupby("period", sort=True)
    .tail(3)
)

print("Latest first-payment records driving the timing bounds:")

display(
    latest_records[
        KEYS
        + [
            "period",
            "first_payment_month",
            "window_end_month",
            "planning_available_month",
        ]
        + available_columns
    ].to_string(index=False)
)

print("Target-audit fields available:", available_columns)
print("STOP: reconcile these records before setting historical cutoffs.")

First-payment ranges by assigned cohort:


,loans,earliest_first_payment,latest_first_payment,latest_full_window
period,,,,
2015-H1,23602,2015-02,2016-12,2018-11
2015-H2,23642,2015-08,2017-05,2019-04
2016-H1,24039,2016-02,2020-10,2022-09
2016-H2,24037,2016-08,2018-07,2020-06
2017-H1,24091,2017-02,2020-05,2022-04
2017-H2,24187,2017-08,2020-03,2022-02
2018-H1,23699,2018-02,2020-01,2021-12
2018-H2,23718,2018-08,2020-12,2022-11
2019-H1,24326,2019-02,2021-03,2023-02


Latest first-payment records driving the timing bounds:


' SOURCE YEAR LOAN IDENTIFIER  period first_payment_month window_end_month planning_available_month  HORIZON_MONTHS  TARGET_COMPOSITE_CREDIT_EVENT\n        2015    F15Q20401762 2015-H1             2016-07          2018-06                  2018-07              24                              0\n        2015    F15Q10358286 2015-H1             2016-09          2018-08                  2018-09              24                              0\n        2015    F15Q20402267 2015-H1             2016-12          2018-11                  2018-12              24                              0\n        2015    F15Q40326188 2015-H2             2017-02          2019-01                  2019-02              24                              0\n        2015    F15Q40327210 2015-H2             2017-02          2019-01                  2019-02              24                              0\n        2015    F15Q40327294 2015-H2             2017-05          2019-04                  2019-05              24   

Target-audit fields available: ['HORIZON_MONTHS', 'TARGET_COMPOSITE_CREDIT_EVENT']
STOP: reconcile these records before setting historical cutoffs.


In [41]:
# Stage 5A: inspect existing target-audit records.
# Read-only: no target, feature, model, or file changes.

loans_to_check = {
    2016: ["F16Q20434001"],
    2017: ["F17Q20338706", "F17Q40360215"],
    2023: ["F23Q40207518"],
}

target_audit_columns = [
    "LOAN IDENTIFIER",
    "HORIZON_MONTHS",
    "FIRST_OBSERVED_MONTH",
    "LAST_OBSERVED_MONTH",
    "OBSERVED_MONTHS",
    "LAST_MONTH_WITHIN_HORIZON",
    "LEFT_TRUNCATED",
    "FOLLOW_UP_COMPLETE",
    "TARGET_ELIGIBLE",
    "TERMINATION_WITHIN_HORIZON",
    "TARGET_COMPOSITE_CREDIT_EVENT",
]

diagnostic_parts = []

for year, loan_ids in loans_to_check.items():
    matches = list(
        DATA.rglob(f"performance_target_candidates_{year}.parquet")
    )

    require(
        len(matches) == 1,
        f"{year}: expected one candidate file; found {matches}"
    )

    candidates = pd.read_parquet(
        matches[0],
        columns=target_audit_columns
    )

    candidates["LOAN IDENTIFIER"] = (
        candidates["LOAN IDENTIFIER"].astype("string")
    )

    selected = candidates.loc[
        candidates["HORIZON_MONTHS"].eq(24)
        & candidates["LOAN IDENTIFIER"].isin(loan_ids)
    ].copy()

    require(
        len(selected) == len(loan_ids)
        and not selected["LOAN IDENTIFIER"].duplicated().any(),
        f"{year}: requested loan records did not match exactly."
    )

    selected["SOURCE YEAR"] = year
    diagnostic_parts.append(selected)

target_timing_check = pd.concat(
    diagnostic_parts,
    ignore_index=True
)

target_timing_check = target_timing_check.merge(
    timing[
        KEYS + [
            "period",
            "first_payment_month",
            "window_end_month",
        ]
    ],
    on=KEYS,
    how="left",
    validate="one_to_one"
)

require(
    target_timing_check["first_payment_month"].notna().all(),
    "A requested loan did not match the timing table."
)

print("Four-loan target-timing diagnostic:")
print(target_timing_check.to_string(index=False))

print(
    "\nSTOP: inspect observation and termination evidence "
    "before revising availability dates."
)

Four-loan target-timing diagnostic:
LOAN IDENTIFIER  HORIZON_MONTHS  FIRST_OBSERVED_MONTH  LAST_OBSERVED_MONTH  OBSERVED_MONTHS  LAST_MONTH_WITHIN_HORIZON  LEFT_TRUNCATED  FOLLOW_UP_COMPLETE  TARGET_ELIGIBLE  TERMINATION_WITHIN_HORIZON  TARGET_COMPOSITE_CREDIT_EVENT  SOURCE YEAR  period first_payment_month window_end_month
   F16Q20434001              24                     0                   33               24                       23.0           False                True             True                       False                              0         2016 2016-H1             2020-10          2022-09
   F17Q20338706              24                     0                   70               24                       23.0           False                True             True                       False                              0         2017 2017-H1             2020-05          2022-04
   F17Q40360215              24                    -1                   24               24      

In [42]:
# Stage 5A: locate source files before checking the four loans.
# Read-only; no files or project settings are changed.

from pathlib import Path
import json

check_years = [2016, 2017, 2023]
raw_root = ROOT / "data" / "raw"

print("RAW FILES FOR THE THREE YEARS")
print("----------------------------")

require(raw_root.is_dir(), f"Raw-data folder missing: {raw_root}")

raw_matches = sorted(
    path
    for path in raw_root.rglob("*")
    if path.is_file()
    and any(str(year) in str(path.relative_to(raw_root))
            for year in check_years)
)

for path in raw_matches:
    print(
        path.relative_to(ROOT),
        f"({path.stat().st_size / 1024**2:,.1f} MB)"
    )

if not raw_matches:
    print("No matching filenames found. Available raw subfolders:")
    for path in sorted(raw_root.iterdir()):
        print(path.name)

print("\nSOURCE PATHS RECORDED BY THE TARGET PIPELINE")
print("-------------------------------------------")

for year in check_years:
    matches = sorted(
        DATA.rglob(f"performance_target_{year}_validation.json")
    )

    if not matches:
        print(f"{year}: validation JSON not found")
        continue

    for path in matches:
        payload = json.loads(path.read_text(encoding="utf-8"))

        print(f"\nYear: {year}")
        print("Validation file:", path)
        print("Origination source:", payload.get("origination_file"))
        print("Performance source:", payload.get("performance_file"))

RAW FILES FOR THE THREE YEARS
----------------------------
data\raw\Freddie_Mac\sample_2016\sample_orig_2016.txt (6.0 MB)
data\raw\Freddie_Mac\sample_2016\sample_perf_2016.txt (349.4 MB)
data\raw\Freddie_Mac\sample_2016.zip (52.2 MB)
data\raw\Freddie_Mac\sample_2017\sample_orig_2017.txt (6.0 MB)
data\raw\Freddie_Mac\sample_2017\sample_perf_2017.txt (292.6 MB)
data\raw\Freddie_Mac\sample_2017.zip (43.6 MB)
data\raw\Freddie_Mac\sample_2023\sample_orig_2023.txt (6.0 MB)
data\raw\Freddie_Mac\sample_2023\sample_perf_2023.txt (162.1 MB)
data\raw\Freddie_Mac\sample_2023.zip (21.8 MB)

SOURCE PATHS RECORDED BY THE TARGET PIPELINE
-------------------------------------------

Year: 2016
Validation file: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\performance_target_2016_validation.json
Origination source: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2016_clean.parquet
Performance source: e:\capstone_project\mortgage-model-rel

In [43]:
# Stage 5A: compare four loans with raw origination/performance records.
# Read-only and memory-efficient.

from src.freddie_config import ORIGINATION_COLUMNS
from src.freddie_performance import PERFORMANCE_COLUMNS

check_loans = {
    2016: ["F16Q20434001"],
    2017: ["F17Q20338706", "F17Q40360215"],
    2023: ["F23Q40207518"],
}

performance_fields = [
    "LOAN IDENTIFIER",
    "PERIOD",
    "LOAN AGE",
    "CURRENT LOAN DELINQUENCY STATUS",
    "MODIFICATION FLAG",
    "ZERO BALANCE CODE",
    "ZERO BALANCE EFFECTIVE DATE",
]

raw_origination_parts = []
raw_performance_parts = []

for year, loan_ids in check_loans.items():
    folder = ROOT / "data" / "raw" / "Freddie_Mac" / f"sample_{year}"
    orig_file = folder / f"sample_orig_{year}.txt"
    perf_file = folder / f"sample_perf_{year}.txt"

    require(orig_file.is_file(), f"Missing: {orig_file}")
    require(perf_file.is_file(), f"Missing: {perf_file}")

    # Verify the expected layouts before reading selected columns.
    for path, columns in [
        (orig_file, ORIGINATION_COLUMNS),
        (perf_file, PERFORMANCE_COLUMNS),
    ]:
        with path.open(encoding="utf-8-sig") as stream:
            field_count = len(
                stream.readline().rstrip("\r\n").split("|")
            )
        require(
            field_count == len(columns),
            f"{path.name}: unexpected field count {field_count}"
        )

    orig = pd.read_csv(
        orig_file,
        sep="|",
        header=None,
        names=ORIGINATION_COLUMNS,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    selected_orig = orig.loc[
        orig["LOAN IDENTIFIER"].isin(loan_ids),
        [
            "LOAN IDENTIFIER",
            "FIRST PAYMENT DATE",
            "MATURITY DATE",
            "ORIGINAL LOAN TERM",
        ],
    ].copy()

    require(
        len(selected_orig) == len(loan_ids)
        and not selected_orig["LOAN IDENTIFIER"].duplicated().any(),
        f"{year}: raw origination loans did not match exactly."
    )

    selected_orig["SOURCE YEAR"] = year
    raw_origination_parts.append(selected_orig)

    print(f"Scanning {year} performance file...")

    for chunk in pd.read_csv(
        perf_file,
        sep="|",
        header=None,
        names=PERFORMANCE_COLUMNS,
        usecols=performance_fields,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig",
        chunksize=100_000,
    ):
        selected_perf = chunk.loc[
            chunk["LOAN IDENTIFIER"].isin(loan_ids)
        ].copy()

        if not selected_perf.empty:
            selected_perf["SOURCE YEAR"] = year
            raw_performance_parts.append(selected_perf)

    del orig

raw_origination_check = pd.concat(
    raw_origination_parts, ignore_index=True
)

raw_performance_check = pd.concat(
    raw_performance_parts, ignore_index=True
).sort_values(KEYS + ["PERIOD"])

require(
    not raw_performance_check.duplicated(KEYS + ["PERIOD"]).any(),
    "Duplicate monthly records found for a selected loan."
)

raw_origination_check["raw_first_payment_month"] = pd.to_datetime(
    raw_origination_check["FIRST PAYMENT DATE"],
    format="%Y%m",
    errors="raise",
).dt.to_period("M")

date_comparison = raw_origination_check.merge(
    timing[KEYS + ["period", "first_payment_month"]],
    on=KEYS,
    how="left",
    validate="one_to_one",
)

date_comparison["raw_matches_cleaned"] = (
    date_comparison["raw_first_payment_month"]
    == date_comparison["first_payment_month"]
)

print("\nRAW VERSUS CLEANED FIRST-PAYMENT DATES")
print(date_comparison.to_string(index=False))

summary_rows = []

for loan in date_comparison.itertuples(index=False):
    loan_id = getattr(loan, "_1", None)

# Use explicit column access because the original names contain spaces.
for _, loan in date_comparison.iterrows():
    loan_id = loan["LOAN IDENTIFIER"]
    year = loan["SOURCE YEAR"]

    records = raw_performance_check.loc[
        raw_performance_check["LOAN IDENTIFIER"].eq(loan_id)
        & raw_performance_check["SOURCE YEAR"].eq(year)
    ].copy()

    require(not records.empty, f"No raw performance for {loan_id}")

    reporting_months = pd.to_datetime(
        records["PERIOD"], format="%Y%m", errors="raise"
    ).dt.to_period("M")

    first_payment = loan["raw_first_payment_month"]
    relative_month = reporting_months.map(
        lambda month: month.ordinal - first_payment.ordinal
    )

    in_window = relative_month.between(0, 23)

    termination = (
        records["ZERO BALANCE CODE"].str.strip().ne("")
    )

    modification = (
        records["MODIFICATION FLAG"].str.strip().eq("Y")
    )

    summary_rows.append({
        "loan": loan_id,
        "raw_first_payment": str(first_payment),
        "first_reporting_month": str(reporting_months.min()),
        "last_reporting_month": str(reporting_months.max()),
        "records_before_first_payment": int(relative_month.lt(0).sum()),
        "observed_months_in_24m_window": int(
            reporting_months.loc[in_window].nunique()
        ),
        "modification_Y_rows": int(modification.sum()),
        "termination_reporting_periods": ", ".join(
            records.loc[termination, "PERIOD"].tolist()
        ),
        "termination_codes": ", ".join(
            records.loc[termination, "ZERO BALANCE CODE"].tolist()
        ),
    })

    # Show first/last records and up to ten flagged records.
    evidence = pd.concat([
        records.head(3),
        records.loc[modification | termination].head(10),
        records.tail(3),
    ]).drop_duplicates(KEYS + ["PERIOD"])

    print(f"\nRAW MONTHLY EVIDENCE: {loan_id}")
    print(evidence[performance_fields].to_string(index=False))

print("\nFOUR-LOAN SOURCE SUMMARY")
print(pd.DataFrame(summary_rows).to_string(index=False))

print("\nSTOP: source verification only; no records were changed.")

Scanning 2016 performance file...
Scanning 2017 performance file...
Scanning 2023 performance file...

RAW VERSUS CLEANED FIRST-PAYMENT DATES
LOAN IDENTIFIER FIRST PAYMENT DATE MATURITY DATE ORIGINAL LOAN TERM  SOURCE YEAR raw_first_payment_month  period first_payment_month  raw_matches_cleaned
   F16Q20434001             202010        204606                309         2016                 2020-10 2016-H1             2020-10                 True
   F17Q20338706             202005        205004                360         2017                 2020-05 2017-H1             2020-05                 True
   F17Q40360215             201904        204903                360         2017                 2019-04 2017-H2             2019-04                 True
   F23Q40207518             202506        205505                360         2023                 2025-06 2023-H2             2025-06                 True

RAW MONTHLY EVIDENCE: F16Q20434001
LOAN IDENTIFIER PERIOD LOAN AGE CURRENT LOAN DELINQU

In [44]:
# Stage 5A: quantify reporting delays across all eligible loans.
# First reporting month is reconstructed from saved audit fields.
# It is NOT the historical publication/availability date.

delay_parts = []

for year in YEARS:
    matches = list(
        DATA.rglob(f"performance_target_candidates_{year}.parquet")
    )

    require(
        len(matches) == 1,
        f"{year}: expected one candidate file; found {matches}"
    )

    candidates = pd.read_parquet(
        matches[0],
        columns=[
            "LOAN IDENTIFIER",
            "HORIZON_MONTHS",
            "FIRST_OBSERVED_MONTH",
            "TARGET_ELIGIBLE",
        ],
    )

    selected = candidates.loc[
        candidates["HORIZON_MONTHS"].eq(24)
        & candidates["TARGET_ELIGIBLE"]
    ].copy()

    selected["SOURCE YEAR"] = year
    selected["LOAN IDENTIFIER"] = (
        selected["LOAN IDENTIFIER"].astype("string")
    )

    delay_parts.append(
        selected[KEYS + ["FIRST_OBSERVED_MONTH"]]
    )

saved_timing_audit = pd.concat(delay_parts, ignore_index=True)

require(
    not saved_timing_audit.duplicated(KEYS).any(),
    "Duplicate eligible loan keys in saved audit files."
)

delay_audit = timing.merge(
    saved_timing_audit,
    on=KEYS,
    how="outer",
    validate="one_to_one",
    indicator=True,
)

require(
    delay_audit["_merge"].eq("both").all(),
    "Saved audit loans differ from the existing eligible population."
)

require(
    delay_audit["FIRST_OBSERVED_MONTH"].notna().all(),
    "Missing first-observation offsets."
)

offsets = pd.to_numeric(
    delay_audit["FIRST_OBSERVED_MONTH"], errors="raise"
)

require(
    offsets.eq(np.floor(offsets)).all(),
    "First-observation offsets must be whole months."
)

delay_audit["first_reporting_month"] = pd.PeriodIndex(
    [
        payment_month + int(offset)
        for payment_month, offset in zip(
            delay_audit["first_payment_month"], offsets
        )
    ],
    freq="M",
)

delay_audit["cohort_end_month"] = pd.PeriodIndex(
    [
        f"{period[:4]}-{'06' if period.endswith('H1') else '12'}"
        for period in delay_audit["period"]
    ],
    freq="M",
)

delay_audit["months_after_cohort_end"] = [
    reporting_month.ordinal - cohort_end.ordinal
    for reporting_month, cohort_end in zip(
        delay_audit["first_reporting_month"],
        delay_audit["cohort_end_month"],
    )
]

delay_audit["role"] = np.where(
    delay_audit["SOURCE YEAR"].le(2017),
    "development",
    "evaluation",
)

# These are descriptive reporting-delay groups, not warning thresholds.
delay_audit["starts_after_cohort_end"] = (
    delay_audit["months_after_cohort_end"].gt(0)
)
delay_audit["starts_over_6_months_later"] = (
    delay_audit["months_after_cohort_end"].gt(6)
)
delay_audit["starts_over_12_months_later"] = (
    delay_audit["months_after_cohort_end"].gt(12)
)


def summarize_reporting_delays(frame, group_columns):
    result = (
        frame.groupby(group_columns, sort=True)
        .agg(
            eligible_loans=("LOAN IDENTIFIER", "size"),
            starts_after_cohort_end=(
                "starts_after_cohort_end", "sum"
            ),
            starts_over_6_months_later=(
                "starts_over_6_months_later", "sum"
            ),
            starts_over_12_months_later=(
                "starts_over_12_months_later", "sum"
            ),
            maximum_months_after_cohort_end=(
                "months_after_cohort_end", "max"
            ),
        )
        .reset_index()
    )

    for column in [
        "starts_after_cohort_end",
        "starts_over_6_months_later",
        "starts_over_12_months_later",
    ]:
        result[column + "_pct"] = (
            100 * result[column] / result["eligible_loans"]
        ).round(3)

    return result


cohort_delay_summary = summarize_reporting_delays(
    delay_audit, ["role", "period"]
)

role_delay_summary = summarize_reporting_delays(
    delay_audit, ["role"]
)

require(
    len(delay_audit) == 437668,
    "Eligible loan total changed."
)

print("BY HALF-YEAR COHORT")
print(cohort_delay_summary.to_string(index=False))

print("\nDEVELOPMENT VERSUS EVALUATION")
print(role_delay_summary.to_string(index=False))

print(
    "\nNote: starts after cohort end can include normal reporting lag. "
    "The 6- and 12-month groups are descriptive, not exclusion rules."
)
print(
    "This audit covers eligible loans only. Reporting month does not "
    "establish when a record or its predictor values were published."
)
print("STOP: review these summaries before choosing historical cutoffs.")

BY HALF-YEAR COHORT
       role  period  eligible_loans  starts_after_cohort_end  starts_over_6_months_later  starts_over_12_months_later  maximum_months_after_cohort_end  starts_after_cohort_end_pct  starts_over_6_months_later_pct  starts_over_12_months_later_pct
development 2015-H1           23602                     4087                          17                            2                               17                       17.316                           0.072                            0.008
development 2015-H2           23642                     4360                          34                            3                               16                       18.442                           0.144                            0.013
development 2016-H1           24039                     4794                          25                            3                               52                       19.943                           0.104                            0.012


In [45]:
# Stage 5A: conservative information-availability table.
# Read-only. No exclusions, model fitting, thresholds, or exports.
#
# Important:
# Reporting month is NOT publication date.
# Current-snapshot records do NOT establish historical data vintages.
# Eligibility is retrospective and is not assumed known at arrival.

candidate_parts = []

for year in YEARS:
    matches = list(
        DATA.rglob(f"performance_target_candidates_{year}.parquet")
    )

    require(
        len(matches) == 1,
        f"{year}: expected one candidate file; found {matches}"
    )

    part = pd.read_parquet(
        matches[0],
        columns=[
            "LOAN IDENTIFIER",
            "HORIZON_MONTHS",
            "FIRST_OBSERVED_MONTH",
            "LAST_MONTH_WITHIN_HORIZON",
            "TARGET_ELIGIBLE",
            "TERMINATION_WITHIN_HORIZON",
        ],
    )

    part = part.loc[part["HORIZON_MONTHS"].eq(24)].copy()
    part["SOURCE YEAR"] = year
    part["LOAN IDENTIFIER"] = (
        part["LOAN IDENTIFIER"].astype("string")
    )

    candidate_parts.append(part)

availability_records = pd.concat(
    candidate_parts, ignore_index=True
)

require(
    not availability_records.duplicated(KEYS).any(),
    "Duplicate candidate loan keys."
)

availability_records = (
    originations[KEYS + ["period"]]
    .merge(
        timing_source,
        on=KEYS,
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)

require(
    availability_records["_merge"].eq("both").all(),
    "Origination and first-payment loan sets differ."
)

availability_records = availability_records.drop(columns="_merge")

candidate_audit = pd.concat(candidate_parts, ignore_index=True)

availability_records = availability_records.merge(
    candidate_audit,
    on=KEYS,
    how="outer",
    validate="one_to_one",
    indicator=True,
)

require(
    availability_records["_merge"].eq("both").all(),
    "Origination and candidate-audit loan sets differ."
)

availability_records = availability_records.drop(columns="_merge")

# Verify that the existing eligible population is preserved exactly.
eligible_keys = pd.MultiIndex.from_frame(modeling[KEYS])
audit_eligible_keys = pd.MultiIndex.from_frame(
    availability_records.loc[
        availability_records["TARGET_ELIGIBLE"], KEYS
    ]
)

require(
    len(eligible_keys) == len(audit_eligible_keys)
    and eligible_keys.isin(audit_eligible_keys).all()
    and audit_eligible_keys.isin(eligible_keys).all(),
    "Eligible population differs from the existing modeling dataset."
)

first_offsets = pd.to_numeric(
    availability_records["FIRST_OBSERVED_MONTH"],
    errors="raise",
)

require(
    first_offsets.dropna().eq(
        np.floor(first_offsets.dropna())
    ).all(),
    "First-observation offsets must be whole months."
)

availability_records["first_report_proxy"] = pd.PeriodIndex(
    [
        payment + int(offset) if pd.notna(offset) else pd.NaT
        for payment, offset in zip(
            availability_records["first_payment_month"],
            first_offsets,
        )
    ],
    freq="M",
)

eligible_availability = availability_records.loc[
    availability_records["TARGET_ELIGIBLE"]
].copy()

require(
    eligible_availability["first_report_proxy"].notna().all(),
    "An eligible loan has no first-report timing."
)

last_in_window = pd.to_numeric(
    eligible_availability["LAST_MONTH_WITHIN_HORIZON"],
    errors="raise",
)

require(
    last_in_window.notna().all()
    and last_in_window.between(0, 23).all()
    and last_in_window.eq(np.floor(last_in_window)).all(),
    "Invalid last-observed month within the target window."
)

# For nonterminated loans, wait through month 23.
#
# For terminated loans, use the last observed month within the window.
# This is a conservative completion proxy, not the exact termination date:
# some performance records may continue after termination.
completion_offsets = np.where(
    eligible_availability["TERMINATION_WITHIN_HORIZON"],
    last_in_window.to_numpy(),
    23,
).astype(int)

eligible_availability["completion_proxy"] = pd.PeriodIndex(
    [
        payment + int(offset)
        for payment, offset in zip(
            eligible_availability["first_payment_month"],
            completion_offsets,
        )
    ],
    freq="M",
)

# A reporting month is treated as usable no earlier than the next month.
# Actual publication lag and revisions remain unverified.
eligible_availability["label_ready_proxy"] = (
    eligible_availability["completion_proxy"] + 1
)

model_ready_proxy = eligible_availability.loc[
    eligible_availability["SOURCE YEAR"].le(2017),
    "label_ready_proxy",
].max()

cohort_label_ready = (
    eligible_availability.groupby("period")["label_ready_proxy"]
    .max()
)

availability_rows = []

for period in PERIODS:
    year = int(period[:4])
    end_month = 6 if period.endswith("H1") else 12
    nominal_decision = pd.Period(
        f"{year}-{end_month:02d}", freq="M"
    ) + 1

    full_cohort = availability_records.loc[
        availability_records["period"].eq(period)
    ]

    eligible_cohort = eligible_availability.loc[
        eligible_availability["period"].eq(period)
    ]

    earlier_periods = PERIODS[:PERIODS.index(period)]

    matured_earlier = [
        earlier
        for earlier in earlier_periods
        if cohort_label_ready.loc[earlier] <= nominal_decision
    ]

    availability_rows.append({
        "cohort": period,
        "nominal_decision_month": str(nominal_decision),
        "model_label_ready_proxy": str(model_ready_proxy),
        "model_labels_ready_by_nominal_decision": (
            model_ready_proxy <= nominal_decision
        ),
        "eligible_loans": len(eligible_cohort),
        "eligible_records_not_yet_reported": int(
            eligible_cohort["first_report_proxy"]
            .ge(nominal_decision).sum()
        ),
        "full_sample_records_not_yet_reported": int(
            full_cohort["first_report_proxy"]
            .ge(nominal_decision).sum()
        ),
        "full_sample_records_without_reporting_date": int(
            full_cohort["first_report_proxy"].isna().sum()
        ),
        "cohort_label_ready_proxy": str(
            cohort_label_ready.loc[period]
        ),
        "matured_earlier_cohort_count": len(matured_earlier),
        "matured_earlier_cohorts": (
            ", ".join(matured_earlier)
            if matured_earlier else "None"
        ),
    })

information_availability_table = pd.DataFrame(availability_rows)

print("CONSERVATIVE INFORMATION-AVAILABILITY TABLE")
print(information_availability_table.to_string(index=False))

print("\nInterpretation limits:")
print("- Completion estimates account conservatively for early termination.")
print("- Reporting dates do not establish publication or predictor availability.")
print("- Existing eligible membership was determined retrospectively.")
print("- No cohort is declared feasible solely from these proxy dates.")
print("- No data, labels, features, models, or prior results were changed.")
print("STOP: review this table before selecting decision dates.")

CONSERVATIVE INFORMATION-AVAILABILITY TABLE
 cohort nominal_decision_month model_label_ready_proxy  model_labels_ready_by_nominal_decision  eligible_loans  eligible_records_not_yet_reported  full_sample_records_not_yet_reported  full_sample_records_without_reporting_date cohort_label_ready_proxy  matured_earlier_cohort_count                     matured_earlier_cohorts
2018-H1                2018-07                 2022-10                                   False           23699                               4244                                  5167                                           0                  2021-11                             0                                        None
2018-H2                2019-01                 2022-10                                   False           23718                               3779                                  4617                                           0                  2022-12                             0                    

In [46]:
# Stage 5B: retrospective reliability changes and uncertainty.
# Reference: each model's own 2018-H1 results.
# No model fitting, rescoring, exclusions, or warning thresholds.

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)

CHANGE_REFERENCE = "2018-H1"
CHANGE_BOOTSTRAPS = 1000
CHANGE_SEED = 42

CHANGE_METRICS = [
    "roc_auc",
    "average_precision",
    "brier",
    "absolute_risk_gap",
]

require(
    sha256(paths["predictions"]) == input_hashes["predictions"],
    "Saved predictions changed."
)

require(
    sha256(paths["reliability"]) == input_hashes["reliability"],
    "Saved reliability summary changed."
)

require(
    not pred.duplicated(KEYS).any(),
    "Duplicate prediction loan keys."
)

require(
    set(pred["period"]) == set(PERIODS),
    "Prediction cohort coverage differs."
)


def prepare_change_metric_order(probabilities):
    """Cache score ordering and tie groups for weighted metrics."""
    order = np.argsort(-probabilities, kind="stable")
    sorted_scores = probabilities[order]
    group_ends = np.r_[
        np.flatnonzero(np.diff(sorted_scores) != 0),
        len(sorted_scores) - 1,
    ]
    return order, group_ends


def change_metrics(y, probabilities, weights, order, group_ends):
    """Metrics for an ordinary loan bootstrap represented by counts."""
    total = weights.sum()
    events = np.dot(weights, y)
    non_events = total - events

    signed_gap = (
        events - np.dot(weights, probabilities)
    ) / total

    brier = np.dot(
        weights, (y - probabilities) ** 2
    ) / total

    auc = np.nan
    ap = np.nan

    if events > 0 and non_events > 0:
        ranked_weights = weights[order]
        ranked_y = y[order]

        tp = np.cumsum(
            ranked_weights * ranked_y
        )[group_ends]

        fp = np.cumsum(
            ranked_weights * (1 - ranked_y)
        )[group_ends]

        previous_tp = np.r_[0, tp[:-1]]
        previous_fp = np.r_[0, fp[:-1]]

        auc = np.sum(
            (fp - previous_fp) * (tp + previous_tp) / 2
        ) / (events * non_events)

        precision = np.divide(
            tp,
            tp + fp,
            out=np.zeros_like(tp, dtype=float),
            where=(tp + fp) > 0,
        )

        ap = np.sum(
            (tp - previous_tp) * precision
        ) / events

    return np.array([auc, ap, brier, abs(signed_gap)])


change_points = {}
change_draws = {}
change_counts = {}

# Separate reproducible random streams by cohort.
change_streams = np.random.SeedSequence(
    CHANGE_SEED
).spawn(len(PERIODS))

for period, stream in zip(PERIODS, change_streams):
    part = (
        pred.loc[pred["period"].eq(period)]
        .sort_values(KEYS)
        .reset_index(drop=True)
    )

    y = part["y_true"].to_numpy(dtype=int)
    n = len(y)

    require(n > 0, f"{period}: empty cohort.")
    require(
        np.isin(y, [0, 1]).all(),
        f"{period}: invalid outcomes."
    )

    probabilities = {
        model: part[f"p_{model}"].to_numpy(dtype=float)
        for model in FAMILIES
    }

    ordering = {
        model: prepare_change_metric_order(values)
        for model, values in probabilities.items()
    }

    change_counts[period] = {
        "n": n,
        "events": int(y.sum()),
        "prevalence": float(y.mean()),
    }

    unit_weights = np.ones(n, dtype=int)

    for model in FAMILIES:
        p = probabilities[model]

        require(
            np.isfinite(p).all()
            and ((p >= 0) & (p <= 1)).all(),
            f"{period}/{model}: invalid probabilities."
        )

        point = change_metrics(
            y, p, unit_weights, *ordering[model]
        )

        # Independently verify point estimates against sklearn.
        sklearn_point = np.array([
            roc_auc_score(y, p),
            average_precision_score(y, p),
            brier_score_loss(y, p),
            abs(y.mean() - p.mean()),
        ])

        np.testing.assert_allclose(
            point, sklearn_point,
            rtol=1e-12, atol=1e-14
        )

        saved = summary.loc[
            summary["period_kind"].eq("half_year")
            & summary["period"].eq(period)
            & summary["model"].eq(model)
        ]

        require(
            len(saved) == 1,
            f"{period}/{model}: missing saved summary."
        )

        saved = saved.iloc[0]

        np.testing.assert_allclose(
            point,
            [
                saved["roc_auc"],
                saved["average_precision"],
                saved["brier"],
                abs(saved["risk_gap"]),
            ],
            rtol=1e-9,
            atol=1e-12,
        )

        change_points[(period, model)] = point
        change_draws[(period, model)] = np.full(
            (CHANGE_BOOTSTRAPS, len(CHANGE_METRICS)),
            np.nan,
        )

    rng = np.random.default_rng(stream)

    for replicate in range(CHANGE_BOOTSTRAPS):
        # Ordinary, non-stratified loan-level bootstrap.
        sampled_indices = rng.integers(0, n, size=n)
        weights = np.bincount(sampled_indices, minlength=n)

        # Same sampled loans for both models.
        for model in FAMILIES:
            change_draws[(period, model)][replicate] = (
                change_metrics(
                    y,
                    probabilities[model],
                    weights,
                    *ordering[model],
                )
            )

    print("Bootstrap completed:", period, flush=True)

change_rows = []

for period in PERIODS:
    if period == CHANGE_REFERENCE:
        continue

    for model in FAMILIES:
        reference_point = change_points[
            (CHANGE_REFERENCE, model)
        ]
        current_point = change_points[(period, model)]

        # Independent cohort resamples; reuse the same reference draws
        # across all comparisons to preserve the shared-reference structure.
        raw_changes = (
            change_draws[(period, model)]
            - change_draws[(CHANGE_REFERENCE, model)]
        )

        for j, metric in enumerate(CHANGE_METRICS):
            raw_change = current_point[j] - reference_point[j]

            # Put every metric on a common interpretation:
            # positive = worse relative to the reference.
            direction = (
                -1 if metric in ["roc_auc", "average_precision"]
                else 1
            )

            worsening_change = direction * raw_change
            worsening_draws = direction * raw_changes[:, j]

            valid = np.isfinite(worsening_draws)
            valid_draws = int(valid.sum())

            if valid_draws:
                ci_low, ci_high = np.quantile(
                    worsening_draws[valid], [0.025, 0.975]
                )

                if ci_low > 0:
                    interpretation = "worsening_direction_supported"
                elif ci_high < 0:
                    interpretation = "improvement_direction_supported"
                else:
                    interpretation = "direction_uncertain"
            else:
                ci_low = np.nan
                ci_high = np.nan
                interpretation = "unavailable"

            change_rows.append({
                "period": period,
                "model": model,
                "metric": metric,
                "reference_period": CHANGE_REFERENCE,
                "reference_value": reference_point[j],
                "current_value": current_point[j],
                "raw_change_current_minus_reference": raw_change,
                "worsening_change": worsening_change,
                "worsening_ci_low": ci_low,
                "worsening_ci_high": ci_high,
                "valid_draws": valid_draws,
                "invalid_draws": CHANGE_BOOTSTRAPS - valid_draws,
                "reference_n": change_counts[CHANGE_REFERENCE]["n"],
                "reference_events": change_counts[CHANGE_REFERENCE]["events"],
                "reference_prevalence": change_counts[CHANGE_REFERENCE]["prevalence"],
                "current_n": change_counts[period]["n"],
                "current_events": change_counts[period]["events"],
                "current_prevalence": change_counts[period]["prevalence"],
                "direction_interpretation": interpretation,
            })

reliability_change_uncertainty = pd.DataFrame(change_rows)

require(
    len(reliability_change_uncertainty) == 88,
    "Expected 11 cohorts × 2 models × 4 metrics = 88 rows."
)

require(
    not reliability_change_uncertainty.duplicated(
        ["period", "model", "metric"]
    ).any(),
    "Duplicate change comparisons."
)

require(
    sha256(paths["predictions"]) == input_hashes["predictions"],
    "Prediction file changed during analysis."
)

change_output = (
    DATA
    / "notebook_12_reliability_changes"
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
change_output.mkdir(parents=True, exist_ok=False)

change_csv = (
    change_output / "half_year_reliability_change_uncertainty.csv"
)

reliability_change_uncertainty.to_csv(
    change_csv, index=False
)

change_readback = pd.read_csv(
    change_csv,
    dtype={"period": "string", "reference_period": "string"},
    float_precision="round_trip",
)

pd.testing.assert_frame_equal(
    reliability_change_uncertainty.reset_index(drop=True),
    change_readback,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-15,
)

display(
    reliability_change_uncertainty[
        [
            "period", "model", "metric",
            "worsening_change",
            "worsening_ci_low", "worsening_ci_high",
            "direction_interpretation",
        ]
    ]
)

print("\nStage 5B calculations completed; review pending.")
print("Upload this small CSV:", change_csv)
print("Positive worsening_change means worse than 2018-H1.")
print("Intervals are descriptive and pointwise, not simultaneous.")
print("Loan resampling does not capture shared economic shocks,")
print("model-fitting uncertainty, or dependence across outcome windows.")
print("Supported direction does not establish practical importance.")
print("An uncertain direction does not establish stable reliability.")
print("No warning thresholds or retraining decisions were made.")
print("STOP: review these results before proceeding.")

Bootstrap completed: 2018-H1
Bootstrap completed: 2018-H2
Bootstrap completed: 2019-H1
Bootstrap completed: 2019-H2
Bootstrap completed: 2020-H1
Bootstrap completed: 2020-H2
Bootstrap completed: 2021-H1
Bootstrap completed: 2021-H2
Bootstrap completed: 2022-H1
Bootstrap completed: 2022-H2
Bootstrap completed: 2023-H1
Bootstrap completed: 2023-H2


,period,model,metric,worsening_change,worsening_ci_low,worsening_ci_high,direction_interpretation
0,2018-H2,xgboost,roc_auc,0.033920,0.002139,0.065522,worsening_direction_supported
1,2018-H2,xgboost,average_precision,-0.045019,-0.060755,-0.028174,improvement_direction_supported
2,2018-H2,xgboost,brier,0.023003,0.020300,0.025667,worsening_direction_supported
3,2018-H2,xgboost,absolute_risk_gap,0.022954,0.020202,0.025580,worsening_direction_supported
4,2018-H2,logistic,roc_auc,0.025363,-0.005875,0.056604,direction_uncertain
...,...,...,...,...,...,...,...
83,2023-H2,xgboost,absolute_risk_gap,0.000482,-0.001548,0.002704,direction_uncertain
84,2023-H2,logistic,roc_auc,-0.062265,-0.093739,-0.030207,improvement_direction_supported
85,2023-H2,logistic,average_precision,-0.027054,-0.044925,-0.011400,improvement_direction_supported
86,2023-H2,logistic,brier,0.003273,0.001304,0.005326,worsening_direction_supported



Stage 5B calculations completed; review pending.
Upload this small CSV: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_reliability_changes\20261018T014434026687Z\half_year_reliability_change_uncertainty.csv
Positive worsening_change means worse than 2018-H1.
Intervals are descriptive and pointwise, not simultaneous.
Loan resampling does not capture shared economic shocks,
model-fitting uncertainty, or dependence across outcome windows.
Supported direction does not establish practical importance.
An uncertain direction does not establish stable reliability.
No warning thresholds or retraining decisions were made.
STOP: review these results before proceeding.


## Stage 5B review completed — 17 October 2026

Reliability changes relative to each model’s own 2018-H1 results
were evaluated for 11 later half-year cohorts, two models, and four
metrics: ROC-AUC, Average Precision, Brier score, and absolute risk gap.

All 88 comparisons passed review of the exported summaries.
Each comparison used 1,000 valid bootstrap differences, resampling
both the reference and later cohort while preserving each sampled
loan’s outcome and both models’ predictions.

The results show metric-specific changes rather than uniform
deterioration. Ranking and probability reliability can move differently.
Average Precision and Brier results require event-prevalence context.

Intervals are descriptive and pointwise. They do not capture shared
economic shocks, model-fitting uncertainty, or dependence across
overlapping outcome windows. Supported direction does not establish
practical importance; uncertain direction does not establish stability.

The timing audit identified delayed reporting and limitations in
reconstructing historical information availability. Reporting dates
are not verified publication dates. These findings must accompany
any subsequent warning or intervention evaluation.

No loans, targets, feature definitions, or frozen models were changed.
No warning thresholds or intervention rules were established.
Chronological early-warning validation remains incomplete.

Next: join these reliability-change estimates and intervals with the
five previously selected drift signals, retaining both population
views, model identity, event counts, and prevalence. This is a
retrospective evidence linkage, not a validated warning rule.

In [47]:
# Stage 5C: connect existing drift signals with reliability changes.
# Retrospective evidence linkage only—not a validated warning rule.

require(change_csv.is_file(), "Stage 5B CSV is missing.")

link_changes = pd.read_csv(
    change_csv,
    dtype={"period": "string", "reference_period": "string"},
    float_precision="round_trip",
)

require(
    len(link_changes) == 88
    and not link_changes.duplicated(
        ["period", "model", "metric"]
    ).any(),
    "Stage 5B comparison coverage differs."
)

require(
    link_changes["reference_period"].eq("2018-H1").all(),
    "Reliability reference changed."
)

link_signals = [rule[0] for rule in SIGNAL_RULES.values()]
link_statuses = [signal + "_status" for signal in link_signals]

require(
    len(link_signals) == 5,
    "Expected the five previously selected signals."
)

# Keep absolute reliability values for reconciliation.
link_columns = [
    "population", "period", "model",
    "n", "positives", "negatives", "prevalence",
    "mean_predicted_risk", "risk_gap",
    "roc_auc", "average_precision", "brier",
    "absolute_risk_gap",
    "drift_current_n", "drift_reference_n",
    "reliability_population", "population_pairing",
    *link_signals,
    *link_statuses,
]

link_base = cohort_comparison[link_columns].copy()

require(
    len(link_base) == 48
    and not link_base.duplicated(
        ["population", "period", "model"]
    ).any(),
    "Original half-year comparison coverage differs."
)

# The anchor has no change estimate against itself.
link_base = link_base.loc[
    link_base["period"].ne("2018-H1")
].copy()

link_base["period"] = link_base["period"].astype("string")

# Four metric changes per model/cohort, repeated across two views.
# Both views deliberately reuse the same eligible-loan reliability.
drift_reliability_evidence = link_base.merge(
    link_changes,
    on=["period", "model"],
    how="outer",
    validate="many_to_many",
    indicator=True,
)

require(
    drift_reliability_evidence["_merge"].eq("both").all(),
    "A drift or reliability-change record failed to match."
)

drift_reliability_evidence = (
    drift_reliability_evidence
    .drop(columns="_merge")
    .sort_values(["population", "period", "model", "metric"])
    .reset_index(drop=True)
)

expected_keys = {
    (population, period, model, metric)
    for population in ["eligible", "full_sample"]
    for period in PERIODS
    if period != "2018-H1"
    for model in FAMILIES
    for metric in CHANGE_METRICS
}

actual_keys = set(
    drift_reliability_evidence[
        ["population", "period", "model", "metric"]
    ].itertuples(index=False, name=None)
)

require(
    len(drift_reliability_evidence) == 176
    and actual_keys == expected_keys,
    "Expected 176 unique population/cohort/model/metric rows."
)

require(
    not drift_reliability_evidence.duplicated(
        ["population", "period", "model", "metric"]
    ).any(),
    "Duplicate evidence-table keys."
)

# Verify counts and prevalence survived the join.
for original, joined in [
    ("n", "current_n"),
    ("positives", "current_events"),
    ("prevalence", "current_prevalence"),
]:
    require(
        np.allclose(
            drift_reliability_evidence[original],
            drift_reliability_evidence[joined],
            rtol=1e-12,
            atol=1e-15,
        ),
        f"Joined {joined} differs from original reliability."
    )

# Verify each change refers to the correct absolute outcome.
for metric in CHANGE_METRICS:
    part = drift_reliability_evidence.loc[
        drift_reliability_evidence["metric"].eq(metric)
    ]

    require(
        np.allclose(
            part[metric],
            part["current_value"],
            rtol=1e-12,
            atol=1e-15,
        ),
        f"{metric}: absolute reliability and change result differ."
    )

# Confirm reliability and its change intervals are identical across views.
view_check_columns = [
    "current_value",
    "reference_value",
    "worsening_change",
    "worsening_ci_low",
    "worsening_ci_high",
    "current_n",
    "current_events",
    "current_prevalence",
    "direction_interpretation",
]

eligible_view = (
    drift_reliability_evidence.loc[
        drift_reliability_evidence["population"].eq("eligible")
    ]
    .set_index(["period", "model", "metric"])[view_check_columns]
    .sort_index()
)

full_view = (
    drift_reliability_evidence.loc[
        drift_reliability_evidence["population"].eq("full_sample")
    ]
    .set_index(["period", "model", "metric"])[view_check_columns]
    .sort_index()
)

pd.testing.assert_frame_equal(
    eligible_view, full_view, check_exact=True
)

drift_reliability_evidence["analysis_role"] = np.where(
    drift_reliability_evidence["population"].eq("eligible"),
    "retrospective_population_matched",
    "full_sample_drift_with_eligible_outcomes",
)

drift_reliability_evidence["validated_warning_rule"] = False
drift_reliability_evidence["practical_deterioration_threshold_set"] = False

# A long-format table keeps each signal's measure, units and status explicit.
signal_metadata = {
    rule[0]: {
        "feature": feature,
        "measure": rule[1],
        "units": rule[2],
    }
    for feature, rule in SIGNAL_RULES.items()
}

long_parts = []

for signal in link_signals:
    part = drift_reliability_evidence[
        [
            "population", "period", "model", "metric",
            "current_n", "current_events", "current_prevalence",
            "reference_value", "current_value",
            "worsening_change",
            "worsening_ci_low", "worsening_ci_high",
            "direction_interpretation",
            "analysis_role",
            signal, signal + "_status",
        ]
    ].copy()

    part = part.rename(columns={
        signal: "signal_value",
        signal + "_status": "signal_calculation_status",
    })

    part["signal"] = signal
    part["feature"] = signal_metadata[signal]["feature"]
    part["signal_measure"] = signal_metadata[signal]["measure"]
    part["signal_units"] = signal_metadata[signal]["units"]

    long_parts.append(part)

drift_reliability_evidence_long = pd.concat(
    long_parts, ignore_index=True
)

require(
    len(drift_reliability_evidence_long) == 880,
    "Expected 176 comparisons × 5 signals = 880 detail rows."
)

require(
    not drift_reliability_evidence_long.duplicated(
        ["population", "period", "model", "metric", "signal"]
    ).any(),
    "Duplicate signal-detail keys."
)

# Export only aggregate evidence to a new timestamped folder.
link_output = (
    DATA
    / "notebook_12_drift_change_linkage"
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
link_output.mkdir(parents=True, exist_ok=False)

link_exports = {
    "half_year_drift_reliability_change_evidence.csv":
        drift_reliability_evidence,
    "half_year_drift_reliability_change_signal_details.csv":
        drift_reliability_evidence_long,
}

for name, frame in link_exports.items():
    frame.to_csv(
        link_output / name,
        index=False,
        na_rep="__MISSING__",
    )

    recovered = pd.read_csv(
        link_output / name,
        float_precision="round_trip",
        keep_default_na=False,
        na_values=["__MISSING__"],
    )

    pd.testing.assert_frame_equal(
        frame.reset_index(drop=True),
        recovered.reset_index(drop=True),
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-15,
    )

link_specification = {
    "stage": "Notebook 12 Stage 5C - retrospective evidence linkage",
    "calculation_checks": "passed",
    "interpretation_review_completed": False,
    "reliability_anchor": "2018-H1",
    "drift_reference_years": [2015, 2016, 2017],
    "later_half_years": 11,
    "signals": signal_metadata,
    "metrics": CHANGE_METRICS,
    "evidence_rows": 176,
    "signal_detail_rows": 880,
    "stage5b_source": str(change_csv),
    "stage5b_sha256": sha256(change_csv),
    "new_correlations_computed": False,
    "warning_thresholds_set": False,
    "models_changed": False,
    "limitations": [
        "Eleven later temporal cohorts; repeated views, models, metrics "
        "and signals do not create independent temporal observations.",
        "Both population views reuse eligible-loan reliability.",
        "Eligible membership depends on later follow-up availability.",
        "Pointwise bootstrap intervals do not establish practical importance.",
        "Uncertain change does not establish stability or harmlessness.",
        "AP and Brier require prevalence context.",
        "Historical input publication dates remain unverified.",
        "This table does not validate warning accuracy, lead time, causation, "
        "or intervention benefit.",
    ],
}

spec_name = "stage5c_linkage_specification.json"

(link_output / spec_name).write_text(
    json.dumps(link_specification, indent=2, allow_nan=False),
    encoding="utf-8",
)

link_bundle = link_output / "notebook_12_stage5c_review.zip"

with zipfile.ZipFile(
    link_bundle, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name in [*link_exports, spec_name]:
        archive.write(link_output / name, arcname=name)

print("Stage 5C linkage checks passed; interpretation pending.")
print("176 evidence rows and 880 signal-detail rows.")
print("These represent 11 later half-years, not 880 independent cases.")
print("Upload this review ZIP:", link_bundle)
print("STOP: no warning rule or intervention has been evaluated.")

Stage 5C linkage checks passed; interpretation pending.
176 evidence rows and 880 signal-detail rows.
These represent 11 later half-years, not 880 independent cases.
Upload this review ZIP: E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_drift_change_linkage\20261018T015029878944Z\notebook_12_stage5c_review.zip
STOP: no warning rule or intervention has been evaluated.


## Stage 5C — Reviewed interpretation of drift and reliability changes

### What was completed

The five existing population-drift signals were linked to reliability
changes relative to 2018-H1 for Logistic Regression and XGBoost.

The review verified 176 evidence rows and 880 signal-detail rows.
The linked reliability values and uncertainty intervals match Stage 5B.
Both population views use the same eligible-loan reliability outcomes.

### What the evidence shows

Population drift does not consistently indicate whether reliability
will worsen. The relationship depends on the signal, cohort and
reliability metric.

For example, in the eligible population:
- In 2019-H2, DTI drift was approximately 0.039 development-IQR units,
  while XGBoost ROC-AUC showed supported deterioration.
- In 2023-H2, DTI drift was approximately 0.292 development-IQR units,
  while XGBoost ROC-AUC showed supported improvement.
- In 2023-H2, XGBoost Brier score nevertheless worsened, showing that
  ranking and probability accuracy can move in different directions.

These examples illustrate mixed associations; they do not establish
causation or prove that any drift signal is universally useful or useless.

### Interpretation limits

- The analysis contains 11 later half-year cohorts. Repeated population
  views, models, metrics and signals do not create additional independent
  temporal observations.
- Bootstrap intervals are descriptive and pointwise. They do not capture
  shared economic shocks, model-fitting uncertainty or dependence across
  outcome windows.
- Supported direction does not establish practical importance.
- Uncertain direction does not establish stable reliability.
- Average precision and Brier score require prevalence context.
- Historical predictor publication dates remain unverified, and eligible
  membership was determined retrospectively.

### Research status and next step

The retrospective evidence linkage is complete and reviewed.
Chronological early-warning validation remains incomplete.

Next, assess a defensible chronological evaluation design that respects
model readiness, input availability and delayed outcomes. Define practical
deterioration separately from statistical direction before evaluating alerts.

The previously explored 2018–2023 outcomes cannot be described as an
untouched warning-validation sample.

No warning thresholds, false-alarm rates, missed-deterioration rates,
lead times or intervention benefits have been validated.
No data, labels, features or models were changed.

## Stage 5D — Chronological early-warning evaluation design

### Objective

Assess whether population-drift signals could provide useful information
before a cohort's reliability outcomes become available.

### Required separation

A warning uses population information available at its decision date.
Its later evaluation uses outcomes that mature afterward.

Any threshold selection using reliability outcomes may use only earlier
cohorts whose labels were available by that decision date.

### Feasibility constraints identified

The model label-readiness proxy is October 2022. This is not a verified
historical model-deployment date.

Some cohort records were first reported after the nominal decision dates.
Reporting dates do not establish predictor publication dates.

Eligible-loan membership was determined retrospectively.

The 2018–2023 reliability outcomes have already been explored, so a
simulation using them is exploratory rather than untouched validation.

### Next calculation

For each candidate decision date, establish:
1. Whether model readiness is supported under the stated assumptions.
2. Whether the complete population used to calculate each signal was
   available, or whether availability remains unverified.
3. Which earlier cohorts had mature labels.
4. Whether the evaluated cohort's labels were still unavailable.

Use this assessment to determine which chronological comparisons are
supportable and which require explicit simulation assumptions.

Do not choose alert thresholds or report warning accuracy until this
design has been reviewed.

Preserve the existing datasets, cohort membership, models and results.

In [48]:
# Stage 5D: chronological design feasibility — read-only.
# Reuse Stage 5A timing. No thresholds, exclusions, fitting or exports.
#
# First reporting is a timing proxy, NOT proof of predictor availability.
# The reporting-aligned scenario is hypothetical, not a deployment claim.

required_objects = [
    "availability_records",
    "eligible_availability",
    "cohort_label_ready",
    "model_ready_proxy",
    "information_availability_table",
]

require(
    all(name in globals() for name in required_objects),
    "Run the existing Stage 5A information-availability cell first."
)

stage5d_anchor = "2018-H1"
stage5d_periods = list(PERIODS)

require(
    stage5d_anchor in cohort_label_ready.index,
    "The reliability anchor has no label-readiness proxy."
)

stage5d_rows = []

for position, period in enumerate(stage5d_periods):
    full = availability_records.loc[
        availability_records["period"].eq(period)
    ]
    eligible = eligible_availability.loc[
        eligible_availability["period"].eq(period)
    ]

    require(
        len(full) > 0 and len(eligible) > 0,
        f"{period}: missing population records."
    )
    require(
        full["first_report_proxy"].notna().all(),
        f"{period}: missing first-report proxies; review before proceeding."
    )

    closing_month = 6 if period.endswith("H1") else 12
    nominal = pd.Period(
        f"{period[:4]}-{closing_month:02d}", freq="M"
    ) + 1

    # Wait until the month after every sampled loan's first report.
    # This does not establish historical predictor publication dates.
    all_first_reports_ready_proxy = (
        full["first_report_proxy"].max() + 1
    )

    reporting_aligned = max(
        nominal,
        model_ready_proxy,
        all_first_reports_ready_proxy,
    )

    scenarios = {
        "nominal": nominal,
        "reporting_aligned_proxy": reporting_aligned,
    }

    earlier_periods = stage5d_periods[:position]

    for scenario, decision in scenarios.items():
        matured_earlier = [
            earlier
            for earlier in earlier_periods
            if cohort_label_ready.loc[earlier] <= decision
        ]

        # Treat reporting-month records as usable from the next month.
        full_not_reported = int(
            full["first_report_proxy"].ge(decision).sum()
        )
        eligible_not_reported = int(
            eligible["first_report_proxy"].ge(decision).sum()
        )

        current_ready = cohort_label_ready.loc[period]
        anchor_ready = cohort_label_ready.loc[stage5d_anchor]

        stage5d_rows.append({
            "cohort": period,
            "scenario": scenario,
            "decision_month": str(decision),
            "months_after_nominal": decision.ordinal - nominal.ordinal,
            "model_labels_ready_proxy": model_ready_proxy <= decision,
            "full_sample_not_yet_reported": full_not_reported,
            "eligible_not_yet_reported": eligible_not_reported,
            "anchor_labels_ready_proxy": anchor_ready <= decision,
            "cohort_label_ready_proxy": str(current_ready),
            "cohort_labels_still_incomplete_proxy": current_ready > decision,
            "months_until_full_cohort_labels_proxy":
                current_ready.ordinal - decision.ordinal,
            "matured_earlier_cohort_count": len(matured_earlier),
            "matured_earlier_cohorts":
                ", ".join(matured_earlier) if matured_earlier else "None",
            "is_later_than_reliability_anchor": position > 0,
            "historical_predictor_availability": "unverified",
            "eligible_membership_known_at_decision": "unverified",
            "independent_untouched_validation": False,
        })

stage5d_feasibility = pd.DataFrame(stage5d_rows)

require(
    len(stage5d_feasibility) == 2 * len(stage5d_periods)
    and not stage5d_feasibility.duplicated(
        ["cohort", "scenario"]
    ).any(),
    "Unexpected scenario coverage."
)

# Reconcile nominal scenarios against the existing Stage 5A audit.
stage5d_nominal = (
    stage5d_feasibility.loc[
        stage5d_feasibility["scenario"].eq("nominal")
    ]
    .set_index("cohort")
    .sort_index()
)

stage5d_previous = (
    information_availability_table
    .set_index("cohort")
    .sort_index()
)

require(
    stage5d_nominal.index.equals(stage5d_previous.index),
    "Stage 5D cohort coverage differs from Stage 5A."
)

for new_column, previous_column in {
    "decision_month": "nominal_decision_month",
    "model_labels_ready_proxy":
        "model_labels_ready_by_nominal_decision",
    "full_sample_not_yet_reported":
        "full_sample_records_not_yet_reported",
    "eligible_not_yet_reported":
        "eligible_records_not_yet_reported",
    "cohort_label_ready_proxy": "cohort_label_ready_proxy",
    "matured_earlier_cohort_count": "matured_earlier_cohort_count",
}.items():
    require(
        stage5d_nominal[new_column].eq(
            stage5d_previous[previous_column]
        ).all(),
        f"Stage 5A reconciliation failed: {new_column}"
    )

stage5d_aligned = stage5d_feasibility.loc[
    stage5d_feasibility["scenario"].eq("reporting_aligned_proxy")
]

require(
    stage5d_aligned["model_labels_ready_proxy"].all()
    and stage5d_aligned["full_sample_not_yet_reported"].eq(0).all(),
    "Reporting-aligned scenario failed its stated proxy conditions."
)

for scenario in ["nominal", "reporting_aligned_proxy"]:
    print(f"\nSCENARIO: {scenario}")
    display(
        stage5d_feasibility.loc[
            stage5d_feasibility["scenario"].eq(scenario)
        ].reset_index(drop=True)
    )

print("\nStage 5D calculation checks passed.")
print("Positive months-until-labels = time before full cohort label maturity.")
print("This is NOT validated warning lead time.")
print("Earlier cohorts are counted only after their label-readiness proxies.")
print("First reporting does NOT establish origination predictor availability.")
print("Retrospective eligibility and previously explored outcomes remain limits.")
print("No dates are declared historically feasible solely from these proxies.")
print("No data, models, thresholds or prior results were changed.")
print("STOP: review both scenario tables before designing an alert evaluation.")


SCENARIO: nominal


,cohort,scenario,decision_month,months_after_nominal,model_labels_ready_proxy,full_sample_not_yet_reported,eligible_not_yet_reported,anchor_labels_ready_proxy,cohort_label_ready_proxy,cohort_labels_still_incomplete_proxy,months_until_full_cohort_labels_proxy,matured_earlier_cohort_count,matured_earlier_cohorts,is_later_than_reliability_anchor,historical_predictor_availability,eligible_membership_known_at_decision,independent_untouched_validation
0,2018-H1,nominal,2018-07,0,False,5167,4244,False,2021-11,True,40,0,None,False,unverified,unverified,False
1,2018-H2,nominal,2019-01,0,False,4617,3779,False,2022-12,True,47,0,None,True,unverified,unverified,False
2,2019-H1,nominal,2019-07,0,False,4879,4514,False,2023-03,True,44,0,None,True,unverified,unverified,False
3,2019-H2,nominal,2020-01,0,False,4534,4191,False,2023-06,True,41,0,None,True,unverified,unverified,False
4,2020-H1,nominal,2020-07,0,False,5310,5221,False,2023-11,True,40,0,None,True,unverified,unverified,False
5,2020-H2,nominal,2021-01,0,False,4382,4274,False,2024-02,True,37,0,None,True,unverified,unverified,False
6,2021-H1,nominal,2021-07,0,False,4892,4678,False,2024-07,True,36,0,None,True,unverified,unverified,False
7,2021-H2,nominal,2022-01,0,False,3990,3864,True,2025-02,True,37,1,2018-H1,True,unverified,unverified,False
8,2022-H1,nominal,2022-07,0,False,4456,4176,True,2025-08,True,37,1,2018-H1,True,unverified,unverified,False
9,2022-H2,nominal,2023-01,0,True,4184,4063,True,2026-01,True,36,2,"2018-H1, 2018-H2",True,unverified,unverified,False



SCENARIO: reporting_aligned_proxy


,cohort,scenario,decision_month,months_after_nominal,model_labels_ready_proxy,full_sample_not_yet_reported,eligible_not_yet_reported,anchor_labels_ready_proxy,cohort_label_ready_proxy,cohort_labels_still_incomplete_proxy,months_until_full_cohort_labels_proxy,matured_earlier_cohort_count,matured_earlier_cohorts,is_later_than_reliability_anchor,historical_predictor_availability,eligible_membership_known_at_decision,independent_untouched_validation
0,2018-H1,reporting_aligned_proxy,2022-10,51,True,0,0,True,2021-11,False,-11,0,None,False,unverified,unverified,False
1,2018-H2,reporting_aligned_proxy,2022-10,45,True,0,0,True,2022-12,True,2,1,2018-H1,True,unverified,unverified,False
2,2019-H1,reporting_aligned_proxy,2022-10,39,True,0,0,True,2023-03,True,5,1,2018-H1,True,unverified,unverified,False
3,2019-H2,reporting_aligned_proxy,2022-10,33,True,0,0,True,2023-06,True,8,1,2018-H1,True,unverified,unverified,False
4,2020-H1,reporting_aligned_proxy,2022-10,27,True,0,0,True,2023-11,True,13,1,2018-H1,True,unverified,unverified,False
5,2020-H2,reporting_aligned_proxy,2022-10,21,True,0,0,True,2024-02,True,16,1,2018-H1,True,unverified,unverified,False
6,2021-H1,reporting_aligned_proxy,2022-12,17,True,0,0,True,2024-07,True,19,2,"2018-H1, 2018-H2",True,unverified,unverified,False
7,2021-H2,reporting_aligned_proxy,2023-02,13,True,0,0,True,2025-02,True,24,2,"2018-H1, 2018-H2",True,unverified,unverified,False
8,2022-H1,reporting_aligned_proxy,2023-08,13,True,0,0,True,2025-08,True,24,4,"2018-H1, 2018-H2, 2019-H1, 2019-H2",True,unverified,unverified,False
9,2022-H2,reporting_aligned_proxy,2024-05,16,True,0,0,True,2026-01,True,20,6,"2018-H1, 2018-H2, 2019-H1, 2019-H2, 2020-H1, 2...",True,unverified,unverified,False



Stage 5D calculation checks passed.
Positive months-until-labels = time before full cohort label maturity.
This is NOT validated warning lead time.
Earlier cohorts are counted only after their label-readiness proxies.
First reporting does NOT establish origination predictor availability.
Retrospective eligibility and previously explored outcomes remain limits.
No dates are declared historically feasible solely from these proxies.
No data, models, thresholds or prior results were changed.
STOP: review both scenario tables before designing an alert evaluation.


In [49]:
# Stage 5D follow-up: label maturity at candidate decision dates.
# Read-only. Reuses existing timing proxies; no raw-file scans or changes.

require(
    "stage5d_feasibility" in globals()
    and "eligible_availability" in globals(),
    "Run Stage 5D and the existing Stage 5A availability cell first."
)

maturity_rows = []

for row in stage5d_feasibility.itertuples(index=False):
    decision = pd.Period(row.decision_month, freq="M")

    cohort = eligible_availability.loc[
        eligible_availability["period"].eq(row.cohort)
    ]

    ready_dates = cohort["label_ready_proxy"]

    require(
        len(cohort) > 0 and ready_dates.notna().all(),
        f"{row.cohort}: missing loans or label-readiness proxies."
    )

    # A label is counted as ready only under the existing timing proxy.
    mature = ready_dates.le(decision)
    remaining_months = np.array([
        ready.ordinal - decision.ordinal
        for ready in ready_dates.loc[~mature]
    ], dtype=int)

    maturity_rows.append({
        "cohort": row.cohort,
        "scenario": row.scenario,
        "decision_month": str(decision),
        "eligible_loans": len(cohort),
        "labels_ready_proxy": int(mature.sum()),
        "labels_not_ready_proxy": int((~mature).sum()),
        "labels_ready_pct": round(100 * mature.mean(), 2),
        "labels_not_ready_pct": round(100 * (~mature).mean(), 2),
        "median_months_remaining_among_not_ready": (
            float(np.median(remaining_months))
            if len(remaining_months) else np.nan
        ),
        "maximum_months_remaining": (
            int(remaining_months.max())
            if len(remaining_months) else 0
        ),
    })

stage5d_label_maturity = pd.DataFrame(maturity_rows)

require(
    len(stage5d_label_maturity) == len(stage5d_feasibility)
    and not stage5d_label_maturity.duplicated(
        ["cohort", "scenario"]
    ).any(),
    "Unexpected maturity-table coverage."
)

require(
    (
        stage5d_label_maturity["labels_ready_proxy"]
        + stage5d_label_maturity["labels_not_ready_proxy"]
    ).eq(stage5d_label_maturity["eligible_loans"]).all(),
    "Mature and immature label counts do not reconcile."
)

for scenario in ["nominal", "reporting_aligned_proxy"]:
    print(f"\nLABEL MATURITY: {scenario}")
    print(
        stage5d_label_maturity.loc[
            stage5d_label_maturity["scenario"].eq(scenario)
        ].to_string(index=False)
    )

print("\nLabel-maturity calculation checks passed.")
print("Readiness remains a proxy, not verified historical label publication.")
print("These counts do not establish predictor availability or warning accuracy.")
print("No outcome values were used to choose dates or thresholds.")
print("No data, labels, models or prior results were changed.")
print("STOP: review the maturity proportions before choosing a warning design.")


LABEL MATURITY: nominal
 cohort scenario decision_month  eligible_loans  labels_ready_proxy  labels_not_ready_proxy  labels_ready_pct  labels_not_ready_pct  median_months_remaining_among_not_ready  maximum_months_remaining
2018-H1  nominal        2018-07           23699                  88                   23611              0.37                 99.63                                     22.0                        40
2018-H2  nominal        2019-01           23718                  80                   23638              0.34                 99.66                                     20.0                        47
2019-H1  nominal        2019-07           24326                 122                   24204              0.50                 99.50                                     20.0                        44
2019-H2  nominal        2020-01           24285                  75                   24210              0.31                 99.69                                     20.0       

In [50]:
# Stage 5E: exploratory deterioration-size sensitivity.
# Candidate effect sizes only — NOT drift-alert thresholds.
# No model changes, threshold optimization, or file exports.

require(
    "change_csv" in globals() and change_csv.is_file(),
    "Run Stage 5B first so its reviewed results are available."
)

stage5e_changes = pd.read_csv(change_csv)

# Absolute changes on each metric's original scale.
# ROC-AUC: 1, 3, 5 percentage points.
# Average precision: 0.5, 1, 2 percentage points.
# Absolute risk gap: 0.25, 0.5, 1 percentage point.
# Brier changes remain in Brier-score units.
stage5e_candidates = {
    "roc_auc": [0.01, 0.03, 0.05],
    "average_precision": [0.005, 0.01, 0.02],
    "brier": [0.001, 0.0025, 0.005],
    "absolute_risk_gap": [0.0025, 0.005, 0.01],
}

require(
    len(stage5e_changes) == 88
    and not stage5e_changes.duplicated(
        ["period", "model", "metric"]
    ).any(),
    "Unexpected Stage 5B coverage."
)

stage5e_parts = []

for metric, candidates in stage5e_candidates.items():
    source = stage5e_changes.loc[
        stage5e_changes["metric"].eq(metric)
    ]

    require(len(source) == 22, f"{metric}: expected 22 comparisons.")

    for candidate in candidates:
        part = source.copy()
        part["candidate_worsening_size"] = candidate

        part["point_estimate_exceeds_candidate"] = (
            part["worsening_change"] > candidate
        )

        part["interval_position"] = np.select(
            [
                part["worsening_ci_low"] > candidate,
                part["worsening_ci_high"] <= candidate,
            ],
            [
                "entire_interval_above_candidate",
                "entire_interval_at_or_below_candidate",
            ],
            default="interval_crosses_candidate",
        )

        stage5e_parts.append(part)

stage5e_deterioration_sensitivity = pd.concat(
    stage5e_parts, ignore_index=True
)

require(
    len(stage5e_deterioration_sensitivity) == 264,
    "Expected 88 comparisons × 3 candidate effect sizes."
)

stage5e_summary = (
    stage5e_deterioration_sensitivity
    .groupby(
        ["model", "metric", "candidate_worsening_size"],
        as_index=False,
    )
    .agg(
        cohorts=("period", "size"),
        point_estimates_above_candidate=(
            "point_estimate_exceeds_candidate", "sum"
        ),
        intervals_entirely_above_candidate=(
            "interval_position",
            lambda s: s.eq("entire_interval_above_candidate").sum(),
        ),
        intervals_crossing_candidate=(
            "interval_position",
            lambda s: s.eq("interval_crosses_candidate").sum(),
        ),
    )
)

require(
    stage5e_summary["cohorts"].eq(11).all(),
    "Each sensitivity setting must retain all 11 later cohorts."
)

print("EXPLORATORY DETERIORATION-SIZE SENSITIVITY")
print(stage5e_summary.to_string(index=False))

print("\nChecks passed: all cohorts and all candidate sizes retained.")
print("Candidate sizes are illustrative, not validated business tolerances.")
print("Metrics are evaluated separately; no combined deterioration label.")
print("Intervals remain descriptive and pointwise.")
print("Below a candidate size does not establish stable reliability.")
print("AP and Brier comparisons still require prevalence context.")
print("No alert thresholds were selected or optimized.")
print("STOP: review sensitivity before constructing warning rules.")

EXPLORATORY DETERIORATION-SIZE SENSITIVITY
   model            metric  candidate_worsening_size  cohorts  point_estimates_above_candidate  intervals_entirely_above_candidate  intervals_crossing_candidate
logistic absolute_risk_gap                    0.0025       11                                7                                   5                             5
logistic absolute_risk_gap                    0.0050       11                                5                                   5                             1
logistic absolute_risk_gap                    0.0100       11                                5                                   4                             1
logistic average_precision                    0.0050       11                                3                                   1                             2
logistic average_precision                    0.0100       11                                2                                   1                      

In [51]:
# Stage 5F: development-only candidate drift cutoffs.
# Primary monitoring population: full sampled originations.
# No outcome-based tuning, warning evaluation, fitting or exports.

require(
    all(name in globals() for name in [
        "full_predictors", "reference_specs", "reference_values",
        "SIGNAL_RULES", "feature_drift"
    ]),
    "Run the existing Stage 2 and Stage 3A cells first."
)

stage5f_reference_before = json.dumps(
    reference_specs, sort_keys=True, allow_nan=False
)

def stage5f_signal_value(frame, feature):
    """Reuse the existing five drift definitions and frozen reference."""
    spec = reference_specs["full_sample"]["features"][feature]
    _, measure, _ = SIGNAL_RULES[feature]

    if measure == "wasserstein_iqr_scaled":
        values = observed_numeric(frame[feature])
        require(
            len(values) > 0 and spec["iqr"] > 0,
            f"{feature}: unavailable scaled Wasserstein comparison."
        )
        return float(
            wasserstein_distance(
                reference_values["full_sample"][feature], values
            ) / spec["iqr"]
        )

    if measure == "total_variation":
        reference_counts = {
            level["token"]: level["count"]
            for level in spec["levels"]
        }
        current_counts = (
            category_tokens(frame[feature]).value_counts().to_dict()
        )
        levels = set(reference_counts) | set(current_counts)

        return float(0.5 * sum(
            abs(
                current_counts.get(level, 0) / len(frame)
                - reference_counts.get(level, 0) / spec["n"]
            )
            for level in levels
        ))

    if measure == "binary_absolute_change_pp":
        return float(
            100 * abs(frame[feature].mean() - spec["proportion_one"])
        )

    raise ValueError(f"Unexpected signal measure: {measure}")


# Verify that the helper reproduces existing evaluation drift values.
# Evaluation outcomes are not read or used.
for period in PERIODS:
    current = full_predictors.loc[
        full_predictors["period"].eq(period)
    ]

    for feature, (_, measure, _) in SIGNAL_RULES.items():
        saved = feature_drift.loc[
            feature_drift["population"].eq("full_sample")
            & feature_drift["period"].eq(period)
            & feature_drift["feature"].eq(feature)
        ]

        require(len(saved) == 1, "Expected one saved drift comparison.")
        require(
            np.isclose(
                stage5f_signal_value(current, feature),
                float(saved.iloc[0][measure]),
                rtol=1e-10,
                atol=1e-12,
            ),
            f"Drift calculation differs: {period}/{feature}"
        )


stage5f_development = full_predictors.loc[
    full_predictors["SOURCE YEAR"].between(2015, 2017)
]

stage5f_dev_periods = [
    f"{year}-H{half}"
    for year in range(2015, 2018)
    for half in [1, 2]
]

require(
    len(stage5f_development) == 150000
    and set(stage5f_development["period"]) == set(stage5f_dev_periods),
    "Development population or half-year coverage differs."
)

stage5f_rows = []

for period in stage5f_dev_periods:
    current = stage5f_development.loc[
        stage5f_development["period"].eq(period)
    ]

    require(len(current) == 25000, f"{period}: expected 25,000 loans.")

    for feature, (signal, measure, units) in SIGNAL_RULES.items():
        stage5f_rows.append({
            "development_cohort": period,
            "population": "full_sample",
            "signal": signal,
            "feature": feature,
            "measure": measure,
            "units": units,
            "signal_value": stage5f_signal_value(current, feature),
        })

stage5f_development_signals = pd.DataFrame(stage5f_rows)

require(
    len(stage5f_development_signals) == 30
    and np.isfinite(
        stage5f_development_signals["signal_value"]
    ).all(),
    "Expected 30 finite development signal values."
)

stage5f_cutoffs = (
    stage5f_development_signals
    .groupby(["population", "signal", "units"], as_index=False)
    .agg(
        development_cohorts=("development_cohort", "nunique"),
        candidate_cutoff=("signal_value", "max"),
    )
)

stage5f_cutoffs["candidate_rule"] = (
    "alert if signal_value > candidate_cutoff"
)

require(
    len(stage5f_cutoffs) == 5
    and stage5f_cutoffs["development_cohorts"].eq(6).all(),
    "Expected five cutoffs, each based on six development cohorts."
)

require(
    json.dumps(reference_specs, sort_keys=True, allow_nan=False)
    == stage5f_reference_before,
    "Frozen reference specifications changed."
)

print("DEVELOPMENT HALF-YEAR SIGNALS")
print(
    stage5f_development_signals.pivot(
        index="development_cohort",
        columns="signal",
        values="signal_value",
    ).to_string()
)

print("\nCANDIDATE DEVELOPMENT-ONLY CUTOFFS")
print(stage5f_cutoffs.to_string(index=False))

print("\nChecks passed; frozen reference unchanged.")
print("No reliability outcomes were used to calculate cutoffs.")
print("Development cohorts overlap the pooled development reference.")
print("Six cohorts do not establish a calibrated false-alarm rate.")
print("Signals remain separate; no combined alert rule was created.")
print("Historical predictor availability remains unverified.")
print("STOP: review these cutoffs before calculating candidate alerts.")

DEVELOPMENT HALF-YEAR SIGNALS
signal              channel_total_variation  dti_missing_absolute_change_pp  dti_wasserstein_iqr_scaled  fico_wasserstein_iqr_scaled  ltv_wasserstein_iqr_scaled
development_cohort                                                                                                                                              
2015-H1                            0.008067                        3.670667                    0.056982                     0.062679                    0.047711
2015-H2                            0.005827                        1.270667                    0.019107                     0.008456                    0.012844
2016-H1                            0.008547                        0.698667                    0.024092                     0.006402                    0.010593
2016-H2                            0.003827                        1.725333                    0.016174                     0.029944                    0.061398
2017

In [52]:
# Stage 5G: exploratory candidate-alert evaluation.
# Fixed development cutoffs; all deterioration sizes retained.
# No fitting, cutoff optimization, exclusions or model changes.

import zipfile
from datetime import datetime, timezone

require(
    all(name in globals() for name in [
        "stage5f_cutoffs",
        "stage5e_deterioration_sensitivity",
        "stage5d_feasibility",
        "stage5d_label_maturity",
    ]),
    "Run Stages 5D, 5E and 5F first."
)

# 1. Calculate each signal's alert independently.
stage5g_alert_rows = []

for period in PERIODS:
    if period == "2018-H1":
        continue  # Reference cohort, not a later change comparison.

    for feature, (signal, measure, units) in SIGNAL_RULES.items():
        source = feature_drift.loc[
            feature_drift["population"].eq("full_sample")
            & feature_drift["period"].eq(period)
            & feature_drift["feature"].eq(feature)
        ]
        cutoff = stage5f_cutoffs.loc[
            stage5f_cutoffs["signal"].eq(signal),
            "candidate_cutoff",
        ]

        require(
            len(source) == 1 and len(cutoff) == 1,
            f"Missing or duplicate signal/cutoff: {period}/{signal}"
        )

        value = float(source.iloc[0][measure])
        threshold = float(cutoff.iloc[0])

        require(
            np.isfinite(value) and np.isfinite(threshold),
            f"Nonfinite signal or cutoff: {period}/{signal}"
        )

        stage5g_alert_rows.append({
            "period": period,
            "signal": signal,
            "signal_value": value,
            "candidate_cutoff": threshold,
            "signal_units": units,
            "candidate_alert": value > threshold,
        })

stage5g_alerts = pd.DataFrame(stage5g_alert_rows)

require(
    len(stage5g_alerts) == 55
    and not stage5g_alerts.duplicated(["period", "signal"]).any(),
    "Expected 11 cohorts × 5 separate alerts."
)

# 2. Link each alert to every metric/model/effect-size comparison.
stage5g_details = stage5g_alerts.merge(
    stage5e_deterioration_sensitivity,
    on="period",
    how="inner",
    validate="many_to_many",
)

stage5g_keys = [
    "period", "signal", "model", "metric", "candidate_worsening_size"
]

require(
    len(stage5g_details) == 1320
    and not stage5g_details.duplicated(stage5g_keys).any(),
    "Expected 11 × 5 × 2 × 4 × 3 comparisons."
)

# 3. Attach the hypothetical reporting-aligned dates and maturity.
stage5g_timing = stage5d_feasibility.loc[
    stage5d_feasibility["scenario"].eq("reporting_aligned_proxy")
].copy()

stage5g_maturity = stage5d_label_maturity.loc[
    stage5d_label_maturity["scenario"].eq("reporting_aligned_proxy"),
    [
        "cohort", "labels_ready_pct", "labels_not_ready_pct",
        "median_months_remaining_among_not_ready",
    ],
]

stage5g_timing = stage5g_timing.merge(
    stage5g_maturity,
    on="cohort",
    validate="one_to_one",
)

stage5g_details = stage5g_details.merge(
    stage5g_timing.rename(columns={"cohort": "period"}),
    on="period",
    how="left",
    validate="many_to_one",
)

require(
    stage5g_details["decision_month"].notna().all(),
    "An evaluated cohort has no scenario date."
)

alert = stage5g_details["candidate_alert"]
deterioration = stage5g_details["point_estimate_exceeds_candidate"]

stage5g_details["point_estimate_classification"] = np.select(
    [
        alert & deterioration,
        alert & ~deterioration,
        ~alert & deterioration,
        ~alert & ~deterioration,
    ],
    ["hit", "false_alarm", "miss", "correct_non_alert"],
    default="invalid",
)

stage5g_details["outcome_interval_uncertain"] = (
    stage5g_details["interval_position"].eq("interval_crosses_candidate")
)

# 4. Summarize without selecting a winning signal or effect size.
stage5g_group_keys = [
    "model", "metric", "candidate_worsening_size", "signal"
]

stage5g_summary = (
    stage5g_details
    .groupby(stage5g_group_keys, as_index=False)
    .agg(
        cohorts=("period", "size"),
        alerts=("candidate_alert", "sum"),
        point_estimate_deterioration=(
            "point_estimate_exceeds_candidate", "sum"
        ),
        hits=("point_estimate_classification",
              lambda s: s.eq("hit").sum()),
        false_alarms=("point_estimate_classification",
                      lambda s: s.eq("false_alarm").sum()),
        misses=("point_estimate_classification",
                lambda s: s.eq("miss").sum()),
        correct_non_alerts=("point_estimate_classification",
                            lambda s: s.eq("correct_non_alert").sum()),
        uncertain_outcome_intervals=("outcome_interval_uncertain", "sum"),
    )
)

require(
    len(stage5g_summary) == 120
    and stage5g_summary["cohorts"].eq(11).all(),
    "Unexpected evaluation-summary coverage."
)

require(
    stage5g_summary[
        ["hits", "false_alarms", "misses", "correct_non_alerts"]
    ].sum(axis=1).eq(11).all(),
    "Classification counts do not reconcile."
)

# Descriptive rates only; undefined denominators stay missing.
stage5g_summary["precision"] = (
    stage5g_summary["hits"]
    / stage5g_summary["alerts"].replace(0, np.nan)
)
stage5g_summary["recall"] = (
    stage5g_summary["hits"]
    / stage5g_summary["point_estimate_deterioration"].replace(0, np.nan)
)
stage5g_summary["false_alarm_rate"] = (
    stage5g_summary["false_alarms"]
    / (
        stage5g_summary["false_alarms"]
        + stage5g_summary["correct_non_alerts"]
    ).replace(0, np.nan)
)

# Always-alert and never-alert reference counts for comparison.
stage5g_summary["always_alert_hits"] = (
    stage5g_summary["point_estimate_deterioration"]
)
stage5g_summary["always_alert_false_alarms"] = (
    11 - stage5g_summary["point_estimate_deterioration"]
)
stage5g_summary["never_alert_misses"] = (
    stage5g_summary["point_estimate_deterioration"]
)

# 5. Save aggregate results in a new folder.
stage5g_output = (
    DATA / "notebook_12_candidate_alerts"
    / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
stage5g_output.mkdir(parents=True, exist_ok=False)

stage5g_files = {
    "candidate_alerts.csv": stage5g_alerts,
    "candidate_alert_evaluation_details.csv": stage5g_details,
    "candidate_alert_evaluation_summary.csv": stage5g_summary,
    "development_candidate_cutoffs.csv": stage5f_cutoffs,
}

for filename, frame in stage5g_files.items():
    frame.to_csv(stage5g_output / filename, index=False)

stage5g_specification = {
    "stage": "5G exploratory delayed-monitoring simulation",
    "population_for_signals": "full_sample",
    "population_for_reliability": "retrospectively eligible loans",
    "cutoff_rule": "strictly exceed maximum of six development half-years",
    "signals_evaluated_separately": True,
    "all_candidate_deterioration_sizes_retained": True,
    "classification_basis": "point estimate; interval uncertainty retained",
    "historical_predictor_availability": "unverified",
    "independent_untouched_validation": False,
    "validated_warning_lead_time": False,
    "cutoffs_optimized_on_evaluation_outcomes": False,
    "limitations": [
        "Development cohorts overlap the pooled reference.",
        "Only six development half-years establish candidate cutoffs.",
        "Eleven evaluation cohorts are temporally dependent.",
        "Candidate deterioration sizes are illustrative business tolerances.",
        "Pointwise intervals do not cover all sources of uncertainty.",
        "Some hypothetical dates occur after almost all labels mature.",
        "Remaining time to full maturity is not warning lead time.",
        "Previously explored outcomes make this evaluation exploratory.",
        "Results do not establish intervention or retraining benefit.",
    ],
}

(stage5g_output / "evaluation_specification.json").write_text(
    json.dumps(stage5g_specification, indent=2),
    encoding="utf-8",
)

stage5g_zip = stage5g_output / "notebook_12_stage5g_review.zip"

with zipfile.ZipFile(
    stage5g_zip, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for filename in [
        *stage5g_files,
        "evaluation_specification.json",
    ]:
        archive.write(stage5g_output / filename, arcname=filename)

print("ALERT COUNTS BY SIGNAL")
print(
    stage5g_alerts.groupby("signal")["candidate_alert"]
    .agg(["sum", "count"]).to_string()
)
print("\nStage 5G calculation checks passed.")
print("Outcome classifications are provisional and use point estimates.")
print("Interval uncertainty is retained; no uncertain cohorts were removed.")
print("No cutoff was optimized and no winning rule was selected.")
print("Upload this small review ZIP:")
print(stage5g_zip)
print("STOP: review before interpreting warning usefulness.")

ALERT COUNTS BY SIGNAL
                                sum  count
signal                                    
channel_total_variation          10     11
dti_missing_absolute_change_pp   11     11
dti_wasserstein_iqr_scaled        8     11
fico_wasserstein_iqr_scaled       4     11
ltv_wasserstein_iqr_scaled       10     11

Stage 5G calculation checks passed.
Outcome classifications are provisional and use point estimates.
Interval uncertainty is retained; no uncertain cohorts were removed.
No cutoff was optimized and no winning rule was selected.
Upload this small review ZIP:
E:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\notebook_12_candidate_alerts\20261018T020618171800Z\notebook_12_stage5g_review.zip
STOP: review before interpreting warning usefulness.


## Stages 5D–5G — Reviewed monitoring evaluation

### What was completed

- Compared nominal cohort decision dates with hypothetical dates that wait
  for model-label readiness and all sampled loans' first reports.
- Measured the proportion of eligible-loan labels already mature at each
  candidate date.
- Evaluated three illustrative deterioration sizes for each reliability metric.
- Calculated five separate candidate alert cutoffs using the maximum drift
  observed across the six development half-years, 2015–2017.
- Evaluated those fixed rules across all 11 later half-year cohorts.

The review reconciled 55 candidate alerts, 1,320 detailed comparisons and
120 evaluation-summary rows with the earlier drift and reliability results.

### Main findings

Candidate alerts did not demonstrate a dependable general-purpose warning
rule across reliability metrics and deterioration sizes.

DTI missingness alerted on all 11 cohorts and therefore matched the
always-alert benchmark. Channel and LTV each alerted on 10 cohorts,
DTI distribution on eight, and FICO distribution on four.

Frequent alerts captured deterioration but also produced false alarms.
Less frequent alerts could miss deterioration. Results depended on the
reliability metric and candidate deterioration size.

For XGBoost Brier worsening greater than 0.0025:

| Signal | Hits | False alarms | Misses | Correct non-alerts |
|---|---:|---:|---:|---:|
| Channel | 7 | 3 | 1 | 0 |
| DTI missingness | 8 | 3 | 0 | 0 |
| DTI distribution | 6 | 2 | 2 | 1 |
| FICO distribution | 2 | 2 | 6 | 1 |
| LTV distribution | 7 | 3 | 1 | 0 |

These classifications use point estimates as provisional outcomes.
Three cohort intervals cross this candidate deterioration size.

### Timing interpretation

At the reporting-aligned dates, almost all labels were already mature for
2018-H2 through 2020-H1. These comparisons provide little advance-monitoring
opportunity despite preceding the final few labels.

For 2021-H1 through 2023-H2, approximately 83%–95% of labels remained
immature under the timing proxies. Median remaining time among immature
labels was 4–9 months.

These durations are not validated warning lead times. Label readiness is
a proxy, predictor publication dates remain unverified, and some candidate
decision dates are shared across cohorts.

### Limitations

- The evaluation is exploratory; 2018–2023 outcomes were previously explored.
- Development cutoffs use only six half-years, each overlapping the pooled
  development reference. They do not establish a calibrated false-alarm rate.
- Deterioration sizes are illustrative effect-size candidates, not validated
  business tolerances.
- Full-sample drift is compared with retrospectively eligible-loan reliability.
- Pointwise bootstrap intervals do not capture shared economic shocks,
  model-fitting uncertainty or all temporal dependence.
- False-alarm and miss counts depend on uncertain deterioration classifications.
- Below a candidate deterioration size does not establish stable reliability.

### Research status

Retrospective drift–reliability analysis and exploratory candidate-alert
evaluation are completed and reviewed.

Verified historical warnings at cohort arrival, independent warning
validation and operational lead time remain unestablished.

Recalibration and retraining benefits have not been evaluated.
No intervention benefit, cost saving or automatic retraining trigger is claimed.

Fannie Mae compatibility checks and external evaluation remain pending.

No existing loans, labels, features, models or prior results were changed.
Candidate cutoffs were not optimized on evaluation outcomes.